In [57]:
import os
import json
import faiss
import numpy as np
from openai import OpenAI
import traceback
import openai  # 추가
from dotenv import load_dotenv
import os
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
import jsonlines
import MeCab
from rank_bm25 import BM25Okapi


# .env 파일 로드
load_dotenv('/upstage-ai-advanced-ir7/.env')

# API_KEY 값을 가져옴
openai_api_key = os.getenv('OPENAI_API_KEY')
upstage_api_key = os.getenv('UPSTAGE_API_KEY')



os.environ["OPENAI_API_KEY"] = openai_api_key

# Upstage API 클라이언트 설정
client = OpenAI(
    api_key= upstage_api_key,
    base_url="https://api.upstage.ai/v1/solar"
)

client_gpt = OpenAI()

In [58]:
import ast

In [59]:
mecab = MeCab.Tagger()

# JSONL 파일 경로
jsonl_file_path = '/upstage-ai-advanced-ir7/data/documents.jsonl'

# 문서와 docid 저장할 리스트
documents = []
docids = []

stoptags = {"E", "J", "SC", "SE", "SF", "VCN", "VCP", "VX"}

# JSONL 파일 읽기
with jsonlines.open(jsonl_file_path) as reader:
    for obj in reader:
        docids.append(obj['docid'])      # docid 저장
        documents.append(obj['content']) # content 저장

# Mecab을 사용하여 문서 토큰화
def tokenize_with_mecab(text):
    tokens = mecab.parse(text).splitlines()  # Mecab 결과를 라인 단위로 분리
    processed_tokens = []
    
    for token in tokens:
        if "\t" in token:  # 형태소와 품사 태그가 \t로 구분됨
            word, tag_info = token.split("\t")
            pos_tag = tag_info.split(",")[0]  # 품사 태그는 ,로 구분된 첫 번째 요소
            if pos_tag not in stoptags:  # 불필요한 품사 태그가 아닌 경우에만 추가
                processed_tokens.append(word)
    
    return processed_tokens

tokenized_corpus = [tokenize_with_mecab(doc) for doc in documents]

# BM25 인덱서 생성
bm25 = BM25Okapi(tokenized_corpus)

In [60]:
with open("/upstage-ai-advanced-ir7/data/eval.jsonl", "r") as f:
    eval_doc_mapping = [json.loads(line) for line in f]
    

doc_mapping = {}
with open("/upstage-ai-advanced-ir7/data/documents.jsonl", "r") as f:
    for line in f:
        doc = json.loads(line)
        doc_mapping[doc['docid']] = doc 

index = faiss.read_index("knn_index_cosine.faiss")

# gpu_index = faiss.index_cpu_to_gpu(res, 0, index)

with open("chunk_mappings.json", "r") as f:
    chunk_doc_mapping = json.load(f)
    
model_path = 'Dongjin-kr/ko-reranker'

def exp_normalize(x):
    b = x.max()
    y = np.exp(x - b)
    return y / y.sum()
    
from transformers import AutoModelForSequenceClassification, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.to('cuda')
model.eval()

XLMRobertaForSequenceClassification(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 1024, padding_idx=1)
      (position_embeddings): Embedding(514, 1024, padding_idx=1)
      (token_type_embeddings): Embedding(1, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-23): 24 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSdpaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=1024, ou

순서
1. Query Transformation
2. Non-Science Question Detector

#### 1. Query Transformer

In [6]:
def query_transformer(messages):
    """
    LLM을 사용하여 사용자의 여러 메시지를 기반으로 검색에 적합한 단일 쿼리 생성.
    """
    # 시스템 메시지로 LLM에게 과제 부여 (한국어로)
    system_message = {
        "role": "system",
        "content": """
        당신은 사용자의 여러 대화 메시지를 하나의 검색 쿼리로 변환하는 전문가입니다. 
        사용자의 대화 중 중요한 정보만을 추출하여 간결하고 명확한 검색용 쿼리로 변환하세요. 
        대화 형식이 아닌, 연구나 조사를 위한 검색어처럼 정확한 질문을 생성하세요.
        절대 답을 생성하지 마시오.
        """
    }

    # 사용자 메시지 준비
    dialogue_messages = [{"role": msg['role'], "content": msg['content']} for msg in messages]

    # LLM 호출을 위한 메시지 배열 생성
    full_message = [system_message] + dialogue_messages
    
    print(dialogue_messages)
    print(full_message)
    

    # OpenAI API 호출하여 적절한 검색 쿼리 생성
    result = client_gpt.chat.completions.create(
        model="chatgpt-4o-latest",  # LLM 모델 지정
        messages=full_message,
        temperature=0
    )

    # LLM이 생성한 쿼리 반환 (ChatCompletionMessage 형식에서 content에 직접 접근)
    transformed_query = result.choices[0].message.content
    print(f"변환된 쿼리: {transformed_query}")
    return transformed_query

In [61]:
def query_transformer(messages):
    """
    LLM을 사용하여 사용자의 여러 메시지를 기반으로 검색에 적합한 단일 쿼리 생성.
    """
    # # 시스템 메시지로 LLM에게 과제 부여 (한국어로)
    # system_message = {
    #     "role": "system",
    #     "content": """
    #     당신은 사용자의 여러 대화 메시지를 하나의 검색 쿼리로 변환하는 전문가입니다. 
    #     사용자의 대화 중 중요한 정보만을 추출하여 간결하고 명확한 검색용 쿼리로 변환하세요. 
    #     대화 형식이 아닌, 연구나 조사를 위한 검색어처럼 정확한 질문을 생성하세요.
    #     절대 답을 생성하지 마시오.
    #     """
    # }

    # # 사용자 메시지 준비
    # dialogue_messages = [{"role": msg['role'], "content": msg['content']} for msg in messages]

    # # LLM 호출을 위한 메시지 배열 생성
    # full_message = [system_message] + dialogue_messages
    
    # print(dialogue_messages)
    # print(full_message)
    

    # # OpenAI API 호출하여 적절한 검색 쿼리 생성
    # result = client_gpt.chat.completions.create(
    #     model="chatgpt-4o-latest",  # LLM 모델 지정
    #     messages=full_message,
    #     temperature=0
    # )

    # # LLM이 생성한 쿼리 반환 (ChatCompletionMessage 형식에서 content에 직접 접근)
    # transformed_query = result.choices[0].message.content
    # print(f"변환된 쿼리: {transformed_query}")
    
    transformed_query = ''
    for message in messages:
        transformed_query += message['content'] + ', '
    return transformed_query

In [25]:
messages = [
    {"role": "user", "content": "기억 상실증 걸리면 너무 무섭겠다."},
    {"role": "assistant", "content": "네 맞습니다."},
    {"role": "user", "content": "원인이 뭘까."}
]

In [29]:
transformed_query = ''
for message in messages:
    transformed_query += message['content'] + ', '

In [30]:
transformed_query

'기억 상실증 걸리면 너무 무섭겠다., 네 맞습니다., 원인이 뭘까., '

In [44]:
messages = [
    
    {"role": "user", "content": "사람이나 물체가 지구 위에서 땅속으로 꺼지거나 바깥으로 튕겨나가지 않고 가만히 서 있을 수 있잖아?"}, {"role": "assistant", "content": "네 맞습니다."}, {"role": "user", "content": "그 이유를 힘의 원리로 설명해줘."}
    
]

In [188]:
messages = [
    {"role": "user", "content": "기억 상실증 걸리면 너무 무섭겠다."},
    {"role": "assistant", "content": "네 맞습니다."},
    {"role": "user", "content": "원인이 뭘까."}
]

In [189]:
transformed_query = query_transformer(messages)

[{'role': 'user', 'content': '기억 상실증 걸리면 너무 무섭겠다.'}, {'role': 'assistant', 'content': '네 맞습니다.'}, {'role': 'user', 'content': '원인이 뭘까.'}]
[{'role': 'system', 'content': '\n        당신은 사용자의 여러 대화 메시지를 하나의 검색 쿼리로 변환하는 전문가입니다. \n        사용자의 대화 중 중요한 정보만을 추출하여 간결하고 명확한 검색용 쿼리로 변환하세요. \n        대화 형식이 아닌, 연구나 조사를 위한 검색어처럼 정확한 질문을 생성하세요.\n        절대 답을 생성하지 마시오.\n        '}, {'role': 'user', 'content': '기억 상실증 걸리면 너무 무섭겠다.'}, {'role': 'assistant', 'content': '네 맞습니다.'}, {'role': 'user', 'content': '원인이 뭘까.'}]
변환된 쿼리: 기억 상실증의 원인


In [ ]:
기억 상실증 걸리면 너무 무섭겠다. 원인이 뭘까.

In [12]:
transformed_query

'기억 상실증의 원인'

#### 2. Non-Science Query Detector

In [6]:
def science_query_detector(message):
    """
    LLM을 사용하여 단일 사용자의 질문을 검색에 적합한 쿼리로 변환.
    질문이 과학과 관련되어 않으면 '과학 관련 질문이 아닙니다'라고 답변.
    """
    # 시스템 메시지로 LLM에게 과제 부여 (한국어로)
    system_message = {
        "role": "system",
        "content": """
            당신은 과학적 또는 학문적 논의와 관련된 질문에 답변하는 전문가입니다.
            질문이 전혀 과학적이지 않은 경우에만 '과학 관련 질문이 아닙니다.'라고 답변하세요.
            질문이 과학과 조금이라도 아주 조금이라도 관련이 있다면, 예를 들어 음식, 여가 등 매우 기초적인 것이라도 그 질문을 구체적이고 명확한 검색 쿼리로 변환하세요.
            과학적 논의는 자연과학, 사회과학, 기술, 심리학, 역사적 연구 등 다양한 학문 분야를 포함할 수 있습니다.
            단, 변환된 쿼리를 명확히 출력하되, 변환 과정을 설명하거나 예시로 출력하지 말고, **변환된 쿼리만 출력하세요.**
            
            예를 들어:
            - '복잡한 데이터 구조 설계 방법을 알려줘.' -> '복잡한 데이터 구조 설계 방법'
            - '인간의 감정이 사회적 관계에 미치는 영향' -> '인간의 감정이 사회적 관계에 미치는 영향'
            변환된 쿼리만 출력하세요.
        """
    }

    # 사용자 질문 준비
    user_message = {"role": message[0]['role'], "content": message[0]['content']}

    # LLM 호출을 위한 전체 메시지 배열 생성
    full_message = [system_message, user_message]

    # OpenAI API 호출하여 적절한 검색 쿼리 생성
    result = client_gpt.chat.completions.create(
        model="gpt-4o",  # LLM 모델 지정
        messages=full_message,
        temperature=0
    )

    # LLM이 생성한 응답을 반환
    llm_response = result.choices[0].message.content

    # 과학 관련 질문 여부 판단 및 처리
    if "과학 관련 질문이 아닙니다" in llm_response:
        print("과학 관련 질문이 아닙니다.")
        return "과학 관련 질문이 아닙니다."
    else:
        # 과학 관련 질문일 경우 변환된 쿼리 반환
        refined_query = llm_response.strip()
        print(f"변환된 쿼리: {refined_query}")
        return refined_query

In [62]:
def science_query_detector(message):
    """
    LLM을 사용하여 단일 사용자의 질문을 검색에 적합한 쿼리로 변환.
    질문이 과학과 관련되어 않으면 '과학 관련 질문이 아닙니다'라고 답변.
    """
    # 시스템 메시지로 LLM에게 과제 부여 (한국어로)
    system_message = {
        "role": "system",
        "content": """
            당신은 일상 대화를 찾아내는 전문가 입니다
            질문이 매우 일상적인 경우에만 '과학 관련 질문이 아닙니다.'라고 답변하세요.
            예: 당신은 누구십니까? -> 과학 관련 질문이 아닙니다.

            위와 같은 질문이 아닌 경우 질문을 문서 검색에 알맞은 형태로 조금 바꾸어 내어 주세요.
            예: 착한 사마리아인에 대해 알려줘 -> 착한 사마리아인
            예: 여행은 뭐야? -> 여행의 뜻
            예: 우리는 왜 지구 위를 걸을 수 있을까? -> 지구 위를 걸을 수 있는 이유
            예: 야채 샐러드의 특징에 대해 알려줘.-> 야채 샐러드의 특징
        """
    }

    # 사용자 질문 준비
    user_message = {"role": message[0]['role'], "content": message[0]['content']}

    # LLM 호출을 위한 전체 메시지 배열 생성
    full_message = [system_message, user_message]

    # OpenAI API 호출하여 적절한 검색 쿼리 생성
    result = client_gpt.chat.completions.create(
        model="chatgpt-4o-latest",  # LLM 모델 지정
        messages=full_message,
        temperature=0
    )

    # LLM이 생성한 응답을 반환
    llm_response = result.choices[0].message.content

    # 과학 관련 질문 여부 판단 및 처리
    if "과학 관련 질문이 아닙니다" in llm_response:
        print("과학 관련 질문이 아닙니다.")
        return "과학 관련 질문이 아닙니다."
    else:
        # 과학 관련 질문일 경우 변환된 쿼리 반환
        # refined_query = llm_response.strip()
        # print(f"변환된 쿼리: {refined_query}")
        # return refined_query
        return message[0]['content']

In [249]:

message = [
    {"role": "user", "content": "과일 샐러드의 특징에 대해 알려줘."}
]

In [239]:

message = [
    {"role": "user", "content": "사람이나 물체가 지구 위에서 땅속으로 꺼지거나 바깥으로 튕겨나가지 않고 가만히 서 있을 수 있잖아?"}
]

In [241]:

message = [
    {"role": "user", "content": "캠핑의 긍정적 효과"}
]

In [247]:
message = [
    {"role": "user", "content": "너는 누구야?"}
]

In [245]:
message = [
    {"role": "user", "content": "지구상에 존재하는 물 중 가장 많은 물은 바닷물이잖아?"}
]

In [11]:
message = [
    {"role": "user", "content": "많은 사람들에게 선한 영향력을 준 벽돌공의 일대기에 대해 알려줘."}
]

In [12]:
science_query_detector(message)

'많은 사람들에게 선한 영향력을 준 벽돌공의 일대기에 대해 알려줘.'

#### 3. Retrival KNN + Reranker

In [6]:
def normalize(embeddings):
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    return (embeddings / norms).astype(np.float32)  # float32로 변환

def retrieval(query):
    query_result = client.embeddings.create(
        model="solar-embedding-1-large-query",
        input=query
    ).data[0].embedding

    query_embedding = np.array(query_result).reshape(1, -1)

    # 쿼리 임베딩 정규화
    normalized_query = normalize(query_embedding)

    # FAISS로 50개의 유사한 chunk 검색
    k = 200
    distances, indices = index.search(normalized_query, k)

    retrieved_chunks = []
    retrieved_doc_ids = []
    for idx in indices[0]:
        chunk_info = chunk_doc_mapping[idx]
        content = chunk_info['content']
        doc_id = chunk_info['doc_id']
        retrieved_chunks.append((query, content))
        retrieved_doc_ids.append(doc_id)
        # print(f"검색된 Chunk ID: {chunk_info['chunk_id']}")
        # print(f"연관된 문서 ID: {doc_id}")
        # print(f"Chunk 내용: {content}")
        # print("-" * 50)
        
    with torch.no_grad():
        inputs = tokenizer(retrieved_chunks, padding=True, truncation=True, return_tensors='pt', max_length=512).to('cuda')
        scores = model(**inputs, return_dict=True).logits.view(-1, ).float()

    # 상위 5개의 문서 선택 (내림차순 정렬)
    top_5_indices = torch.argsort(scores, descending=True)[:10]

    # 상위 5개의 문서 출력
    final_doc_ids = []
    print("상위 5개의 문서:")
    for i in top_5_indices:
        doc_index = retrieved_chunks[i][0]
        doc_text = retrieved_chunks[i][1]
        doc_id = retrieved_doc_ids[i]
        final_doc_ids.append(doc_id)
        # print(f"문서 인덱스: {doc_index}")
        # print(f"연관된 문서 ID: {doc_id}")
        # print(f"문서 내용: {doc_text}")
        
        # print(f"유사도 점수: {scores[i].item()}")
        # print("-" * 50)
    return final_doc_ids

In [ ]:
def retrieval_with_score(query):
    query_result = client.embeddings.create(
        model="solar-embedding-1-large-query",
        input=query
    ).data[0].embedding

    query_embedding = np.array(query_result).reshape(1, -1)

    # 쿼리 임베딩 정규화
    normalized_query = normalize(query_embedding)

    # FAISS로 50개의 유사한 chunk 검색
    k = 500
    distances, indices = index.search(normalized_query, k)

    retrieved_chunks = []
    retrieved_doc_ids = []
    for idx in indices[0]:
        chunk_info = chunk_doc_mapping[idx]
        content = chunk_info['content']
        doc_id = chunk_info['doc_id']
        retrieved_chunks.append((query, content))
        retrieved_doc_ids.append(doc_id)
        # print(f"검색된 Chunk ID: {chunk_info['chunk_id']}")
        # print(f"연관된 문서 ID: {doc_id}")
        # print(f"Chunk 내용: {content}")
        # print("-" * 50)
        
    with torch.no_grad():
        inputs = tokenizer(retrieved_chunks, padding=True, truncation=True, return_tensors='pt', max_length=512).to('cuda')
        scores = model(**inputs, return_dict=True).logits.view(-1, ).float()

    # 상위 5개의 문서 선택 (내림차순 정렬)
    top_5_indices = torch.argsort(scores, descending=True)[:10]

    # 상위 5개의 문서 출력
    final_doc_ids = []
    print("상위 5개의 문서:")
    for i in top_5_indices:
        doc_index = retrieved_chunks[i][0]
        doc_text = retrieved_chunks[i][1]
        doc_id = retrieved_doc_ids[i]
        final_doc_ids.append((doc_id, scores[i].item()))
        # print(f"문서 인덱스: {doc_index}")
        # print(f"연관된 문서 ID: {doc_id}")
        # print(f"문서 내용: {doc_text}")
        
        # print(f"유사도 점수: {scores[i].item()}")
        # print("-" * 50)

In [12]:
def normalize(embeddings):
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    return (embeddings / norms).astype(np.float32)  # float32로 변환

def min_max_normalize(ranked_docs):
    """
    Min-Max 정규화를 수행하는 함수
    Args:
        ranked_docs (list of tuple): [(id, score), (id, score), ...] 형태의 리스트
    Returns:
        normalized_ranked_docs (list of tuple): [(id, normalized_score), ...] 형태의 리스트
    """
    # 점수만 추출
    scores = np.array([score for _, score in ranked_docs])

    # Min-Max 정규화
    min_score = np.min(scores)
    max_score = np.max(scores)
    
    # 0으로 나누는 오류를 방지 (최소값과 최대값이 같은 경우)
    if max_score == min_score:
        normalized_scores = np.ones_like(scores)
    else:
        normalized_scores = (scores - min_score) / (max_score - min_score)

    # 정규화된 점수와 id를 다시 결합
    normalized_ranked_docs = [(docid, score) for (docid, _), score in zip(ranked_docs, normalized_scores)]
    
    return normalized_ranked_docs

def retrieval_with_score(query):
    query_result = client.embeddings.create(
        model="solar-embedding-1-large-query",
        input=query
    ).data[0].embedding

    query_embedding = np.array(query_result).reshape(1, -1)

    # 쿼리 임베딩 정규화
    normalized_query = normalize(query_embedding)

    # FAISS로 50개의 유사한 chunk 검색
    k = 200
    distances, indices = index.search(normalized_query, k)

    retrieved_chunks = []
    retrieved_doc_ids = []

    for i, idx in enumerate(indices[0]):
        chunk_info = chunk_doc_mapping[idx]
        content = chunk_info['content']
        doc_id = chunk_info['doc_id']
        
        # 코사인 거리를 코사인 유사도로 변환
        cosine_distance = distances[0][i]
        cosine_similarity = 1 - cosine_distance  # 유사도는 1에서 거리값을 뺀 것

        retrieved_chunks.append((query, content))
        retrieved_doc_ids.append((doc_id, cosine_similarity))
        # retrieved_scores.append(cosine_similarity)  # 유사도 점수 추가
    
    retrieved_doc_ids = min_max_normalize(retrieved_doc_ids)
        
    with torch.no_grad():
        inputs = tokenizer(retrieved_chunks, padding=True, truncation=True, return_tensors='pt', max_length=512).to('cuda')
        scores = model(**inputs, return_dict=True).logits.view(-1, ).float()

    # 상위 5개의 문서 선택 (내림차순 정렬)
    top_5_indices = torch.argsort(scores, descending=True)[:10]

    # 상위 5개의 문서 출력
    final_doc_ids = []
    print("상위 5개의 문서:")
    for i in top_5_indices:
        doc_index = retrieved_chunks[i][0]
        doc_text = retrieved_chunks[i][1]
        doc_id = retrieved_doc_ids[i]
        final_doc_ids.append((doc_id, scores[i].item()))
        # print(f"문서 인덱스: {doc_index}")
        # print(f"연관된 문서 ID: {doc_id}")
        # print(f"문서 내용: {doc_text}")
        
        # print(f"유사도 점수: {scores[i].item()}")
        # print("-" * 50)
    return final_doc_ids

In [15]:
def normalize(embeddings):
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    return (embeddings / norms).astype(np.float32)  # float32로 변환

def min_max_normalize(ranked_docs):
    """
    Min-Max 정규화를 수행하는 함수
    Args:
        ranked_docs (list of tuple): [(id, score), (id, score), ...] 형태의 리스트
    Returns:
        normalized_ranked_docs (list of tuple): [(id, normalized_score), ...] 형태의 리스트
    """
    # 점수만 추출
    scores = np.array([score for _, score in ranked_docs])

    # Min-Max 정규화
    min_score = np.min(scores)
    max_score = np.max(scores)
    
    # 0으로 나누는 오류를 방지 (최소값과 최대값이 같은 경우)
    if max_score == min_score:
        normalized_scores = np.ones_like(scores)
    else:
        normalized_scores = (scores - min_score) / (max_score - min_score)

    # 정규화된 점수와 id를 다시 결합
    normalized_ranked_docs = [(docid, score) for (docid, _), score in zip(ranked_docs, normalized_scores)]
    
    return normalized_ranked_docs

def merge_and_sum_scores(knn_retrieved_docs, bm25_retrieved_docs, p):
    """
    knn_retrieved_docs와 bm25_retrieved_docs에서 동일한 id를 가진 항목은 스코어를 p와 (1 - p)의 비율로 가중 합산하고,
    그렇지 않은 항목은 그대로 유지하며, 마지막에 점수대로 정렬하는 함수.
    
    Args:
        knn_retrieved_docs (list of tuple): [(id, score), ...] 형태의 리스트
        bm25_retrieved_docs (list of tuple): [(id, score), ...] 형태의 리스트
        p (float): knn 점수에 적용할 가중치 (0 <= p <= 1)
    
    Returns:
        merged_docs (list of tuple): [(id, combined_score), ...] 형태의 리스트, 점수 내림차순 정렬
    """
    # 두 리스트를 딕셔너리로 변환하여 빠르게 검색할 수 있게 함
    knn_dict = {doc_id: score for doc_id, score in knn_retrieved_docs}
    bm25_dict = {doc_id: score for doc_id, score in bm25_retrieved_docs}
    
    # 모든 unique id를 set으로 결합
    all_ids = set(knn_dict.keys()).union(set(bm25_dict.keys()))

    # 동일한 id가 있으면 스코어를 p와 (1 - p) 비율로 가중 합산
    merged_docs = []
    for doc_id in all_ids:
        knn_score = knn_dict.get(doc_id, 0)  # knn에 없으면 0으로 간주
        bm25_score = bm25_dict.get(doc_id, 0)  # bm25에 없으면 0으로 간주
        combined_score = p * knn_score + (1 - p) * bm25_score
        merged_docs.append((doc_id, combined_score))
    
    # 점수 내림차순으로 정렬
    merged_docs = sorted(merged_docs, key=lambda x: x[1], reverse=True)
    
    return merged_docs

def retrieval_with_score(query):
    query_result = client.embeddings.create(
        model="solar-embedding-1-large-query",
        input=query
    ).data[0].embedding

    query_embedding = np.array(query_result).reshape(1, -1)

    # 쿼리 임베딩 정규화
    normalized_query = normalize(query_embedding)

    # FAISS로 500개의 유사한 chunk 검색
    k = 200
    distances, indices = index.search(normalized_query, k)


    retrieved_doc_ids = []

    # 코사인 거리를 코사인 유사도로 변환 (1 - 거리)
    for i, idx in enumerate(indices[0]):
        chunk_info = chunk_doc_mapping[idx]
        # content = chunk_info['content']
        doc_id = chunk_info['doc_id']
        
        # 코사인 거리를 코사인 유사도로 변환
        cosine_distance = distances[0][i]
        cosine_similarity = 1 - cosine_distance  # 유사도는 1에서 거리값을 뺀 것

        # retrieved_chunks.append((query, content))
        retrieved_doc_ids.append((doc_id, cosine_similarity))
        # retrieved_scores.append(cosine_similarity)  # 유사도 점수 추가
    
    knn_retrieved_docs = min_max_normalize(retrieved_doc_ids)
    
    # BM25
    
    tokenized_query = tokenize_with_mecab(query)

    # BM25로 점수 계산
    doc_scores = bm25.get_scores(tokenized_query)

    # 점수가 높은 순서대로 문서 정렬 (상위 200개만)
    ranked_docs = sorted(zip(docids, doc_scores), key=lambda x: x[1], reverse=True)[:k]
    
    bm25_retrieved_docs = min_max_normalize(ranked_docs)

    merged_docs = merge_and_sum_scores(knn_retrieved_docs, bm25_retrieved_docs, 0.6)
    
    retrieved_chunks = []
    for i, idx in enumerate(merged_docs):
        doc_info = doc_mapping[idx[0]]
        content = doc_info['content']
        retrieved_chunks.append((query, content))



    with torch.no_grad():
        inputs = tokenizer(retrieved_chunks, padding=True, truncation=True, return_tensors='pt', max_length=512).to('cuda')
        scores = model(**inputs, return_dict=True).logits.view(-1, ).float()

    # 상위 5개의 문서 선택 (내림차순 정렬)
    top_5_indices = torch.argsort(scores, descending=True)

    # 상위 5개의 문서 출력
    final_docs = []
    print("상위 5개의 문서:")
    for i in top_5_indices:
        doc_index = retrieved_chunks[i][0]
        doc_text = retrieved_chunks[i][1]
        doc_id = merged_docs[i][0]
        final_docs.append((doc_id, scores[i].item()))
        # print(f"문서 인덱스: {doc_index}")
        # print(f"연관된 문서 ID: {doc_id}")
        # print(f"문서 내용: {doc_text}")
        
        # print(f"유사도 점수: {scores[i].item()}")
        # print("-" * 50)
        
    final_docs = min_max_normalize(final_docs)
    
    final_merged_docs = merge_and_sum_scores(final_docs, merged_docs, 0.45)
    
    
    return final_merged_docs[:5]
    

In [63]:
def normalize(embeddings):
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    return (embeddings / norms).astype(np.float32)  # float32로 변환

def min_max_normalize(ranked_docs):
    """
    Min-Max 정규화를 수행하는 함수
    Args:
        ranked_docs (list of tuple): [(id, score), (id, score), ...] 형태의 리스트
    Returns:
        normalized_ranked_docs (list of tuple): [(id, normalized_score), ...] 형태의 리스트
    """
    # 점수만 추출
    scores = np.array([score for _, score in ranked_docs])

    # Min-Max 정규화
    min_score = np.min(scores)
    max_score = np.max(scores)
    
    # 0으로 나누는 오류를 방지 (최소값과 최대값이 같은 경우)
    if max_score == min_score:
        normalized_scores = np.ones_like(scores)
    else:
        normalized_scores = (scores - min_score) / (max_score - min_score)

    # 정규화된 점수와 id를 다시 결합
    normalized_ranked_docs = [(docid, score) for (docid, _), score in zip(ranked_docs, normalized_scores)]
    
    return normalized_ranked_docs


def z_score_normalize(ranked_docs):
    """
    Z-Score 정규화를 수행하는 함수
    Args:
        ranked_docs (list): (docid, score)의 리스트
    Returns:
        normalized_ranked_docs (list): Z-Score로 정규화된 (docid, normalized_score)의 리스트
    """
    # 점수만 추출
    scores = np.array([score for _, score in ranked_docs])

    # Z-Score 정규화
    mean_score = np.mean(scores)
    std_dev = np.std(scores)
    
    # 표준편차가 0인 경우(모든 점수가 동일한 경우) 처리
    if std_dev == 0:
        normalized_scores = np.zeros_like(scores)
    else:
        normalized_scores = (scores - mean_score) / std_dev

    # 정규화된 점수와 docid를 다시 결합
    normalized_ranked_docs = [(docid, score) for (docid, _), score in zip(ranked_docs, normalized_scores)]
    
    return normalized_ranked_docs

def merge_and_sum_scores(knn_retrieved_docs, bm25_retrieved_docs, p):
    """
    knn_retrieved_docs와 bm25_retrieved_docs에서 동일한 id를 가진 항목은 스코어를 p와 (1 - p)의 비율로 가중 합산하고,
    그렇지 않은 항목은 그대로 유지하며, 마지막에 점수대로 정렬하는 함수.
    
    Args:
        knn_retrieved_docs (list of tuple): [(id, score), ...] 형태의 리스트
        bm25_retrieved_docs (list of tuple): [(id, score), ...] 형태의 리스트
        p (float): knn 점수에 적용할 가중치 (0 <= p <= 1)
    
    Returns:
        merged_docs (list of tuple): [(id, combined_score), ...] 형태의 리스트, 점수 내림차순 정렬
    """
    # 두 리스트를 딕셔너리로 변환하여 빠르게 검색할 수 있게 함
    knn_dict = {doc_id: score for doc_id, score in knn_retrieved_docs}
    bm25_dict = {doc_id: score for doc_id, score in bm25_retrieved_docs}
    
    # 모든 unique id를 set으로 결합
    all_ids = set(knn_dict.keys()).union(set(bm25_dict.keys()))

    # 동일한 id가 있으면 스코어를 p와 (1 - p) 비율로 가중 합산
    merged_docs = []
    for doc_id in all_ids:
        knn_score = knn_dict.get(doc_id, 0)  # knn에 없으면 0으로 간주
        bm25_score = bm25_dict.get(doc_id, 0)  # bm25에 없으면 0으로 간주
        combined_score = p * knn_score + (1 - p) * bm25_score
        merged_docs.append((doc_id, combined_score))
    
    # 점수 내림차순으로 정렬
    merged_docs = sorted(merged_docs, key=lambda x: x[1], reverse=True)
    
    return merged_docs

def retrieval_with_score(query):
    query_result = client.embeddings.create(
        model="solar-embedding-1-large-query",
        input=query
    ).data[0].embedding

    query_embedding = np.array(query_result).reshape(1, -1)

    # 쿼리 임베딩 정규화
    normalized_query = normalize(query_embedding)

    # FAISS로 500개의 유사한 chunk 검색
    k = 200
    distances, indices = index.search(normalized_query, k)


    retrieved_doc_ids = []

    # 코사인 거리를 코사인 유사도로 변환 (1 - 거리)
    for i, idx in enumerate(indices[0]):
        chunk_info = chunk_doc_mapping[idx]
        # content = chunk_info['content']
        doc_id = chunk_info['doc_id']
        
        # 코사인 거리를 코사인 유사도로 변환
        cosine_distance = distances[0][i]
        cosine_similarity = 1 - cosine_distance  # 유사도는 1에서 거리값을 뺀 것

        # retrieved_chunks.append((query, content))
        retrieved_doc_ids.append((doc_id, cosine_similarity))
        # retrieved_scores.append(cosine_similarity)  # 유사도 점수 추가
    
    knn_retrieved_docs = z_score_normalize(retrieved_doc_ids)
    

    # BM25
    
    tokenized_query = tokenize_with_mecab(query)

    # BM25로 점수 계산
    doc_scores = bm25.get_scores(tokenized_query)

    # 점수가 높은 순서대로 문서 정렬 (상위 200개만)
    ranked_docs = sorted(zip(docids, doc_scores), key=lambda x: x[1], reverse=True)[:k]
    
    bm25_retrieved_docs = z_score_normalize(ranked_docs)

    merged_docs = merge_and_sum_scores(knn_retrieved_docs, bm25_retrieved_docs, 0.7)
    
    retrieved_chunks = []
    for i, idx in enumerate(merged_docs):
        doc_info = doc_mapping[idx[0]]
        content = doc_info['content']
        retrieved_chunks.append((query, content))



    with torch.no_grad():
        inputs = tokenizer(retrieved_chunks, padding=True, truncation=True, return_tensors='pt', max_length=512).to('cuda')
        scores = model(**inputs, return_dict=True).logits.view(-1, ).float()

    # 상위 5개의 문서 선택 (내림차순 정렬)
    top_5_indices = torch.argsort(scores, descending=True)

    # 상위 5개의 문서 출력
    final_docs = []
    print("상위 5개의 문서:")
    for i in top_5_indices:
        doc_index = retrieved_chunks[i][0]
        doc_text = retrieved_chunks[i][1]
        doc_id = merged_docs[i][0]
        final_docs.append((doc_id, scores[i].item()))
        # print(f"문서 인덱스: {doc_index}")
        # print(f"연관된 문서 ID: {doc_id}")
        # print(f"문서 내용: {doc_text}")
        
        # print(f"유사도 점수: {scores[i].item()}")
        # print("-" * 50)
        
    final_docs = z_score_normalize(final_docs)
    
    final_merged_docs = merge_and_sum_scores(final_docs, merged_docs, 0.475)
    
    
    return final_merged_docs[:20]
    

In [8]:
result = retrieval_with_score('은하에는 엄청나게 많은 별들이 모여 있잖아? 그 많은 별들이 어떻게 뭉쳐 있을 수 있지?')

상위 5개의 문서:


In [ ]:
"은하에는 엄청나게 많은 별들이 모여 있잖아?"}, {"role": "assistant", "content": "네 맞습니다."}, {"role": "user", "content": "그 많은 별들이 어떻게 뭉쳐 있을 수 있지?

In [9]:
for doc in result:
    print(doc_mapping[doc[0]]['content'])

우주의 모든 은하는 많은 별로 구성되어 있습니다. 은하는 우주에서 가장 큰 천체로, 수많은 별들이 모여 하나의 은하를 형성합니다. 은하는 다양한 형태와 크기를 가지고 있으며, 각각의 은하는 고유한 특징을 가지고 있습니다. 은하는 우리 은하를 포함하여 수많은 은하들로 이루어져 있으며, 이들은 서로의 중력에 의해 서로를 유지하고 있습니다. 은하는 우주의 크기와 다양성을 보여주는 중요한 천체입니다. 많은 별들이 모여 은하를 형성하고, 이들은 우주의 미스터리를 해결하는 열쇠가 될 수 있습니다.
은하계는 우주에서 가장 많은 공간을 차지하는 물체입니다. 은하계는 별, 가스, 먼지 등으로 이루어져 있으며, 수많은 별들이 모여 하나의 큰 천체를 형성합니다. 은하계는 우리 은하를 포함하여 수많은 은하들로 구성되어 있으며, 그 크기는 상상을 초월합니다. 은하계는 우주의 크기와 다양성을 보여주는 대단한 현상 중 하나입니다. 은하계는 무한한 우주 공간을 차지하며, 우리가 아직 알지 못하는 많은 비밀을 감추고 있습니다.
은하수는 맑은 밤에 망원경 없이도 잘 보이는 것입니다. 은하수는 수많은 별들이 모여 있는 은하로, 은하수 안에는 수많은 별들이 빛나고 있습니다. 맑은 밤하늘을 바라보면 은하수의 아름다운 형태를 감상할 수 있습니다. 은하수는 우리 은하인은하수의 일부분이며, 지구에서도 잘 관측할 수 있는 천체입니다. 망원경 없이도 은하수를 관찰할 수 있는 이유는 은하수가 매우 크고 밝기 때문입니다. 은하수는 우리 은하인은하수의 중심을 따라 펼쳐져 있으며, 그 안에는 무수히 많은 별들이 존재합니다. 이러한 별들의 빛이 모여 은하수를 형성하고, 그 빛이 지구로 전해져 우리 눈에 잘 보이게 됩니다. 따라서 망원경 없이도 맑은 밤하늘에서 은하수를 감상할 수 있습니다. 은하수는 우주의 아름다움을 보여주는 대표적인 천체 중 하나이며, 맑은 밤하늘을 통해 우리는 은하수의 아름다움을 즐길 수 있습니다.
은하들은 비슷한 원소를 가지고 있습니다. 이는 우리은하와 다른 은하들이 공통으로 가지고 있는 특징입

In [15]:
result = retrieval_with_score('금성이 행성중 가장 밝게 보이는 이유는?')

상위 5개의 문서:


In [12]:
result =result[:5]

In [16]:
result

[('464ace62-ddf2-423d-a5d7-2f17e6785c8e', 8.518137952908454),
 ('45b8eb6a-87e3-4333-b01b-7c8b772f827f', 3.124107354992668),
 ('da6c8a3f-45a9-4025-a63a-47c05ba2b336', 2.9966446479205953),
 ('35c5dcc7-4720-4318-901e-770105ae63fd', 2.6885562485096655),
 ('52f35132-97a3-4010-b409-3a7e40126c63', 2.2868996574189566)]

In [17]:
for doc in result:
    print(doc_mapping[doc[0]])

{'docid': '464ace62-ddf2-423d-a5d7-2f17e6785c8e', 'src': 'ko_ai2_arc__ARC_Challenge__train', 'content': '금성이 다른 행성들보다 더 밝게 보이는 이유는 지구 쪽으로 가장 많은 햇빛을 반사하기 때문입니다. 케빈은 맑은 밤에 하늘을 관찰하고 있습니다. 그는 맨눈으로 금성, 화성, 목성, 토성을 볼 수 있습니다. 금성은 햇빛을 많이 반사하기 때문에 다른 행성들보다 더 밝게 보입니다. 이는 금성의 표면이 반사율이 높기 때문입니다. 금성은 태양으로부터 받은 햇빛을 표면에 반사하여 지구에서 관찰하기 쉽게 만듭니다. 따라서 케빈은 맑은 밤에 금성을 더 밝게 볼 수 있습니다.'}
{'docid': '45b8eb6a-87e3-4333-b01b-7c8b772f827f', 'src': 'ko_mmlu__astronomy__validation', 'content': '금성은 태양계에서 가장 가까운 행성 중 하나입니다. 그러나 화성이나 지구처럼 계절이 없는 이유는 금성의 자전축이 태양계의 평면에 거의 수직이기 때문입니다. 자전축이 수직이기 때문에 금성은 태양으로부터 받는 햇빛의 양이 일정하게 유지됩니다. 이로 인해 금성은 계절 변화가 없으며 항상 일정한 온도를 유지합니다. 이러한 환경은 생명체에게는 적합하지 않을 수 있지만, 금성의 특이한 기후 조건은 우주 탐사에 대한 연구에 많은 도움을 주고 있습니다. 금성은 여전히 우리에게 알려지지 않은 많은 비밀을 품고 있으며, 미래에 더 많은 연구와 탐사가 이루어질 것으로 기대됩니다.'}
{'docid': 'da6c8a3f-45a9-4025-a63a-47c05ba2b336', 'src': 'ko_mmlu__astronomy__test', 'content': '금성은 태양계의 두 번째로 가까운 행성입니다. 이 행성의 대략적인 나이는 7억 5천만 년으로 추정됩니다. 금성은 지구와 매우 비슷한 크기와 구성을 가지고 있으며, 약 90% 이상이 이산화탄소로 이루어져 있습

In [7]:
result = retrieval_with_score('달의 한쪽 면만 보이는 이유')

상위 5개의 문서:


In [8]:
doc_mapping['fcfee923-af42-43d1-b664-09a0112e4f74']['content']

'지구에서 달의 약 59%가 보이는 이유는 지구의 중력 때문입니다. 지구의 중력은 달을 지속적으로 끌어당기는 힘을 가지고 있습니다. 이로 인해 달은 지구 주변을 공전하면서 동시에 자전을 하게 됩니다. 이 자전과 공전이 같은 기간을 가지기 때문에 달의 한쪽 면이 항상 지구에서 보이게 됩니다. 지구의 중력은 달을 끌어당기는 힘을 가지고 있기 때문에 달의 약 59%가 지구에서 보이는 것입니다. 이러한 현상은 지구와 달 사이의 중력 상호작용으로 인해 발생합니다.'

In [9]:
result

[('fcfee923-af42-43d1-b664-09a0112e4f74', 5.381111486045304),
 ('8a78364e-63bf-4915-b718-fdc461bc62c9', 4.45264525239844),
 ('cefe7caf-6cd1-422a-b41e-e82b543556e9', 4.400076349872069),
 ('892eb0be-055f-47c5-8895-1afb97c8f406', 3.822421126579113),
 ('340485f8-4e78-44f4-a53a-2df21915367f', 2.8626556358845967)]

In [14]:
check_list = []

for doc in result:
    print(result[0][1] * 0.63)
    print(doc[1])
    print('*' * 100)
    if doc[1] > result[0][1] * 0.63:
        check_list.append(doc)
        

3.3901002362085415
5.381111486045304
****************************************************************************************************
3.3901002362085415
4.45264525239844
****************************************************************************************************
3.3901002362085415
4.400076349872069
****************************************************************************************************
3.3901002362085415
3.822421126579113
****************************************************************************************************
3.3901002362085415
2.8626556358845967
****************************************************************************************************


In [103]:
result[4:]

[('340485f8-4e78-44f4-a53a-2df21915367f', 2.8626556358845967)]

In [102]:
check_list

[('fcfee923-af42-43d1-b664-09a0112e4f74', 5.381111486045304),
 ('8a78364e-63bf-4915-b718-fdc461bc62c9', 4.45264525239844),
 ('cefe7caf-6cd1-422a-b41e-e82b543556e9', 4.400076349872069),
 ('892eb0be-055f-47c5-8895-1afb97c8f406', 3.822421126579113)]

In [100]:
new_check_list = check_list + result[len(check_list):]

In [101]:
new_check_list

[('fcfee923-af42-43d1-b664-09a0112e4f74', 5.381111486045304),
 ('8a78364e-63bf-4915-b718-fdc461bc62c9', 4.45264525239844),
 ('cefe7caf-6cd1-422a-b41e-e82b543556e9', 4.400076349872069),
 ('892eb0be-055f-47c5-8895-1afb97c8f406', 3.822421126579113),
 ('340485f8-4e78-44f4-a53a-2df21915367f', 2.8626556358845967)]

#### 4.1 LLM reranker

In [64]:
def llm_reranking(query, documents):
    """
    LLM을 사용하여 사용자의 여러 메시지를 기반으로 검색에 적합한 단일 쿼리 생성.
    """
    # 시스템 메시지로 LLM에게 과제 부여 (한국어로)
    
    
    
    check_doc = ''
    for i, doc in enumerate(documents):
        check_doc += f"문서 {i+1}: {doc_mapping[doc[0]]['content']} \n"

    # 시스템 메시지 생성
    system_message = {
        "role": "user",
        "content": f"""
        당신은 문서와 질문을 비교하여, 질문에 가장 적합한 문서들을 찾고 그 순서를 반환하는 전문가입니다.
        
        다음의 규칙을 반드시 따르세요:
        1. 주어진 질문과 문서들의 내용을 비교하여, 질문의 답을 찾을 수 있는 문서를 가장 적합한 순서대로 나열하세요.
        2. 반드시 리스트 형태로만 출력하세요. 다른 형식은 허용되지 않습니다. 예를 들어 [1, 3, 2, 4]와 같이 반환하세요.
        3. 리스트에 들어갈 문서 번호는 1부터 시작하며, 문서들의 순서를 중요도에 따라 배열하세요.
        4. 리스트 외의 다른 정보를 출력하지 마세요. 리스트 이외의 내용은 모델이 자동으로 무시해야 합니다.

        질문: {query}

        문서들:
        {check_doc}
        """
    }

# 사용자 메시지 준비


    # 사용자 메시지 준비
    # system_message['content'] = system_message['content'] + check_doc

    # LLM 호출을 위한 메시지 배열 생성
    system_message = [system_message]

    # OpenAI API 호출하여 적절한 검색 쿼리 생성
    result = client_gpt.chat.completions.create(
        model="chatgpt-4o-latest",  # LLM 모델 지정
        messages=system_message
    )

    # LLM이 생성한 쿼리 반환 (ChatCompletionMessage 형식에서 content에 직접 접근)
    transformed_query = result.choices[0].message.content
    print(f"변환된 쿼리: {transformed_query}")
    return transformed_query

In [36]:
result = retrieval_with_score('건설 현장에서 망치로 벽을 치는 이유는?')

상위 5개의 문서:


In [ ]:
[{"role": "user", "content": "달을 보면 항상 같은 면만 보이더라구"}, {"role": "assistant", "content": "네 맞습니다."}, {"role": "user", "content": "그 이유가 뭐야?"}]


In [51]:
messages = [{"role": "user", "content": "달을 보면 항상 같은 면만 보이더라구"}, {"role": "assistant", "content": "네 맞습니다."}, {"role": "user", "content": "그 이유가 뭐야?"}]

In [52]:
transformed_query = query_transformer(messages)

In [53]:
transformed_query

'달을 보면 항상 같은 면만 보이더라구, 네 맞습니다., 그 이유가 뭐야?, '

In [54]:
result = retrieval_with_score(transformed_query)

상위 5개의 문서:


In [55]:
i = 1
for doc in result:
    print(i)
    print(doc_mapping[doc[0]]['content'])
    i+=1

1
달의 본질적으로 같은 면을 항상 보는 이유는 달의 자전 주기와 궤도 주기가 같기 때문입니다. 달은 지구 주위를 공전하면서 동시에 자전을 합니다. 이때 달의 자전 주기와 궤도 주기가 정확히 일치하면서, 항상 같은 면을 우리에게 보여줍니다. 이러한 현상을 '달의 결합 자전'이라고 합니다. 달의 결합 자전은 지구의 중력과 달의 자력이 서로 작용하여 발생합니다. 이로 인해 달은 항상 같은 면을 우리에게 보여주는 것입니다. 이러한 현상은 우리가 달의 다른 면을 볼 수 없는 이유이기도 합니다. 달의 결합 자전은 우리에게 매우 익숙한 모습을 제공하며, 우리가 달을 관찰하고 연구하는 데 많은 도움을 주고 있습니다.
2
지구에서 보이는 달의 면이 항상 같은 이유는 달이 지구 주위를 한 바퀴 돌 때마다 한 번 회전하기 때문입니다. 이는 달의 자전과 지구 주위를 공전하는 속도가 동일하기 때문에 발생합니다. 달은 지구 주위를 약 27.3일 동안 공전하며, 동시에 자전하면서 항상 같은 면이 지구에서 보입니다. 이러한 현상은 달의 자전 주기와 공전 주기가 정확히 일치하기 때문에 발생합니다. 달의 자전 주기와 공전 주기가 일치하는 이유는 지구의 조력과 달의 조력이 서로 상쇄되기 때문입니다. 이러한 현상은 천문학적으로 매우 흥미로운 현상으로 알려져 있습니다.
3
달이 항상 지구에 같은 면을 보여준다는 사실은 달이 축을 중심으로 약 한 달을 주기로 회전한다는 증거입니다. 이는 달의 자전 주기로 인해 발생하는 현상으로 알려져 있습니다. 달은 지구 주변을 공전하면서 동시에 자전을 하기 때문에 항상 같은 면을 보여주는 것입니다. 이러한 현상은 달의 중력과 지구의 중력이 서로 상호작용하여 발생하는 것으로 알려져 있습니다. 달의 축을 중심으로 약 한 달을 주기로 회전하는 이러한 현상은 천문학자들에 의해 연구되고 있으며, 우주 공간에서의 달의 운동을 이해하는 데 중요한 역할을 합니다.
4
지구에서 달의 약 59%가 보이는 이유는 지구의 중력 때문입니다. 지구의 중력은 달을 지속적으로 끌어당기는 힘

In [56]:
re_ranked = llm_reranking(transformed_query, result)

변환된 쿼리: [1, 2, 4, 3, 5]


In [21]:
result

[('dafabda1-035e-46ba-8a8e-31c01ce1772b', 6.787048890681855),
 ('199f238b-b9e4-4fbf-95a9-db42b1ad71cc', 5.42315968075257),
 ('416e57d1-d61f-4ab5-a75f-95d53f8a0fce', 5.05190427758545),
 ('4a5a84c1-fe67-4892-9106-2dd7bfbe8cf8', 3.392962201322387),
 ('a8925d82-c326-40cd-b327-496c7e8ff1cb', 2.856638055214902),
 ('9265fa0a-b23b-470b-bf45-eca7ac84e735', 2.8047384706246463),
 ('57f11451-44d4-4d44-815e-c6efcc4c512f', 2.4794148786926167),
 ('87a1d456-fc03-409f-80b9-f34d4e0a5f6e', 2.425637492529453),
 ('3ab69b86-f2a5-466e-b187-275878890a78', 2.29117060679903),
 ('a0996f7e-b4bc-48b7-b2c9-03af5dbe28c3', 1.8720735861937847),
 ('bbfcadd2-9873-440e-85e6-32164e3689b5', 1.75300791496299),
 ('5fd02f31-0599-4064-8045-0c62417feb85', 1.7073998816858946),
 ('317aa57a-fca3-4891-bee5-0012b7b31eb6', 1.6960880906975855),
 ('9679589a-1f32-4e9e-9841-ce8d7a742316', 1.6404791222577115),
 ('68418bb9-fd2b-4e54-972c-89532b906cf4', 1.5472526937786024)]

In [27]:
re_ranked

'[1, 3, 2]'

In [12]:
from openai import OpenAI
client = OpenAI()

response = client.chat.completions.create(
    model="o1-preview",
    messages=[
        {
            "role": "user", 
            "content": "Write a bash script that takes a matrix represented as a string with format '[1,2],[3,4],[5,6]' and prints the transpose in the same format."
        }
    ]
)

print(response.choices[0].message.content)

Here is a Bash script that transposes a matrix from the given format:

```bash
#!/bin/bash

INPUT="$1"

# Remove outer brackets and replace '],[' with newlines
MATRIX=$(echo "$INPUT" | sed 's/^\[\(.*\)\]$/\1/' | sed 's/\],\[/\n/g')

# Use awk to transpose the matrix
echo "$MATRIX" | awk -F',' '
{
    for (i = 1; i <= NF; i++) {
        a[i, NR] = $i;
    }
    if (NF > max_nf) { max_nf = NF }
    max_nr = NR
}
END {
    for (i = 1; i <= max_nf; i++) {
        printf "["
        for (j = 1; j <= max_nr; j++) {
            printf "%s", a[i, j]
            if (j < max_nr) {
                printf ","
            }
        }
        printf "]"
        if (i < max_nf) {
            printf ","
        }
    }
    printf "\n"
}'
```

**Usage:**

Save the script to a file, for example, `transpose_matrix.sh`, give it execute permissions, and run it with the matrix string as an argument:

```bash
chmod +x transpose_matrix.sh
./transpose_matrix.sh '[1,2],[3,4],[5,6]'
```

**Output:**

```
[1,3,5]

In [12]:
def query_maker(query):
    
    persona_function_calling = """
        당신은 사용자의 질문을 분석하여 가장 적절한 transformed_query를 제안하는 전문가입니다.
        변형의 목표는 query와 reference content의 높은 연관성을 유지하면서도 구체적이고 명확하며 **과학적으로 표현된** 검색 질의를 생성하는 것입니다.
        변형 기준:
        - 핵심 개념 강조: standalone_query에서 다루고 있는 핵심 개념을 더 구체적이고 명확하게 설명하여 검색 질의를 풍부하게 만듭니다.
        - 관련 정보 추가: reference content의 내용을 참고하여 추가적으로 포함될 수 있는 정보나 맥락을 질의에 반영합니다.
        - **과학적 용어와 표현 사용**: 질문을 과학적인 어조로 재구성하고, 적절한 과학 용어를 활용합니다.
        - 간결한 질문 유지: 너무 길거나 복잡하지 않도록 질의를 간결하게 유지합니다.
        예시:
        질문: "기체의 부피나 형태가 왜 일정하지 않을까?"
        응답: "기체의 부피와 형태가 일정하지 않은 이유는 무엇인가요?"
    """

    # 질문 변형을 위한 메시지 구성
    system_message = {
        "role": "system",
        "content": persona_function_calling
    }
    user_message = {
        "role": "user",
        "content": f'''
        질문: "{query}"

        위의 지침을 따라 질문을 변형하여 6개의 updated_query를 생성하세요.
        각 transformed_query는 다음과 같은 형식을 따라주세요:

        transformed_query 1: 변형된 질문 1
        transformed_query 2: 변형된 질문 2
        transformed_query 3: 변형된 질문 3
        transformed_query 4: 변형된 질문 4
        transformed_query 5: 변형된 질문 5
        transformed_query 6: 변형된 질문 6
        '''
    }
    
    messages = [system_message, user_message]
    
    print(messages)
    # LLM 호출 - 단계 1.5: 질문 변형하기
    result = client_gpt.chat.completions.create(
        model="chatgpt-4o-latest",
        messages=messages,
        temperature=0
    )
    transformed_queries_text = result.choices[0].message.content.strip()
    
    # 변형된 질문들을 파싱하여 리스트로 저장
    transformed_queries = []
    for line in transformed_queries_text.split('\n'):
        if line.startswith('transformed_query'):
            _, query_text = line.split(':', 1)
            transformed_queries.append(query_text.strip())
            
    return transformed_queries

In [13]:
a = query_maker('나무의 분류에 대해 조사해 보기 위한 방법은?')

[{'role': 'system', 'content': '\n        당신은 사용자의 질문을 분석하여 가장 적절한 transformed_query를 제안하는 전문가입니다.\n        변형의 목표는 query와 reference content의 높은 연관성을 유지하면서도 구체적이고 명확하며 **과학적으로 표현된** 검색 질의를 생성하는 것입니다.\n        변형 기준:\n        - 핵심 개념 강조: standalone_query에서 다루고 있는 핵심 개념을 더 구체적이고 명확하게 설명하여 검색 질의를 풍부하게 만듭니다.\n        - 관련 정보 추가: reference content의 내용을 참고하여 추가적으로 포함될 수 있는 정보나 맥락을 질의에 반영합니다.\n        - **과학적 용어와 표현 사용**: 질문을 과학적인 어조로 재구성하고, 적절한 과학 용어를 활용합니다.\n        - 간결한 질문 유지: 너무 길거나 복잡하지 않도록 질의를 간결하게 유지합니다.\n        예시:\n        질문: "기체의 부피나 형태가 왜 일정하지 않을까?"\n        응답: "기체의 부피와 형태가 일정하지 않은 이유는 무엇인가요?"\n    '}, {'role': 'user', 'content': '\n        질문: "나무의 분류에 대해 조사해 보기 위한 방법은?"\n\n        위의 지침을 따라 질문을 변형하여 6개의 updated_query를 생성하세요.\n        각 transformed_query는 다음과 같은 형식을 따라주세요:\n\n        transformed_query 1: 변형된 질문 1\n        transformed_query 2: 변형된 질문 2\n        transformed_query 3: 변형된 질문 3\n        transformed_query 4: 변형된 질문 4\n        transformed_query 5: 변형된 질문 5\n        t

In [14]:
a

['나무의 분류 체계를 과학적으로 조사하는 방법은 무엇인가요?',
 '식물학적 기준에 따른 나무의 분류 방법을 연구하는 절차는 어떻게 되나요?',
 '나무의 계통 분류학적 특성을 분석하기 위한 연구 방법은 무엇인가요?',
 '나무의 종과 속을 구분하는 분류학적 방법론을 조사하는 방법은 무엇인가요?',
 '나무의 형태학적 및 유전적 분류를 위한 과학적 조사 방법은 무엇인가요?',
 '나무의 분류학적 연구를 위한 데이터 수집 및 분석 방법은 어떻게 설정되나요?']

In [44]:
def top_agent_without_relevance_check(queries):
    if len(queries) > 1:
        transformed_query = query_transformer(queries)
        query = [
            {"role": "user", "content": transformed_query}
        ]
        original_query = transformed_query
        checked_query = science_query_detector(query)
        
        if checked_query == '과학 관련 질문이 아닙니다.':
            return False
    else:
        original_query = queries[0]['content']
        checked_query = science_query_detector(queries)
        if checked_query == '과학 관련 질문이 아닙니다.':
            return False

    final_result = query_maker(original_query)
    return final_result

In [15]:
a = {1: '사람', 2: '돼지'}

In [18]:
a[3] = '머리'

['머리', '머리', '머리', '머리', '머리', '머리']

In [32]:
b = ['머리' for _ in range(6)]

In [45]:
new_queries = {}

In [47]:
with open('/upstage-ai-advanced-ir7/data/eval.jsonl') as f:
    for line in f:
        eval_doc = json.loads(line)
        top_agent_result = top_agent_without_relevance_check(eval_doc['msg'])
        if top_agent_result ==  False:
            new_queries[eval_doc['eval_id']] = [eval_doc['msg'][0]['content'] for _ in range(6)]
        else: 
            new_queries[eval_doc['eval_id']] = top_agent_result


[{'role': 'system', 'content': '\n        당신은 사용자의 질문을 분석하여 가장 적절한 transformed_query를 제안하는 전문가입니다.\n        변형의 목표는 query와 reference content의 높은 연관성을 유지하면서도 구체적이고 명확하며 **과학적으로 표현된** 검색 질의를 생성하는 것입니다.\n        변형 기준:\n        - 핵심 개념 강조: standalone_query에서 다루고 있는 핵심 개념을 더 구체적이고 명확하게 설명하여 검색 질의를 풍부하게 만듭니다.\n        - 관련 정보 추가: reference content의 내용을 참고하여 추가적으로 포함될 수 있는 정보나 맥락을 질의에 반영합니다.\n        - **과학적 용어와 표현 사용**: 질문을 과학적인 어조로 재구성하고, 적절한 과학 용어를 활용합니다.\n        - 간결한 질문 유지: 너무 길거나 복잡하지 않도록 질의를 간결하게 유지합니다.\n        예시:\n        질문: "기체의 부피나 형태가 왜 일정하지 않을까?"\n        응답: "기체의 부피와 형태가 일정하지 않은 이유는 무엇인가요?"\n    '}, {'role': 'user', 'content': '\n        질문: "나무의 분류에 대해 조사해 보기 위한 방법은?"\n\n        위의 지침을 따라 질문을 변형하여 6개의 updated_query를 생성하세요.\n        각 transformed_query는 다음과 같은 형식을 따라주세요:\n\n        transformed_query 1: 변형된 질문 1\n        transformed_query 2: 변형된 질문 2\n        transformed_query 3: 변형된 질문 3\n        transformed_query 4: 변형된 질문 4\n        transformed_query 5: 변형된 질문 5\n        t

In [48]:
new_queries

{78: ['나무의 분류 체계를 과학적으로 조사하는 방법은 무엇인가요?',
  '식물학적 기준에 따른 나무의 분류 방법을 연구하는 절차는 어떻게 되나요?',
  '나무의 계통 분류학적 특성을 분석하기 위한 연구 방법은 무엇인가요?',
  '나무의 종 분류를 위한 과학적 접근 방법에는 어떤 것들이 있나요?',
  '나무의 분류학적 체계를 조사하기 위한 효율적인 연구 방법은 무엇인가요?',
  '나무의 생물학적 분류를 체계적으로 연구하는 방법론은 무엇인가요?'],
 213: ['각 국가별 공교육에 대한 정부 지출 비율과 그 변화 추이는 어떻게 되나요?',
  '세계 주요 국가들의 GDP 대비 공교육 지출 비율은 어떻게 비교되나요?',
  '국가별 1인당 공교육 지출액과 그에 따른 교육 성과 간의 상관관계는 무엇인가요?',
  '선진국과 개발도상국의 공교육 지출 구조와 그 차이점은 무엇인가요?',
  '각국의 공교육 지출이 교육의 질에 미치는 영향에 대한 연구 결과는 무엇인가요?',
  '공교육에 대한 국가별 재정 투입이 교육 접근성 및 평등성에 미치는 영향은 어떻게 분석되나요?'],
 107: ['기억 상실증의 주요 신경학적 원인은 무엇인가요?',
  '기억 상실증을 유발하는 뇌 손상 또는 질환의 기전은 어떻게 되나요?',
  '기억 상실증의 발생에 기여하는 생리학적 및 심리적 요인은 무엇인가요?',
  '기억 상실증을 초래하는 외상성 뇌 손상과 관련된 병리학적 변화는 무엇인가요?',
  '기억 상실증의 발병과 관련된 신경전달물질의 역할은 무엇인가요?',
  '기억 상실증을 유발할 수 있는 신경계 질환 또는 약물의 종류는 무엇인가요?'],
 81: ['통학 버스가 학생들의 이동에 미치는 경제적, 환경적 이점은 무엇인가요?',
  '통학 버스가 학생들의 안전과 교통 혼잡 완화에 기여하는 방식은 무엇인가요?',
  '통학 버스가 학생들의 학업 성취도와 출석률에 미치는 영향은 무엇인가요?',
  '통학 버스의 운영이 지역 사회의 교통 시스템에 미치는 긍정적 효과는 무엇

'소프트웨어 개발 시 인간의 편향을 제거하기 위한 인공지능 및 자동화 기술의 역할은 무엇인가요?'

In [ ]:
with open('/upstage-ai-advanced-ir7/data/eval.jsonl') as f:
    for line in f:
        eval_doc = json.loads(line)
        top_agent_result = top_agent_without_relevance_check(eval_doc['msg'])
        if top_agent_result ==  False:
            new_queries[eval_doc['eval_id']] = [eval_doc['msg'][0]['content'] for _ in range(6)]
        else: 
            new_queries[eval_doc['eval_id']] = top_agent_result

In [55]:
with open('/upstage-ai-advanced-ir7/data/eval.jsonl') as f, open('/upstage-ai-advanced-ir7/data/eval_6.jsonl', "w") as of:
    for line in f:
        eval_doc = json.loads(line)
        
        output = {"eval_id": eval_doc['eval_id'], "msg": [{"role" : "user", "content": new_queries[eval_doc['eval_id']][5]}]}

        of.write(f'{json.dumps(output, ensure_ascii=False)}\n')

{"eval_id": 78, "msg": [{"role": "user", "content": "나무의 분류에 대해 조사해 보기 위한 방법은?"}]}

# 새로운 raranker

In [8]:
def llm_reranking(query, documents):
    """
    LLM을 사용하여 문서들을 재정렬합니다.
    """
    # 문서 내용 준비
    check_doc = ''
    for i, doc in enumerate(documents):
        check_doc += f"문서 {i+1}:\n{doc_mapping[doc[0]]['content']}\n\n"
    
    # 단계 1: 모든 문서에 대한 평가 얻기 (점수 없이)
    system_message = {
        "role": "system",
        "content": "당신은 질문에 대한 각 문서의 적합성을 평가하는 전문가입니다."
    }
    user_message = {
        "role": "user",
        "content": f'''
        다음의 규칙을 따르세요:

        1. 주어진 질문과 각 문서의 내용을 비교하여, 각 문서가 질문에 얼마나 적합한지 평가하세요.
        2. 각 문서를 보고 질문에 답할 수 있는 내용을 가지고 있고 이 문서를 인용 했을때 이 질문에 답할 수 있는가를 생각해 보고 인용할 이유를 긍정적 부정적 내용을 모두 넣어 작성하세요.
        2. 각 문서별로 간략한 이유를 한두 문장으로 작성하세요.
        3. 아래의 형식을 따라주세요:

        문서 1:
        이유: 간략한 이유

        문서 2:
        이유: 간략한 이유

        ...

        질문: "{query}"

        문서들:
        {check_doc}
        '''
    }
    messages = [system_message, user_message]
    
    # LLM 호출 - 단계 1
    result = client_gpt.chat.completions.create(
        model="chatgpt-4o-latest",
        messages=messages,
        temperature=0
    )
    
    evaluations = result.choices[0].message.content
    
    # return evaluations
    print(evaluations)

    # 단계 2: 평가 결과와 문서 내용, 질문을 기반으로 순위 결정
    system_message = {
        "role": "system",
        "content": "당신은 문서의 평가와 내용을 바탕으로 순위를 결정하는 전문가입니다."
    }
    user_message = {
        "role": "user",
        "content": f'''
        다음의 규칙을 따르세요:

        1. 아래에 제공된 문서 평가 결과와 문서 내용을 기반으로, 질문에 가장 적합한 문서들을 순서대로 나열하세요.
        2. 출력은 문서 번호로만 구성된 리스트 형태로 해주세요. 예를 들어 [1, 3, 2, 4]
        3. 리스트에 들어갈 문서 번호는 1부터 시작하며, 문서들의 순서를 중요도에 따라 배열하세요.
        4. 리스트 외의 다른 정보를 출력하지 마세요. 리스트 이외의 내용은 모델이 자동으로 무시해야 합니다.

        질문: "{query}"

        문서 평가 결과:
        {evaluations}

        문서들:
        {check_doc}
        '''
    }
    messages = [system_message, user_message]
    
    print(messages)
    
    # LLM 호출 - 단계 2
    result = client_gpt.chat.completions.create(
        model="chatgpt-4o-latest",
        messages=messages,
        temperature=0
    )
    
    ranked_list = result.choices[0].message.content.strip()
    print(f"변환된 쿼리: {ranked_list}")
    return ranked_list


In [26]:
def llm_reranking(query, documents):
    """
    LLM을 사용하여 문서들을 재정렬합니다.
    """
    # 문서 내용 준비
    check_doc = ''
    for i, doc in enumerate(documents):
        check_doc += f"문서 {i+1}:\n{doc_mapping[doc[0]]['content']}\n\n"
    
    # 단계 1: 모든 문서에 대한 평가 얻기 (점수 없이)
    system_message = {
        "role": "system",
        "content": "당신은 질문에 대한 각 문서의 적합성을 평가하는 전문가입니다."
    }
    user_message = {
        "role": "user",
        "content": f'''
        다음의 규칙을 따르세요:

        1. 주어진 질문과 각 문서의 내용을 비교하여, 각 문서가 질문에 얼마나 적합한지 평가하세요.
        2. 각 문서를 보고 질문에 답할 수 있는 내용을 가지고 있고 이 문서를 인용했을 때 이 질문에 답할 수 있는가를 생각해 보고 인용할 이유를 긍정적 부정적 내용을 모두 넣어 작성하세요.
        3. 각 문서별로 간략한 이유를 한두 문장으로 작성하세요.
        4. 아래의 형식을 따라주세요:

        문서 1:
        이유: 간략한 이유

        문서 2:
        이유: 간략한 이유

        ...

        질문: "{query}"

        문서들:
        {check_doc}
        '''
    }
    messages = [system_message, user_message]
    
    # LLM 호출 - 단계 1
    result = client_gpt.chat.completions.create(
        model="chatgpt-4o-latest",
        messages=messages,
        temperature=0
    )
    
    evaluations = result.choices[0].message.content
    
    print(evaluations)
    
    # 단계 1.5: 각 문서로부터 질문에 대한 답변 생성
    generated_answers = []
    for i, doc in enumerate(documents):
        system_message = {
            "role": "system",
            "content": "당신은 주어진 문서를 기반으로 질문에 답변하는 전문가입니다."
        }
        user_message = {
            "role": "user",
            "content": f'''
            다음의 규칙을 따르세요:

            1. 제공된 문서의 내용만을 기반으로 질문에 대한 답변을 작성하세요.
            2. 문서에 질문에 대한 정보가 없으면 "해당 문서에는 질문에 대한 답변이 없습니다."라고 적으세요.
            3. 답변은 간결하고 명확하게 작성하세요.

            문서 {i+1}:
            {doc_mapping[doc[0]]['content']}

            질문: "{query}"
            '''
        }
        messages = [system_message, user_message]
        
        # LLM 호출 - 단계 1.5
        result = client_gpt.chat.completions.create(
            model="chatgpt-4o-latest",
            messages=messages,
            temperature=0
        )
        answer = result.choices[0].message.content.strip()
        generated_answers.append(f"문서 {i+1}의 답변:\n{answer}\n")
    
    # 단계 2를 위한 생성된 답변 문자열 생성
    generated_answers_str = "\n".join(generated_answers)
    
    # 단계 2: 평가 결과, 생성된 답변, 문서 내용, 질문을 기반으로 순위 결정
    system_message = {
        "role": "system",
        "content": "당신은 문서의 평가와 내용을 바탕으로 순위를 결정하는 전문가입니다."
    }
    user_message = {
        "role": "user",
        "content": f'''
        다음의 규칙을 따르세요:

        1. 아래에 제공된 문서 평가 결과, 각 문서로부터 생성된 답변, 그리고 문서 내용을 기반으로, 질문에 가장 적합한 문서들을 순서대로 나열하세요.
        2. 출력은 문서 번호로만 구성된 리스트 형태로 해주세요. 예를 들어 [1, 3, 2, 4]
        3. 리스트에 들어갈 문서 번호는 1부터 시작하며, 문서들의 순서를 중요도에 따라 배열하세요.
        4. 리스트 외의 다른 정보를 출력하지 마세요.

        질문: "{query}"

        문서 평가 결과:
        {evaluations}

        생성된 답변들:
        {generated_answers_str}

        문서들:
        {check_doc}
        '''
    }
    messages = [system_message, user_message]
    
    print(messages)
    
    # LLM 호출 - 단계 2
    result = client_gpt.chat.completions.create(
        model="chatgpt-4o-latest",
        messages=messages,
        temperature=0
    )
    
    ranked_list = result.choices[0].message.content.strip()
    print(f"변환된 쿼리: {ranked_list}")
    return ranked_list


In [8]:
def llm_reranking(query, documents):
    """
    LLM을 사용하여 문서들을 재정렬합니다.
    """
    # 문서 내용 준비
    check_doc = ''
    for i, doc in enumerate(documents):
        check_doc += f"문서 {i+1}:\n{doc_mapping[doc[0]]['content']}\n\n"
    
    # 단계 1: 모든 문서에 대한 평가 얻기 (점수 없이)
    system_message = {
        "role": "system",
        "content": "당신은 질문에 대한 각 문서의 적합성을 평가하는 전문가입니다."
    }
    user_message = {
        "role": "user",
        "content": f'''
        다음의 규칙을 따르세요:

        1. 주어진 질문과 각 문서의 내용을 비교하여, 각 문서가 질문을 **과학적으로 얼마나 잘 설명할 수 있는지** 평가하세요.
        2. 각 문서를 보고 질문에 답할 수 있는 내용을 가지고 있고, 이 문서를 인용했을 때 이 질문에 **과학적으로 정확하고 상세하게** 답할 수 있는가를 생각해 보고, 인용할 이유를 긍정적 부정적 내용을 모두 포함하여 작성하세요.
        3. 조금이라도 관련이 있으면 관련이 있는 이유를 작성하세요.
        4. 각 문서별로 간략한 이유를 한두 문장으로 작성하세요.
        5. 아래의 형식을 따라주세요:

        문서 1:
        이유: 간략한 이유

        문서 2:
        이유: 간략한 이유

        ...

        질문: "{query}"

        문서들:
        {check_doc}
        '''
    }

    messages = [system_message, user_message]
    
    # LLM 호출 - 단계 1
    result = client_gpt.chat.completions.create(
        model="chatgpt-4o-latest",
        messages=messages,
        temperature=0
    )
    
    evaluations = result.choices[0].message.content
    
    print(evaluations)
    
    # 단계 1.5: 질문 변형하기 (HYDE 스타일)
    # HYDE를 위한 지침 설정
    persona_function_calling = """
        당신은 사용자의 질문을 분석하여 가장 적절한 updated_query를 제안하는 전문가입니다.
        변형의 목표는 query와 reference content의 높은 연관성을 유지하면서도 구체적이고 명확하며 **과학적으로 표현된** 검색 질의를 생성하는 것입니다.
        변형 기준:
        - 핵심 개념 강조: standalone_query에서 다루고 있는 핵심 개념을 더 구체적이고 명확하게 설명하여 검색 질의를 풍부하게 만듭니다.
        - 관련 정보 추가: reference content의 내용을 참고하여 추가적으로 포함될 수 있는 정보나 맥락을 질의에 반영합니다.
        - **과학적 용어와 표현 사용**: 질문을 과학적인 어조로 재구성하고, 적절한 과학 용어를 활용합니다.
        - 간결한 질문 유지: 너무 길거나 복잡하지 않도록 질의를 간결하게 유지합니다.
        예시:
        질문: "기체의 부피나 형태가 왜 일정하지 않을까?"
        응답: "기체의 부피와 형태가 일정하지 않은 이유는 무엇인가요?"
    """

    # 질문 변형을 위한 메시지 구성
    system_message = {
        "role": "system",
        "content": persona_function_calling
    }
    user_message = {
        "role": "user",
        "content": f'''
        질문: "{query}"
        reference content:
        {check_doc}

        위의 지침을 따라 질문을 변형하여 6개의 updated_query를 생성하세요.
        각 updated_query는 다음과 같은 형식을 따라주세요:

        updated_query 1: 변형된 질문 1
        updated_query 2: 변형된 질문 2
        updated_query 3: 변형된 질문 3
        updated_query 4: 변형된 질문 4
        updated_query 5: 변형된 질문 5
        updated_query 6: 변형된 질문 6
        '''
    }

    messages = [system_message, user_message]
    
    # LLM 호출 - 단계 1.5: 질문 변형하기
    result = client_gpt.chat.completions.create(
        model="chatgpt-4o-latest",
        messages=messages,
        temperature=0
    )
    transformed_queries_text = result.choices[0].message.content.strip()
    
    # 변형된 질문들을 파싱하여 리스트로 저장
    transformed_queries = []
    for line in transformed_queries_text.split('\n'):
        if line.startswith('updated_query'):
            _, query_text = line.split(':', 1)
            transformed_queries.append(query_text.strip())
    
    # 원본 질문도 포함하여 총 5개의 질문
    all_queries = [query] + transformed_queries
    
    print(all_queries)
    # 단계 1.6: HYDE 답변 생성
    hyde_answers = []
    for idx, q in enumerate(all_queries):
        system_message = {
            "role": "system",
            "content": "당신은 주어진 질문에 대해 **과학적으로 정확하고 상세한 답변을 제공하는 전문가**입니다."
        }
        user_message = {
            "role": "user",
            "content": f'''
            다음의 규칙을 따르세요:

            1. 질문에 대한 답변을 작성하세요.
            2. 답변은 **과학적으로 정확하고 상세하게** 작성하세요.
            3. **전문적인 과학 용어를 사용하고, 관련 개념을 설명하세요.**
            4. 답변은 간결하고 명확하게 작성하세요.

            질문 {idx+1}: "{q}"
            '''
        }

        messages = [system_message, user_message]
        
        # LLM 호출 - 단계 1.6
        result = client_gpt.chat.completions.create(
            model="chatgpt-4o-latest",
            messages=messages,
            temperature=0
        )
        answer = result.choices[0].message.content.strip()
        hyde_answers.append(f"질문 {idx+1}의 HYDE 답변:\n{answer}\n")
    
    # 단계 2: HYDE 답변들, 평가 결과, 문서 내용, 질문을 기반으로 순위 결정
    hyde_answers_str = "\n".join(hyde_answers)
    
    system_message = {
        "role": "system",
        "content": "당신은 문서의 평가와 HYDE 답변을 바탕으로 질문에 관련된 순위를 결정하는 전문가입니다."
    }
    user_message = {
        "role": "user",
        "content": f'''
        다음의 규칙을 따르세요:

        1. 아래에 제공된 문서 평가 결과, HYDE 답변들, 그리고 문서 내용을 기반으로, **질문에 과학적으로 가장 정확하고 상세하게 답변할 수 있는** 문서들을 순서대로 나열하세요.
        2. 문서가 많아서 message가 매우 길어지는데 잊지 않도록 처음부터 끝까지 다시한번 보면서 생각하고 답하세요
        3. **질문에서 요구하는 용어가 있는지 확인하는것도 좋은 방식이 될 수 있다**.
        4. **가장 중요한 것은 질문에 적합한 답을 할 수 있느냐 입니다**.
        5. 출력은 문서 번호로만 구성된 리스트 형태로 해주세요. 예를 들어 [1, 3, 2, 4]
        6. 리스트에 들어갈 문서 번호는 1부터 시작하며, 문서들의 순서를 중요도에 따라 배열하세요.
        7. 리스트 외의 다른 정보를 출력하지 마세요.


        질문: "{query}"

        문서 평가 결과:
        {evaluations}

        HYDE 답변들:
        {hyde_answers_str}

        문서들:
        {check_doc}
    '''
    }
    messages = [system_message, user_message]
    
    print(messages)
    
    # LLM 호출 - 단계 2
    result = client_gpt.chat.completions.create(
        model="chatgpt-4o-latest",
        messages=messages,
        temperature=0
    )
    
    ranked_list = result.choices[0].message.content.strip()
    print(f"변환된 쿼리: {ranked_list}")
    return ranked_list


In [10]:
def llm_reranking(query, documents):
    """
    LLM을 사용하여 문서들을 재정렬합니다.
    """
    # 문서 내용 준비
    check_doc = ''
    for i, doc in enumerate(documents):
        check_doc += f"문서 {i+1}:\n{doc_mapping[doc[0]]['content']}\n\n"
    
    # 단계 1: 모든 문서에 대한 평가 얻기 (점수 없이)
    system_message = {
        "role": "system",
        "content": "당신은 질문에 대한 각 문서의 적합성을 평가하는 전문가입니다."
    }
    user_message = {
        "role": "user",
        "content": f'''
    다음의 규칙을 따르세요:

    1. 주어진 질문과 각 문서의 내용을 비교하여, 각 문서가 질문을 **과학적으로 얼마나 잘 설명할 수 있는지** 평가하세요.
    2. 각 문서를 보고 질문에 답할 수 있는 내용을 가지고 있고, 이 문서를 인용했을 때 이 질문에 **과학적으로 정확하고 상세하게** 답할 수 있는가를 생각해 보고, 인용할 이유를 긍정적 부정적 내용을 모두 포함하여 작성하세요.
    3. 조금이라도 관련이 있으면 관련이 있는 이유를 작성하세요.
    4. 각 문서별로 간략한 이유를 한두 문장으로 작성하세요.
    5. 아래의 형식을 따라주세요:

    문서 1:
    이유: 간략한 이유

    문서 2:
    이유: 간략한 이유

    ...

    질문: "{query}"

    문서들:
    {check_doc}
    '''
    }

    messages = [system_message, user_message]
    
    # LLM 호출 - 단계 1
    result = client_gpt.chat.completions.create(
        model="chatgpt-4o-latest",
        messages=messages,
        temperature=0
    )
    
    evaluations = result.choices[0].message.content
    
    print(evaluations)
    
    # 단계 1.5 제거: 추가적인 질문 생성 없이 원본 질문으로 진행

    # 단계 1.6: HYDE 답변 생성
    hyde_answers = []
    system_message = {
        "role": "system",
        "content": "당신은 주어진 질문에 대해 **과학적으로 정확하고 상세한 답변을 제공하는 전문가**입니다."
    }
    user_message = {
        "role": "user",
        "content": f'''
    다음의 규칙을 따르세요:

    1. 질문에 대한 답변을 작성하세요.
    2. 답변은 **과학적으로 정확하고 상세하게** 작성하세요.
    3. **전문적인 과학 용어를 사용하고, 관련 개념을 설명하세요.**
    4. 답변은 간결하고 명확하게 작성하세요.

    질문: "{query}"
    '''
    }

    messages = [system_message, user_message]
    
    # LLM 호출 - 단계 1.6
    result = client_gpt.chat.completions.create(
        model="chatgpt-4o-latest",
        messages=messages,
        temperature=0
    )
    answer = result.choices[0].message.content.strip()
    hyde_answers.append(f"HYDE 답변:\n{answer}\n")
    
    # 단계 2: HYDE 답변, 평가 결과, 문서 내용, 질문을 기반으로 순위 결정
    hyde_answers_str = "\n".join(hyde_answers)
    
    system_message = {
        "role": "system",
        "content": "당신은 문서의 평가와 HYDE 답변을 바탕으로 질문에 관련된 순위를 결정하는 전문가입니다."
    }
    user_message = {
        "role": "user",
        "content": f'''
    다음의 규칙을 따르세요:

    1. 아래에 제공된 문서 평가 결과, HYDE 답변, 그리고 문서 내용을 기반으로, **질문에 과학적으로 가장 정확하고 상세하게 답변할 수 있는** 문서들을 순서대로 나열하세요.
    2. 문서가 많아서 메시지가 매우 길어지는데, 잊지 않도록 처음부터 끝까지 다시 한 번 보면서 생각하고 답하세요.
    3. **질문에서 요구하는 용어가 있는지 확인하는 것도 좋은 방식이 될 수 있습니다**.
    4. **가장 중요한 것은 질문에 적합한 답을 할 수 있느냐입니다**.
    5. 출력은 문서 번호로만 구성된 리스트 형태로 해주세요. 예를 들어 [1, 3, 2, 4]
    6. 리스트에 들어갈 문서 번호는 1부터 시작하며, 문서들의 순서를 중요도에 따라 배열하세요.
    7. 리스트 외의 다른 정보를 출력하지 마세요.


    질문: "{query}"

    문서 평가 결과:
    {evaluations}

    HYDE 답변:
    {hyde_answers_str}

    문서들:
    {check_doc}
    '''
    }
    messages = [system_message, user_message]
    
    print(messages)
    
    # LLM 호출 - 단계 2
    result = client_gpt.chat.completions.create(
        model="chatgpt-4o-latest",
        messages=messages,
        temperature=0
    )
    
    ranked_list = result.choices[0].message.content.strip()
    print(f"변환된 쿼리: {ranked_list}")
    return ranked_list


In [12]:
def llm_reranking(query, documents):
    """
    LLM을 사용하여 문서들을 재정렬합니다.
    """
    # 문서 내용 준비
    check_doc = ''
    for i, doc in enumerate(documents):
        check_doc += f"문서 {i+1}:\n{doc_mapping[doc[0]]['content']}\n\n"
    
    # 단계 1: 모든 문서에 대한 평가 얻기
    system_message = {
        "role": "system",
        "content": "당신은 질문에 대한 각 문서의 적합성을 평가하는 전문가입니다."
    }
    user_message = {
        "role": "user",
        "content": f'''
    다음의 규칙을 따르세요:

    1. 주어진 질문과 각 문서의 내용을 비교하여, 각 문서가 질문을 **과학적으로 얼마나 잘 설명할 수 있는지** 평가하세요.
    2. 각 문서를 보고 질문에 답할 수 있는 내용을 가지고 있고, 이 문서를 인용했을 때 이 질문에 **과학적으로 정확하고 상세하게** 답할 수 있는가를 생각해 보고, 인용할 이유를 긍정적 부정적 내용을 모두 포함하여 작성하세요.
    3. 조금이라도 관련이 있으면 관련이 있는 이유를 작성하세요.
    4. 각 문서별로 간략한 이유를 한두 문장으로 작성하세요.
    5. 아래의 형식을 따라주세요:

    문서 1:
    이유: 간략한 이유

    문서 2:
    이유: 간략한 이유

    ...

    질문: "{query}"

    문서들:
    {check_doc}
    '''
    }

    messages = [system_message, user_message]
    
    # LLM 호출 - 단계 1
    result = client_gpt.chat.completions.create(
        model="chatgpt-4o-latest",
        messages=messages,
        temperature=0
    )
    
    evaluations = result.choices[0].message.content
    
    print(evaluations)
    
    # 단계 1.6: HYDE 답변 생성
    hyde_answers = []
    system_message = {
        "role": "system",
        "content": "당신은 주어진 질문에 대해 **과학적으로 정확하고 상세한 답변을 제공하는 전문가**입니다."
    }
    user_message = {
        "role": "user",
        "content": f'''
    다음의 규칙을 따르세요:

    1. 질문에 대한 답변을 작성하세요.
    2. 답변은 **과학적으로 정확하고 상세하게** 작성하세요.
    3. **전문적인 과학 용어를 사용하고, 관련 개념을 설명하세요.**
    4. 답변은 간결하고 명확하게 작성하세요.

    질문: "{query}"
    '''
    }

    messages = [system_message, user_message]
    
    # LLM 호출 - 단계 1.6
    result = client_gpt.chat.completions.create(
        model="chatgpt-4o-latest",
        messages=messages,
        temperature=0
    )
    answer = result.choices[0].message.content.strip()
    hyde_answers.append(f"HYDE 답변:\n{answer}\n")
    
    # 단계 2: HYDE 답변, 평가 결과, 문서 내용, 질문을 기반으로 순위와 이유 생성
    hyde_answers_str = "\n".join(hyde_answers)
    
    system_message = {
        "role": "system",
        "content": "당신은 문서의 평가와 HYDE 답변을 바탕으로 질문에 관련된 순위를 결정하는 전문가입니다."
    }
    user_message = {
        "role": "user",
        "content": f'''
    다음의 규칙을 따르세요:

    1. 아래에 제공된 문서 평가 결과, HYDE 답변, 그리고 문서 내용을 기반으로, **질문에 과학적으로 가장 정확하고 상세하게 답변할 수 있는** 문서들을 순서대로 나열하고, 각 문서에 대해 순위를 매긴 이유를 간략히 작성하세요.
    2. 문서가 많아서 메시지가 매우 길어지는데, 잊지 않도록 처음부터 끝까지 다시 한 번 보면서 생각하고 답하세요.
    3. **질문에서 요구하는 용어가 있는지 확인하는 것도 좋은 방식이 될 수 있습니다**.
    4. **가장 중요한 것은 질문에 적합한 답을 할 수 있느냐입니다**.
    5. 아래의 형식을 따라주세요:

    문서 순위:
    1위 문서 번호: 이유
    2위 문서 번호: 이유
    3위 문서 번호: 이유
    ...

    질문: "{query}"

    문서 평가 결과:
    {evaluations}

    HYDE 답변:
    {hyde_answers_str}

    문서들:
    {check_doc}
    '''
    }
    messages = [system_message, user_message]
    
    # LLM 호출 - 단계 2
    result = client_gpt.chat.completions.create(
        model="chatgpt-4o-latest",
        messages=messages,
        temperature=0
    )
    
    ranking_with_reasons = result.choices[0].message.content.strip()
    print(f"문서 순위와 이유:\n{ranking_with_reasons}")
    
    # 단계 3: 순위와 이유를 바탕으로 문서 번호 리스트 생성
    system_message = {
        "role": "system",
        "content": "당신은 문서 순위를 바탕으로 최종 리스트를 만드는 전문가입니다."
    }
    user_message = {
        "role": "user",
        "content": f'''
    다음의 규칙을 따르세요:

    1. 아래에 제공된 문서 순위와 이유를 바탕으로, 문서 번호로만 구성된 리스트를 만드세요.
    2. 출력은 문서 번호로만 구성된 리스트 형태로 해주세요. 예를 들어 [1, 3, 2, 4]
    3. 리스트 외의 다른 정보를 출력하지 마세요.

    문서 순위와 이유:
    {ranking_with_reasons}
    '''
    }
    messages = [system_message, user_message]
    
    # LLM 호출 - 단계 3
    result = client_gpt.chat.completions.create(
        model="chatgpt-4o-latest",
        messages=messages,
        temperature=0
    )
    
    ranked_list = result.choices[0].message.content.strip()
    print(f"최종 변환된 리스트: {ranked_list}")
    return ranked_list


In [14]:
def llm_reranking(query, documents):
    """
    LLM을 사용하여 문서들을 재정렬합니다.
    """
    # 문서 내용 준비
    check_doc = ''
    for i, doc in enumerate(documents):
        check_doc += f"문서 {i+1}:\n{doc_mapping[doc[0]]['content']}\n\n"
    
    # 단계 1: 모든 문서에 대한 평가 얻기
    system_message = {
        "role": "system",
        "content": "당신은 질문에 대한 각 문서의 적합성을 평가하는 전문가입니다."
    }
    user_message = {
        "role": "user",
        "content": f'''
    다음의 규칙을 따르세요:

    1. 주어진 질문과 각 문서의 내용을 비교하여, 각 문서가 질문을 **과학적으로 얼마나 잘 설명할 수 있는지** 평가하세요.
    2. 각 문서를 보고 질문에 답할 수 있는 내용을 가지고 있고, 이 문서를 인용했을 때 이 질문에 **과학적으로 정확하고 상세하게** 답할 수 있는가를 생각해 보고, 인용할 이유를 긍정적 부정적 내용을 모두 포함하여 작성하세요.
    3. 조금이라도 관련이 있으면 관련이 있는 이유를 작성하세요.
    4. 각 문서별로 간략한 이유를 한두 문장으로 작성하세요.
    5. 아래의 형식을 따라주세요:

    문서 1:
    이유: 간략한 이유

    문서 2:
    이유: 간략한 이유

    ...

    질문: "{query}"

    문서들:
    {check_doc}
    '''
    }

    messages = [system_message, user_message]
    
    # LLM 호출 - 단계 1
    result = client_gpt.chat.completions.create(
        model="chatgpt-4o-latest",
        messages=messages,
        temperature=0
    )
    
    evaluations = result.choices[0].message.content
    
    print(evaluations)
    
    # HYDE 부분 제거: HYDE 답변 생성 단계 삭제
    
    # 단계 2: 평가 결과, 문서 내용, 질문을 기반으로 순위와 이유 생성
    system_message = {
        "role": "system",
        "content": "당신은 문서의 평가를 바탕으로 질문에 관련된 순위를 결정하는 전문가입니다."
    }
    user_message = {
        "role": "user",
        "content": f'''
    다음의 규칙을 따르세요:

    1. 아래에 제공된 문서 평가 결과와 문서 내용을 기반으로, **질문에 과학적으로 가장 정확하고 상세하게 답변할 수 있는** 문서들을 순서대로 나열하고, 각 문서에 대해 순위를 매긴 이유를 간략히 작성하세요.
    2. 문서가 많아서 메시지가 매우 길어지는데, 잊지 않도록 처음부터 끝까지 다시 한 번 보면서 생각하고 답하세요.
    3. **질문에서 요구하는 용어가 있는지 확인하는 것도 좋은 방식이 될 수 있습니다**.
    4. **가장 중요한 것은 질문에 적합한 답을 할 수 있느냐입니다**.
    5. 아래의 형식을 따라주세요:

    문서 순위:
    1위 문서 번호: 이유
    2위 문서 번호: 이유
    3위 문서 번호: 이유
    ...

    질문: "{query}"

    문서 평가 결과:
    {evaluations}

    문서들:
    {check_doc}
    '''
    }
    messages = [system_message, user_message]
    
    # LLM 호출 - 단계 2
    result = client_gpt.chat.completions.create(
        model="chatgpt-4o-latest",
        messages=messages,
        temperature=0
    )
    
    ranking_with_reasons = result.choices[0].message.content.strip()
    print(f"문서 순위와 이유:\n{ranking_with_reasons}")
    
    # 단계 3: 순위와 이유를 바탕으로 문서 번호 리스트 생성
    system_message = {
        "role": "system",
        "content": "당신은 문서 순위를 바탕으로 최종 리스트를 만드는 전문가입니다."
    }
    user_message = {
        "role": "user",
        "content": f'''
    다음의 규칙을 따르세요:

    1. 아래에 제공된 문서 순위와 이유를 바탕으로, 문서 번호로만 구성된 리스트를 만드세요.
    2. 출력은 문서 번호로만 구성된 리스트 형태로 해주세요. 예를 들어 [1, 3, 2, 4]
    3. 리스트 외의 다른 정보를 출력하지 마세요.

    문서 순위와 이유:
    {ranking_with_reasons}
    '''
    }
    messages = [system_message, user_message]
    
    # LLM 호출 - 단계 3
    result = client_gpt.chat.completions.create(
        model="chatgpt-4o-latest",
        messages=messages,
        temperature=0
    )
    
    ranked_list = result.choices[0].message.content.strip()
    print(f"최종 변환된 리스트: {ranked_list}")
    return ranked_list


In [18]:
def llm_reranking(query, documents):
    """
    LLM을 사용하여 문서들을 재정렬합니다.
    """
    # 문서 내용 준비
    check_doc = ''
    for i, doc in enumerate(documents):
        check_doc += f"문서 {i+1}:\n{doc_mapping[doc[0]]['content']}\n\n"
    
    # 단계 1: 모든 문서에 대한 평가 얻기
    system_message = {
        "role": "system",
        "content": "당신은 질문에 대한 각 문서의 적합성을 평가하는 전문가입니다."
    }
    user_message = {
        "role": "user",
        "content": f'''
    다음의 규칙을 따르세요:

    1. 주어진 질문과 각 문서의 내용을 비교하여, 각 문서가 질문을 **과학적으로 얼마나 잘 설명할 수 있는지** 평가하세요.
    2. 각 문서를 보고 질문에 답할 수 있는 내용을 가지고 있고, 이 문서를 인용했을 때 이 질문에 **과학적으로 정확하고 상세하게** 답할 수 있는가를 생각해 보고, 인용할 이유를 긍정적 부정적 내용을 모두 포함하여 작성하세요.
    3. 조금이라도 관련이 있으면 관련이 있는 이유를 작성하세요.
    4. 각 문서별로 간략한 이유를 한두 문장으로 작성하세요.
    5. 아래의 형식을 따라주세요:

    문서 1:
    이유: 간략한 이유

    문서 2:
    이유: 간략한 이유

    ...

    질문: "{query}"

    문서들:
    {check_doc}
    '''
    }

    messages = [system_message, user_message]
    
    # LLM 호출 - 단계 1
    result = client_gpt.chat.completions.create(
        model="chatgpt-4o-latest",
        messages=messages,
        temperature=0
    )
    
    evaluations = result.choices[0].message.content
    
    print(evaluations)
    
    # 단계 1.5: 질문 변형하기
    persona_function_calling = """
    당신은 사용자의 질문을 분석하여 가장 적절한 updated_query를 제안하는 전문가입니다.
    변형의 목표는 query와 reference content의 높은 연관성을 유지하면서도 구체적이고 명확하며 **과학적으로 표현된** 검색 질의를 생성하는 것입니다.
    변형 기준:
    - 핵심 개념 강조: standalone_query에서 다루고 있는 핵심 개념을 더 구체적이고 명확하게 설명하여 검색 질의를 풍부하게 만듭니다.
    - 관련 정보 추가: reference content의 내용을 참고하여 추가적으로 포함될 수 있는 정보나 맥락을 질의에 반영합니다.
    - **과학적 용어와 표현 사용**: 질문을 과학적인 어조로 재구성하고, 적절한 과학 용어를 활용합니다.
    - 간결한 질문 유지: 너무 길거나 복잡하지 않도록 질의를 간결하게 유지합니다.
    예시:
    질문: "기체의 부피나 형태가 왜 일정하지 않을까?"
    응답: "기체의 부피와 형태가 일정하지 않은 이유는 무엇인가요?"
    """

    # 질문 변형을 위한 메시지 구성
    system_message = {
        "role": "system",
        "content": persona_function_calling
    }
    user_message = {
        "role": "user",
        "content": f'''
    질문: "{query}"
    reference content:
    {check_doc}

    위의 지침을 따라 질문을 변형하여 4개의 updated_query를 생성하세요.
    각 updated_query는 다음과 같은 형식을 따라주세요:

    updated_query 1: 변형된 질문 1
    updated_query 2: 변형된 질문 2
    updated_query 3: 변형된 질문 3
    updated_query 4: 변형된 질문 4
    '''
    }

    messages = [system_message, user_message]
    
    # LLM 호출 - 단계 1.5: 질문 변형하기
    result = client_gpt.chat.completions.create(
        model="chatgpt-4o-latest",
        messages=messages,
        temperature=0
    )
    transformed_queries_text = result.choices[0].message.content.strip()
    
    # 변형된 질문들을 파싱하여 리스트로 저장
    transformed_queries = []
    for line in transformed_queries_text.split('\n'):
        if line.startswith('updated_query'):
            _, query_text = line.split(':', 1)
            transformed_queries.append(query_text.strip())
    
    print("변형된 질문들:")
    print(transformed_queries)
    
    # 단계 2: 평가 결과, 변형된 질문들, 문서 내용, 질문을 기반으로 순위와 이유 생성
    transformed_queries_str = "\n".join([f"변형된 질문 {i+1}: {q}" for i, q in enumerate(transformed_queries)])
    
    system_message = {
        "role": "system",
        "content": "당신은 문서의 평가와 변형된 질문들을 바탕으로 질문에 관련된 순위를 결정하는 전문가입니다."
    }
    user_message = {
        "role": "user",
        "content": f'''
    다음의 규칙을 따르세요:

    1. 아래에 제공된 문서 평가 결과, 변형된 질문들, 그리고 문서 내용을 기반으로, **질문에 과학적으로 가장 정확하고 상세하게 답변할 수 있는** 문서들을 순서대로 나열하고, 각 문서에 대해 순위를 매긴 이유를 간략히 작성하세요.
    2. 변형된 질문들도 고려하여 문서의 적합성을 평가하세요.
    3. 문서가 많아서 메시지가 매우 길어지는데, 잊지 않도록 처음부터 끝까지 다시 한 번 보면서 생각하고 답하세요.
    4. **질문에서 요구하는 용어가 있는지 확인하는 것도 좋은 방식이 될 수 있습니다**.
    5. **가장 중요한 것은 질문에 적합한 답을 할 수 있느냐입니다**.
    6. 아래의 형식을 따라주세요:

    문서 순위:
    1위 문서 번호: 이유
    2위 문서 번호: 이유
    3위 문서 번호: 이유
    ...

    질문: "{query}"

    변형된 질문들:
    {transformed_queries_str}

    문서 평가 결과:
    {evaluations}

    문서들:
    {check_doc}
    '''
    }
    messages = [system_message, user_message]
    
    # LLM 호출 - 단계 2
    result = client_gpt.chat.completions.create(
        model="chatgpt-4o-latest",
        messages=messages,
        temperature=0
    )
    
    ranking_with_reasons = result.choices[0].message.content.strip()
    print(f"문서 순위와 이유:\n{ranking_with_reasons}")
    
    # 단계 3: 순위와 이유를 바탕으로 문서 번호 리스트 생성
    system_message = {
        "role": "system",
        "content": "당신은 문서 순위를 바탕으로 최종 리스트를 만드는 전문가입니다."
    }
    user_message = {
        "role": "user",
        "content": f'''
    다음의 규칙을 따르세요:

    1. 아래에 제공된 문서 순위와 이유를 바탕으로, 문서 번호로만 구성된 리스트를 만드세요.
    2. 출력은 문서 번호로만 구성된 리스트 형태로 해주세요. 예를 들어 [1, 3, 2, 4]
    3. 리스트 외의 다른 정보를 출력하지 마세요.

    문서 순위와 이유:
    {ranking_with_reasons}
    '''
    }
    messages = [system_message, user_message]
    
    # LLM 호출 - 단계 3
    result = client_gpt.chat.completions.create(
        model="chatgpt-4o-latest",
        messages=messages,
        temperature=0
    )
    
    ranked_list = result.choices[0].message.content.strip()
    print(f"최종 변환된 리스트: {ranked_list}")
    return ranked_list


In [44]:
def llm_reranking(query, documents):
    """
    LLM을 사용하여 문서들을 재정렬합니다.
    """
    # 문서 내용 준비
    check_doc = ''
    for i, doc in enumerate(documents):
        check_doc += f"문서 {i+1}:\n{doc_mapping[doc[0]]['content']}\n\n"
    
    # 단계 1: 모든 문서에 대한 평가 얻기
    system_message = {
        "role": "system",
        "content": "당신은 질문에 대한 각 문서의 적합성을 평가하는 전문가입니다."
    }
    user_message = {
        "role": "user",
        "content": f'''
    다음의 규칙을 따르세요:

    1. 주어진 질문과 각 문서의 내용을 비교하여, 각 문서가 질문을 **과학적으로 얼마나 잘 설명할 수 있는지** 평가하세요.
    2. 각 문서를 보고 질문에 답할 수 있는 내용을 가지고 있고, 이 문서를 인용했을 때 이 질문에 **과학적으로 정확하고 상세하게** 답할 수 있는가를 생각해 보고, 인용할 이유를 긍정적 부정적 내용을 모두 포함하여 작성하세요.
    3. 조금이라도 관련이 있으면 관련이 있는 이유를 작성하세요.
    4. 각 문서별로 간략한 이유를 한두 문장으로 작성하세요.
    5. 아래의 형식을 따라주세요:

    문서 1:
    이유: 간략한 이유

    문서 2:
    이유: 간략한 이유

    ...

    질문: "{query}"

    문서들:
    {check_doc}
    '''
    }

    messages = [system_message, user_message]
    
    # LLM 호출 - 단계 1
    result = client_gpt.chat.completions.create(
        model="chatgpt-4o-latest",
        messages=messages,
        temperature=0
    )
    
    evaluations = result.choices[0].message.content
    
    print(evaluations)
    
    # 단계 1.5: 변형된 질문 생성
    persona_function_calling = """
    당신은 사용자의 질문을 분석하여 가장 적절한 updated_query를 제안하는 전문가입니다.
    변형의 목표는 query와 reference content의 높은 연관성을 유지하면서도 구체적이고 명확하며 **과학적으로 표현된** 검색 질의를 생성하는 것입니다.
    변형 기준:
    - 핵심 개념 강조: standalone_query에서 다루고 있는 핵심 개념을 더 구체적이고 명확하게 설명하여 검색 질의를 풍부하게 만듭니다.
    - 관련 정보 추가: reference content의 내용을 참고하여 추가적으로 포함될 수 있는 정보나 맥락을 질의에 반영합니다.
    - **과학적 용어와 표현 사용**: 질문을 과학적인 어조로 재구성하고, 적절한 과학 용어를 활용합니다.
    - 간결한 질문 유지: 너무 길거나 복잡하지 않도록 질의를 간결하게 유지합니다.
    예시:
    질문: "기체의 부피나 형태가 왜 일정하지 않을까?"
    응답: "기체의 부피와 형태가 일정하지 않은 이유는 무엇인가요?"
    """

    # 질문 변형을 위한 메시지 구성
    system_message = {
        "role": "system",
        "content": persona_function_calling
    }
    user_message = {
        "role": "user",
        "content": f'''
    질문: "{query}"
    reference content:
    {check_doc}

    위의 지침을 따라 질문을 변형하여 4개의 updated_query를 생성하세요.
    각 updated_query는 다음과 같은 형식을 따라주세요:

    updated_query 1: 변형된 질문 1
    updated_query 2: 변형된 질문 2
    updated_query 3: 변형된 질문 3
    updated_query 4: 변형된 질문 4
    '''
    }

    messages = [system_message, user_message]
    
    # LLM 호출 - 단계 1.5: 질문 변형하기
    result = client_gpt.chat.completions.create(
        model="chatgpt-4o-latest",
        messages=messages,
        temperature=0
    )
    transformed_queries_text = result.choices[0].message.content.strip()
    
    # 변형된 질문들을 파싱하여 리스트로 저장
    transformed_queries = []
    for line in transformed_queries_text.split('\n'):
        if line.startswith('updated_query'):
            _, query_text = line.split(':', 1)
            transformed_queries.append(query_text.strip())
    
    print("변형된 질문들:")
    print(transformed_queries)
    
    # 단계 2: 평가 결과, 변형된 질문들, 문서 내용, 질문을 기반으로 순위 결정 및 리스트 출력
    transformed_queries_str = "\n".join([f"변형된 질문 {i+1}: {q}" for i, q in enumerate(transformed_queries)])
    
    system_message = {
        "role": "system",
        "content": "당신은 문서의 평가와 변형된 질문들을 바탕으로 질문에 관련된 순위를 결정하는 전문가입니다."
    }
    user_message = {
        "role": "user",
        "content": f'''
    다음의 규칙을 따르세요:

    1. 아래에 제공된 문서 평가 결과, 변형된 질문들, 그리고 문서 내용을 기반으로, **질문에 과학적으로 가장 정확하고 상세하게 답변할 수 있는** 문서들을 순서대로 나열하세요.
    2. 변형된 질문들도 고려하여 문서의 적합성을 평가하세요.
    3. 문서가 많아서 메시지가 매우 길어지는데, 잊지 않도록 처음부터 끝까지 다시 한 번 보면서 생각하고 답하세요.
    4. **질문에서 요구하는 용어가 있는지 확인하는 것도 좋은 방식이 될 수 있습니다**.
    5. **가장 중요한 것은 질문에 적합한 답을 할 수 있느냐입니다**.
    6. 출력은 문서 번호로만 구성된 리스트 형태로 해주세요. 예를 들어 [1, 3, 2, 4]
    7. 리스트에 들어갈 문서 번호는 1부터 시작하며, 문서들의 순서를 중요도에 따라 배열하세요.
    8. 리스트 외의 다른 정보를 출력하지 마세요.

    질문: "{query}"
    
    변형된 질문들:
    {transformed_queries_str}

    문서 평가 결과:
    {evaluations}

    문서들:
    {check_doc}
    
    질문: "{query}" 에 가장 적합한 문서를 찾으세요.
    '''
    }
    messages = [system_message, user_message]
    
    # LLM 호출 - 단계 2
    result = client_gpt.chat.completions.create(
        model="chatgpt-4o-latest",
        messages=messages,
        temperature=0
    )
    
    ranked_list = result.choices[0].message.content.strip()
    print(f"최종 변환된 리스트: {ranked_list}")
    return ranked_list


인간이 2세를 생산할 때 DNA의 결합 과정에 대히 설명해줘.

건설 현장에서 망치로 벽을 치는 이유는?

In [20]:
result = retrieval_with_score('건설 현장에서 망치로 벽을 치는 이유는?')

상위 5개의 문서:


In [52]:
i = 1
for doc in result:

    # print(doc[0])
    print(i, '번째 문서')
    print(doc_mapping[doc[0]]['content'])
    print('*' * 100)
    i += 1
    # print(doc_mapping[doc[0]['content']])

1 번째 문서
한 건설 작업자가 해머로 단단한 철벽을 치려 했지만 벽은 움직이지 않았습니다. 작업자가 사용한 에너지의 일부는 열로 변환되었습니다. 철벽은 고강도 강재로 만들어져 있어서 해머로 강력하게 치더라도 움직이지 않는 것이 정상입니다. 철벽은 건물의 구조를 지탱하고 보호하기 위해 설계되었으며, 그 내부에는 철과 다른 금속들이 견고하게 결합되어 있습니다. 따라서 해머로 치는 작업은 벽을 움직이게 하는 것보다는 에너지를 전달하여 열로 변환시키는 역할을 합니다. 이러한 열 변환은 작업자가 사용한 에너지를 벽에 흡수시키고, 벽의 온도를 조절하는 역할을 합니다. 따라서 작업자가 해머로 철벽을 치는 것은 벽을 움직이게 하는 것보다는 벽과 상호작용하여 열을 발생시키는 과정입니다.
****************************************************************************************************
2 번째 문서
한 목수가 얇은 종이로 나무 조각을 덮었습니다. 그는 덮인 나무 조각을 망치로 쳤습니다. 이 과정에서 충격이 발생하였고, 종이에는 작은 구멍이 생기고 연기 냄새가 났습니다. 이 사건은 역학 에너지에서 열 에너지로 전달되는 에너지 전달 방식을 보여주었습니다. 목수가 망치로 나무 조각을 칠 때, 망치의 운동 에너지가 나무 조각에 전달되어 역학 에너지로 변환됩니다. 이 역학 에너지는 나무 조각과 종이 사이에 전달되어 종이에 충격을 주게 됩니다. 충격으로 인해 종이는 변형되고, 작은 구멍이 생기며 연기 냄새가 나타납니다. 이는 역학 에너지가 종이에 전달되어 열 에너지로 변환되는 과정입니다. 따라서, 이 사건은 역학 에너지에서 열 에너지로 전달되는 에너지 전달 방식을 보여주었을 가능성이 가장 높습니다.
****************************************************************************************************
3 번째 문서
역학적 에너지의

In [56]:
re_ranked = llm_reranking('건설 현장에서 망치로 벽을 치는 이유는?', result)

문서 1:  
이유: 벽을 망치로 치는 과정에서 에너지가 열로 변환된다는 설명이 있지만, 질문의 핵심인 "건설 현장에서 망치로 벽을 치는 이유"와는 직접적인 관련이 없습니다.

문서 2:  
이유: 망치로 나무를 치는 과정에서 에너지가 전달되는 방식에 대한 설명이 있지만, 건설 현장에서 벽을 치는 이유와는 관련이 적습니다.

문서 3:  
이유: 망치로 못을 치는 과정에서 에너지가 전달되는 방식에 대한 설명이 있으며, 이는 건설 현장에서 망치로 벽을 치는 이유와 관련이 있을 수 있습니다. 벽을 치는 것도 물리적 에너지 전달의 한 예로 볼 수 있기 때문입니다.

문서 4:  
이유: 망치로 볼링공을 치는 과정에서 파동이 전달되는 현상을 설명하고 있지만, 건설 현장에서 벽을 치는 이유와는 관련이 없습니다.

문서 5:  
이유: 벽돌의 질량에 대한 설명이지만, 건설 현장에서 망치로 벽을 치는 이유와는 관련이 없습니다.

문서 6:  
이유: 정수 처리장 건설에 대한 내용으로, 건설 현장에서 망치로 벽을 치는 이유와는 관련이 없습니다.

문서 7:  
이유: 망치 사용 시 안전에 대한 설명이지만, 건설 현장에서 망치로 벽을 치는 이유와는 관련이 없습니다.

문서 8:  
이유: 산림 지역에서의 건설 제한에 대한 설명으로, 건설 현장에서 망치로 벽을 치는 이유와는 관련이 없습니다.

문서 9:  
이유: 벽돌의 질량을 측정하는 방법에 대한 설명으로, 건설 현장에서 망치로 벽을 치는 이유와는 관련이 없습니다.

문서 10:  
이유: 쇠못을 연소시키는 방법에 대한 설명으로, 건설 현장에서 망치로 벽을 치는 이유와는 관련이 없습니다.

문서 11:  
이유: 작은 조각들이 결합되는 과정에 대한 설명으로, 건설 현장에서 망치로 벽을 치는 이유와는 관련이 없습니다.

문서 12:  
이유: 금속 막대를 치면 진동이 발생하는 현상에 대한 설명이지만, 건설 현장에서 망치로 벽을 치는 이유와는 관련이 없습니다.

문서 13:  
이유: 망치로 물체를 깨뜨리는 물리적 변화에 대한 

In [26]:
print(re_ranked)

[3, 13, 1, 2, 4, 12]


In [28]:
i = 1
for doc in result:

    # print(doc[0])
    print(i, '번째 문서')
    print(doc_mapping[doc[0]]['content'])
    print('*' * 100)
    i += 1
    # print(doc_mapping[doc[0]['content']])

1 번째 문서
인간 남성과 여성의 생식세포는 같은 수의 염색체를 가지고 있습니다. 남성은 23쌍의 염색체를 가지고 있으며, 여성도 23쌍의 염색체를 가지고 있습니다. 따라서, 일반적인 남성과 여성의 생식세포가 결합하여 자손을 생산할 때, 염색체의 수는 두 배가 됩니다. 이는 염색체의 유전 정보를 조합하여 새로운 개체를 형성하는 과정입니다. 염색체는 DNA를 포함하고 있으며, DNA는 유전 정보를 담고 있는 분자입니다. 따라서, 염색체의 수가 두 배가 되면, 자손은 부모의 유전 정보를 조합하여 새로운 개체를 형성하게 됩니다.
****************************************************************************************************
2 번째 문서
동물들의 성적 번식은 난자와 정자가 결합하는 과정으로 가장 잘 설명됩니다. 이 과정은 생물학적으로 성적 번식이라고도 불리며, 동물들이 자신의 유전 정보를 다음 세대로 전달하는 핵심적인 방법입니다. 난자는 암컷 동물이 생산하며, 정자는 수컷 동물이 생산합니다. 이 두 세포가 만나서 결합하면, 새로운 생명이 탄생합니다. 이러한 성적 번식은 동물들의 다양성을 유지하고 진화를 가능하게 합니다. 성적 번식은 동물들의 생존과 번성에 중요한 역할을 합니다.
****************************************************************************************************
3 번째 문서
DNA 중합효소는 원래의 이중 가닥 DNA에서 템플릿 가닥에 보완적인 뉴클레오티드를 추가함으로써 새로운 DNA를 생성합니다. 이 과정에서 템플릿 가닥 일부의 A:T 염기의 비율이 3:2였다면, 새로 합성된 보완적인 DNA 가닥의 A:T 비율은 2:03입니다. 이는 중합효소가 보완적인 뉴클레오티드를 추가할 때, A와 T 염기를 비율에 맞게 추가하기 때문입니다. 따라서, 새로 합성된 DNA 가닥의 A:T 비율은 2:03이

In [29]:
re_ranked = llm_reranking('인간이 2세를 생산할 때 DNA의 결합 과정에 대히 설명해줘.', result)

문서 1:  
이유: 염색체와 DNA의 결합 과정을 설명하며, 자손 형성 시 유전 정보가 어떻게 조합되는지 다루고 있어 질문에 적합합니다.

문서 2:  
이유: 동물의 성적 번식 과정을 설명하지만, DNA 결합 과정에 대한 구체적인 설명이 부족합니다.

문서 3:  
이유: DNA 중합효소에 의한 DNA 복제 과정을 설명하지만, 생식세포 결합 과정과는 관련이 없습니다.

문서 4:  
이유: DNA의 이중 나선 구조와 상보적 결합을 설명하지만, 생식세포 결합 과정에 대한 설명은 없습니다.

문서 5:  
이유: DNA 리가아제의 역할을 설명하지만, 생식세포 결합 과정과는 관련이 없습니다.

문서 6:  
이유: 생식세포의 유전 정보량과 유전자 재조합 과정을 설명하여 질문에 적합한 정보를 제공합니다.

문서 7:  
이유: 세균 세포 간의 DNA 교환을 설명하지만, 인간의 생식세포 결합 과정과는 관련이 없습니다.

문서 8:  
이유: 유전자 전사 과정을 설명하지만, 생식세포 결합 과정과는 관련이 없습니다.

문서 9:  
이유: DNA 분해효소의 역할을 설명하지만, 생식세포 결합 과정과는 관련이 없습니다.

문서 10:  
이유: 아데닌과 티민의 결합을 설명하지만, 생식세포 결합 과정과는 관련이 없습니다.

문서 11:  
이유: 세포 주기와 DNA 복제 과정을 설명하지만, 생식세포 결합 과정과는 관련이 없습니다.

문서 12:  
이유: 돌연변이와 유전자 조합에 대해 설명하지만, DNA 결합 과정에 대한 구체적인 설명은 부족합니다.

문서 13:  
이유: 융합 과정에서의 질량-에너지 변환을 설명하지만, 생식세포 결합 과정과는 관련이 없습니다.

문서 14:  
이유: 식물의 생식 과정을 설명하지만, 인간의 DNA 결합 과정과는 관련이 없습니다.

문서 15:  
이유: 정자 세포의 염색체 수를 설명하지만, DNA 결합 과정에 대한 구체적인 설명은 부족합니다.
변환된 쿼리: [1, 6, 15, 2, 12]


In [30]:
print(re_ranked)

[1, 6, 15, 2, 12]


In [31]:
result = retrieval_with_score('헬륨이 다른 원소들과 반응을 잘 안하는 이유는?')

상위 5개의 문서:


In [32]:
i = 1
for doc in result:

    # print(doc[0])
    print(i, '번째 문서')
    print(doc_mapping[doc[0]]['content'])
    print('*' * 100)
    i += 1
    # print(doc_mapping[doc[0]['content']])

1 번째 문서
희귀한 기체인 헬륨, 네온, 아르곤, 크립톤, 크세논, 라돈은 다른 원소들과 거의 반응하지 않습니다. 이는 최외각 에너지 준위가 완전하게 채워져 있기 때문입니다. 이러한 특성으로 인해 이러한 희귀 기체들은 안정하고 비활성인 성질을 가지고 있습니다. 이들은 화학 반응에서 거의 관여하지 않으며, 다른 원소들과 결합하여 화합물을 형성하지 않습니다. 이러한 특징은 희귀 기체들을 다양한 산업 분야에서 사용할 수 있게 만들어줍니다. 예를 들어, 헬륨은 기체 냉매로 사용되며, 네온은 광고 표시판에 사용됩니다. 아르곤은 용접 작업에 사용되고, 크세논은 높은 휘도를 가진 조명 장치에 사용됩니다. 이러한 희귀 기체들은 우리 일상 생활에서도 다양한 용도로 활용되고 있습니다.
****************************************************************************************************
2 번째 문서
주기율표는 원소들을 그룹과 주기로 나누어 표현하는 방법입니다. 그룹은 세로로 배열된 열을 의미하며, 가장 반응성이 낮은 원소를 포함하는 그룹은 그룹 18 (8A)입니다. 이 그룹은 흔히 비활성 기체라고도 불리며, 헬륨, 네온, 아르곤 등을 포함하고 있습니다. 이러한 원소들은 전자 껍질이 완전히 채워져 있어 다른 원소와의 화학적 반응이 거의 일어나지 않습니다. 따라서, 그룹 18은 가장 반응성이 낮은 원소들을 포함하고 있습니다.
****************************************************************************************************
3 번째 문서
원소 주기율표에 따르면, He, Ne, Ar 원소 집합은 유사한 성질을 가지고 있습니다. 이들은 모두 비활성 기체로서, 외부 원소와 거의 반응하지 않는 특성을 가지고 있습니다. 또한, 이들은 모두 8개의 전자를 외께 전자껍질에 가지고 있으며, 이는 전자 구성의 안정성을 나타냅니다. 따라서,

In [33]:
re_ranked = llm_reranking('헬륨이 다른 원소들과 반응을 잘 안하는 이유는?', result)

문서 1:  
이유: 헬륨이 다른 원소들과 반응하지 않는 이유를 최외각 전자껍질이 완전히 채워져 있어 안정하다는 점에서 설명하고 있어 질문에 적합합니다.

문서 2:  
이유: 헬륨이 속한 그룹 18의 원소들이 전자 껍질이 완전히 채워져 있어 반응성이 낮다는 설명이 있어 질문에 적합합니다.

문서 3:  
이유: 헬륨이 비활성 기체로서 외부 원소와 거의 반응하지 않는 이유를 전자 구성의 안정성으로 설명하고 있어 질문에 적합합니다.

문서 4:  
이유: 비활성 기체의 전자 구성이 완전하여 화학적으로 안정하다는 설명이 있지만, 헬륨에 대한 구체적인 설명이 부족하여 덜 적합합니다.

문서 5:  
이유: 헬륨의 물리적 특성에 대한 설명이 주를 이루며, 화학적 반응성에 대한 설명이 없어 질문에 부적합합니다.

문서 6:  
이유: 헬륨의 이온화 에너지에 대한 설명이 있지만, 헬륨이 반응하지 않는 이유에 대한 직접적인 설명이 없어 부적합합니다.

문서 7:  
이유: 헬륨이 비반응성 기체로 사용된다는 언급은 있지만, 그 이유에 대한 설명이 부족하여 덜 적합합니다.

문서 8:  
이유: 알칼리 토금속 원소에 대한 설명으로, 헬륨의 반응성에 대한 정보가 없어 부적합합니다.

문서 9:  
이유: 크립톤의 반응성에 대한 설명이 주를 이루며, 헬륨에 대한 정보가 없어 부적합합니다.

문서 10:  
이유: 태양에서의 헬륨 생성에 대한 설명으로, 헬륨의 화학적 반응성에 대한 정보가 없어 부적합합니다.

문서 11:  
이유: 별에서의 헬륨 생성 과정에 대한 설명으로, 헬륨의 반응성에 대한 정보가 없어 부적합합니다.

문서 12:  
이유: 항성에서의 헬륨 생성에 대한 설명으로, 헬륨의 화학적 반응성에 대한 정보가 없어 부적합합니다.

문서 13:  
이유: 나트륨의 전자 구조에 대한 설명으로, 헬륨의 반응성에 대한 정보가 없어 부적합합니다.

문서 14:  
이유: 헬륨의 물리적 특성에 대한 설명이 주를 이루며, 화학적 반응성에 대한 설명이 없어 부적합합니다.

문서 15:

In [34]:
re_ranked

'[1, 2, 3, 4, 7]'

In [ ]:
금성에서 달이 어떻게 보일까?

In [49]:
query = '두 물질이 다른 분자구조나 화학적인 성분으로 이루어져 있다는 것을 어떻게 알 수 있나요?'
result = retrieval_with_score(query)
i = 1
for doc in result:

    # print(doc[0])
    print(i, '번째 문서')
    print(doc_mapping[doc[0]]['content'])
    print('*' * 100)
    i += 1
    # print(doc_mapping[doc[0]['content']])

re_ranked = llm_reranking(query, result)

상위 5개의 문서:
1 번째 문서
두 가지 물질은 빛에 대한 반응이 다릅니다. 이는 두 물질이 서로 다른 종류의 물질로 이루어져 있다는 강력한 증거입니다. 빛은 물질과 상호작용하는데, 두 물질이 빛에 대한 반응이 다르다는 것은 그들이 서로 다른 성질을 가지고 있다는 것을 의미합니다. 이러한 증거는 두 물질이 서로 다른 분자 구조를 가지고 있거나, 서로 다른 화학적 성분을 포함하고 있다는 것을 시사합니다. 따라서, 두 물질이 서로 다른 종류의 물질로 이루어져 있다는 것을 확실하게 나타내는 증거로 빛에 대한 반응의 차이를 들 수 있습니다.
****************************************************************************************************
2 번째 문서
물은 화학적 성질에 대한 예시로서, 수소와 산소로 분해할 수 있는 성질을 가지고 있습니다. 이는 물이 화학적으로 안정한 분자로 구성되어 있지 않고, 분해되어 다른 물질로 변할 수 있다는 것을 의미합니다. 수소와 산소는 물 분자를 구성하는 원자이며, 물은 이 두 원자가 결합하여 형성됩니다. 그러나 적절한 조건이 주어지면, 물은 수소와 산소로 다시 분해될 수 있습니다. 이러한 화학적 성질은 물의 구조와 원자 간 결합에 기인하며, 물의 화학적 특성을 이해하는 데 중요한 역할을 합니다.
****************************************************************************************************
3 번째 문서
알루미늄과 구리는 서로 다른 종류의 물질로 구성되어 있습니다. 알루미늄은 비투명하고 은백색을 띠며, 구리는 붉은색을 띠고 있습니다. 이 두 물질은 각각 고유한 물리적 특성을 가지고 있습니다. 알루미늄은 가벼우며, 구리는 상대적으로 무거운 편입니다. 또한, 알루미늄은 부식에 강하고 열전도성이 높은 반면, 구리는 부식에 약하고 전기전도성이 뛰어납니다. 이러한 특성들은 물질의

In [46]:
query = '건설 현장에서 망치로 벽을 치는 이유는?'
result = retrieval_with_score(query)
i = 1
for doc in result:

    # print(doc[0])
    print(i, '번째 문서')
    print(doc_mapping[doc[0]]['content'])
    print('*' * 100)
    i += 1
    # print(doc_mapping[doc[0]['content']])

re_ranked = llm_reranking(query, result)

상위 5개의 문서:
1 번째 문서
한 건설 작업자가 해머로 단단한 철벽을 치려 했지만 벽은 움직이지 않았습니다. 작업자가 사용한 에너지의 일부는 열로 변환되었습니다. 철벽은 고강도 강재로 만들어져 있어서 해머로 강력하게 치더라도 움직이지 않는 것이 정상입니다. 철벽은 건물의 구조를 지탱하고 보호하기 위해 설계되었으며, 그 내부에는 철과 다른 금속들이 견고하게 결합되어 있습니다. 따라서 해머로 치는 작업은 벽을 움직이게 하는 것보다는 에너지를 전달하여 열로 변환시키는 역할을 합니다. 이러한 열 변환은 작업자가 사용한 에너지를 벽에 흡수시키고, 벽의 온도를 조절하는 역할을 합니다. 따라서 작업자가 해머로 철벽을 치는 것은 벽을 움직이게 하는 것보다는 벽과 상호작용하여 열을 발생시키는 과정입니다.
****************************************************************************************************
2 번째 문서
한 목수가 얇은 종이로 나무 조각을 덮었습니다. 그는 덮인 나무 조각을 망치로 쳤습니다. 이 과정에서 충격이 발생하였고, 종이에는 작은 구멍이 생기고 연기 냄새가 났습니다. 이 사건은 역학 에너지에서 열 에너지로 전달되는 에너지 전달 방식을 보여주었습니다. 목수가 망치로 나무 조각을 칠 때, 망치의 운동 에너지가 나무 조각에 전달되어 역학 에너지로 변환됩니다. 이 역학 에너지는 나무 조각과 종이 사이에 전달되어 종이에 충격을 주게 됩니다. 충격으로 인해 종이는 변형되고, 작은 구멍이 생기며 연기 냄새가 나타납니다. 이는 역학 에너지가 종이에 전달되어 열 에너지로 변환되는 과정입니다. 따라서, 이 사건은 역학 에너지에서 열 에너지로 전달되는 에너지 전달 방식을 보여주었을 가능성이 가장 높습니다.
****************************************************************************************************
3 번째 

In [47]:
query = '자기장이 얼마나 센지 표현하는 방식은?'
result = retrieval_with_score(query)
i = 1
for doc in result:

    # print(doc[0])
    print(i, '번째 문서')
    print(doc_mapping[doc[0]]['content'])
    print('*' * 100)
    i += 1
    # print(doc_mapping[doc[0]['content']])

re_ranked = llm_reranking(query, result)

상위 5개의 문서:
1 번째 문서
자속의 단위는 웨버입니다. 웨버는 자기장의 크기를 나타내는 단위로 사용됩니다. 자속은 자기장이 특정한 공간에서 얼마나 강하게 작용하는지를 나타내는데, 웨버는 이러한 자속의 크기를 측정하는 데 사용됩니다. 웨버는 국제단위계에서 정의된 단위로, 자기장의 세기와 면적의 곱으로 표현됩니다. 자속은 자기장의 세기에 비례하므로, 웨버가 클수록 자속이 강하다는 것을 의미합니다. 따라서, 자속을 측정하고자 할 때는 웨버를 사용하여 자기장의 크기를 정확하게 표현할 수 있습니다.
****************************************************************************************************
2 번째 문서
g 값이 2.0033인 Si-H· 라디칼은 15.5MHz로 분리된 한 쌍의 선을 나타냅니다. 이 분리는 mT, Gauss 및 cm-1 단위로 표현될 수도 있습니다.

15.5MHz의 분리는 자기장 강도에서 7.352mT에 해당합니다. 이 값은 선을 분할하는 데 필요한 자기장의 강도를 나타냅니다.

가우스 단위에서 15.5MHz의 분리는 10.104가우스와 같습니다. 이 값은 보다 일반적으로 사용되는 가우스 단위의 자기장 강도를 나타냅니다.

마지막으로 15.5MHz의 간격은 cm-1로 표현될 수도 있습니다. 이 단위에서 간격은 18.39 x 10^-4 cm-1과 같습니다. 이 값은 두 선 사이의 에너지 차이를 나타냅니다.

요약하면, g 값이 2.0033인 Si-H· 라디칼은 15.5MHz로 분리된 한 쌍의 선을 나타내며, 이는 7.352mT, 10.104 가우스 및 18.39 x 10^-4 cm-1로 표현될 수 있습니다.
****************************************************************************************************
3 번째 문서
자기장은 분자의 자기 모멘트와 상호작용하여 분자를 분극시키는 힘

In [16]:
query = '달의 한쪽 면만 보이는 이유'
result = retrieval_with_score(query)
i = 1
for doc in result:

    # print(doc[0])
    print(i, '번째 문서')
    print(doc_mapping[doc[0]]['content'])
    print('*' * 100)
    i += 1
    # print(doc_mapping[doc[0]['content']])

re_ranked = llm_reranking(query, result)

상위 5개의 문서:
1 번째 문서
지구에서 달의 약 59%가 보이는 이유는 지구의 중력 때문입니다. 지구의 중력은 달을 지속적으로 끌어당기는 힘을 가지고 있습니다. 이로 인해 달은 지구 주변을 공전하면서 동시에 자전을 하게 됩니다. 이 자전과 공전이 같은 기간을 가지기 때문에 달의 한쪽 면이 항상 지구에서 보이게 됩니다. 지구의 중력은 달을 끌어당기는 힘을 가지고 있기 때문에 달의 약 59%가 지구에서 보이는 것입니다. 이러한 현상은 지구와 달 사이의 중력 상호작용으로 인해 발생합니다.
****************************************************************************************************
2 번째 문서
달의 본질적으로 같은 면을 항상 보는 이유는 달의 자전 주기와 궤도 주기가 같기 때문입니다. 달은 지구 주위를 공전하면서 동시에 자전을 합니다. 이때 달의 자전 주기와 궤도 주기가 정확히 일치하면서, 항상 같은 면을 우리에게 보여줍니다. 이러한 현상을 '달의 결합 자전'이라고 합니다. 달의 결합 자전은 지구의 중력과 달의 자력이 서로 작용하여 발생합니다. 이로 인해 달은 항상 같은 면을 우리에게 보여주는 것입니다. 이러한 현상은 우리가 달의 다른 면을 볼 수 없는 이유이기도 합니다. 달의 결합 자전은 우리에게 매우 익숙한 모습을 제공하며, 우리가 달을 관찰하고 연구하는 데 많은 도움을 주고 있습니다.
****************************************************************************************************
3 번째 문서
지구에서 보이는 달의 면이 항상 같은 이유는 달이 지구 주위를 한 바퀴 돌 때마다 한 번 회전하기 때문입니다. 이는 달의 자전과 지구 주위를 공전하는 속도가 동일하기 때문에 발생합니다. 달은 지구 주위를 약 27.3일 동안 공전하며, 동시에 자전하면서 항상 같은 면이 지구에서 보입니다. 이러한 현상

In [48]:

query = '연구자가 갖추어야 할 태도와 자세가 뭘까?'
result = retrieval_with_score(query)
i = 1
for doc in result:

    # print(doc[0])
    print(i, '번째 문서')
    print(doc_mapping[doc[0]]['content'])
    print('*' * 100)
    i += 1
    # print(doc_mapping[doc[0]['content']])

re_ranked = llm_reranking(query, result)

상위 5개의 문서:
1 번째 문서
과학적 발견에 대해 의심하는 태도는 과학계에서 매우 중요합니다. 왜냐하면 모든 과학적 발견은 받아들여지기 전에 비판적으로 검토되어야 하기 때문입니다. 과학은 사실과 증거에 기반한 학문이기 때문에, 새로운 발견이 제시되었을 때 우리는 그것을 의심하고 검증해야 합니다. 이를 통해 우리는 잘못된 정보나 잘못된 결론을 피할 수 있으며, 정확하고 신뢰할 수 있는 지식을 얻을 수 있습니다. 또한, 의심하는 태도는 과학적 방법론의 핵심 원칙 중 하나인 '검증 가능성'을 강조합니다. 과학적 발견은 다른 연구자들에 의해 독립적으로 재현되고 검증되어야 합니다. 이를 통해 우리는 과학적 지식의 신뢰성을 높일 수 있습니다. 따라서, 조앤의 과학 선생님의 충고는 매우 중요하며, 우리는 새로운 과학적 발견에 대해 항상 의심하는 태도를 가져야 합니다.
****************************************************************************************************
2 번째 문서
미셸은 연구를 실시했지만, 그 결과가 그녀의 가설과 일치하지 않았습니다. 이는 미셸에게 큰 도전이었습니다. 하지만 미셸은 포기하지 않았습니다. 그녀는 조사를 반복해야 할 것입니다. 추가적인 실험과 데이터 수집을 통해 미셸은 더 많은 정보를 얻을 수 있을 것입니다. 이를 통해 그녀는 가설을 수정하거나 새로운 가설을 세울 수 있을 것입니다. 미셸은 과학적인 방법을 따라 연구를 진행하고, 결과를 분석하여 결론을 도출할 것입니다. 이는 연구자로서 미셸의 책임입니다. 미셸은 실패를 두려워하지 않고, 계속해서 노력하고 배우는 자세를 갖추어야 합니다. 그녀의 노력과 열정은 결국에는 성과를 가져올 것입니다.
****************************************************************************************************
3 번째 문서
연구자가 자신이 관찰하는 상황

In [38]:
re_ranked

'[2, 1, 4, 5, 7, 9, 10, 13, 11, 14]'

In [15]:
result

[('a729b4f2-c734-4c60-9205-1518ba762593', 3.1256200557328038),
 ('191c4b9f-6feb-49dd-90ad-9f2eebb6113e', 3.0973838877666555),
 ('e4186e86-6782-472a-a688-276965ef2f45', 2.5495843876460094),
 ('1adddd9c-59de-470c-b29a-b49bb6c7b43a', 2.435677301568596),
 ('2b0016b4-f2de-449b-bfe9-a4d5e50c6d65', 2.4245303736072605),
 ('2e49f4c2-934d-45b3-af98-6f8fffa69642', 2.1937642128945454),
 ('09456b60-1627-421f-92b1-b8983b45df39', 2.183083412820212),
 ('11046b98-c0e8-44d9-8187-c7417c2b991b', 2.1414378546132715),
 ('9220f0de-b448-4306-a107-89cf70ac645d', 1.9743598925563028),
 ('62aa0f91-c101-4b84-a8a5-2f88cc5fc178', 1.919294482646016)]

In [179]:
llm_result = llm_reranking('달의 한쪽 면만 보이는 이유', result)

변환된 쿼리: [ ]


In [22]:
llm_result = llm_reranking('나무의 생태계 역할', result)

NotFoundError: Error code: 404 - {'error': {'message': 'The model `o1-preview` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}

In [182]:
llm_result

'[1, 3, 2, 5, 4]'

In [96]:
llm_result = llm_reranking('달의 한쪽 면만 보이는 이유', result)

변환된 쿼리: [2, 3, 1, 4, 5]


In [97]:
for doc in result:

    # print(doc[0])
    print(doc_mapping[doc[0]]['content'])
    print('*' * 100)
    # print(doc_mapping[doc[0]['content']])

지구에서 달의 약 59%가 보이는 이유는 지구의 중력 때문입니다. 지구의 중력은 달을 지속적으로 끌어당기는 힘을 가지고 있습니다. 이로 인해 달은 지구 주변을 공전하면서 동시에 자전을 하게 됩니다. 이 자전과 공전이 같은 기간을 가지기 때문에 달의 한쪽 면이 항상 지구에서 보이게 됩니다. 지구의 중력은 달을 끌어당기는 힘을 가지고 있기 때문에 달의 약 59%가 지구에서 보이는 것입니다. 이러한 현상은 지구와 달 사이의 중력 상호작용으로 인해 발생합니다.
****************************************************************************************************
달의 본질적으로 같은 면을 항상 보는 이유는 달의 자전 주기와 궤도 주기가 같기 때문입니다. 달은 지구 주위를 공전하면서 동시에 자전을 합니다. 이때 달의 자전 주기와 궤도 주기가 정확히 일치하면서, 항상 같은 면을 우리에게 보여줍니다. 이러한 현상을 '달의 결합 자전'이라고 합니다. 달의 결합 자전은 지구의 중력과 달의 자력이 서로 작용하여 발생합니다. 이로 인해 달은 항상 같은 면을 우리에게 보여주는 것입니다. 이러한 현상은 우리가 달의 다른 면을 볼 수 없는 이유이기도 합니다. 달의 결합 자전은 우리에게 매우 익숙한 모습을 제공하며, 우리가 달을 관찰하고 연구하는 데 많은 도움을 주고 있습니다.
****************************************************************************************************
지구에서 보이는 달의 면이 항상 같은 이유는 달이 지구 주위를 한 바퀴 돌 때마다 한 번 회전하기 때문입니다. 이는 달의 자전과 지구 주위를 공전하는 속도가 동일하기 때문에 발생합니다. 달은 지구 주위를 약 27.3일 동안 공전하며, 동시에 자전하면서 항상 같은 면이 지구에서 보입니다. 이러한 현상은 달의 자전 주기와 공전 주기가 정확히 일치하기 때문에 발생합

In [24]:
result

[('fcfee923-af42-43d1-b664-09a0112e4f74', 5.381111486045304),
 ('8a78364e-63bf-4915-b718-fdc461bc62c9', 4.45264525239844),
 ('cefe7caf-6cd1-422a-b41e-e82b543556e9', 4.400076349872069),
 ('892eb0be-055f-47c5-8895-1afb97c8f406', 3.822421126579113),
 ('340485f8-4e78-44f4-a53a-2df21915367f', 2.8626556358845967)]

In [49]:
check_doc = []
for doc in result:
    check_doc.append(doc_mapping[doc[0]])

In [51]:
check_doc[0]['content']

'지구에서 달의 약 59%가 보이는 이유는 지구의 중력 때문입니다. 지구의 중력은 달을 지속적으로 끌어당기는 힘을 가지고 있습니다. 이로 인해 달은 지구 주변을 공전하면서 동시에 자전을 하게 됩니다. 이 자전과 공전이 같은 기간을 가지기 때문에 달의 한쪽 면이 항상 지구에서 보이게 됩니다. 지구의 중력은 달을 끌어당기는 힘을 가지고 있기 때문에 달의 약 59%가 지구에서 보이는 것입니다. 이러한 현상은 지구와 달 사이의 중력 상호작용으로 인해 발생합니다.'

In [67]:
result

[('fcfee923-af42-43d1-b664-09a0112e4f74', 5.381111486045304),
 ('8a78364e-63bf-4915-b718-fdc461bc62c9', 4.45264525239844),
 ('cefe7caf-6cd1-422a-b41e-e82b543556e9', 4.400076349872069),
 ('892eb0be-055f-47c5-8895-1afb97c8f406', 3.822421126579113),
 ('340485f8-4e78-44f4-a53a-2df21915367f', 2.8626556358845967)]

In [75]:

check_doc = ''
i = 0
for doc in result:
    # print(doc_mapping[doc[0]]['content'])
    if i == 0:
        check_doc += f'''문서: {doc_mapping[doc[0]]['content']} \n'''
    else:  
        check_doc += f'''    문서: {doc_mapping[doc[0]]['content']} \n'''
    print(check_doc)
    print('*' * 100)
    i += 1

문서: 지구에서 달의 약 59%가 보이는 이유는 지구의 중력 때문입니다. 지구의 중력은 달을 지속적으로 끌어당기는 힘을 가지고 있습니다. 이로 인해 달은 지구 주변을 공전하면서 동시에 자전을 하게 됩니다. 이 자전과 공전이 같은 기간을 가지기 때문에 달의 한쪽 면이 항상 지구에서 보이게 됩니다. 지구의 중력은 달을 끌어당기는 힘을 가지고 있기 때문에 달의 약 59%가 지구에서 보이는 것입니다. 이러한 현상은 지구와 달 사이의 중력 상호작용으로 인해 발생합니다. 

****************************************************************************************************
문서: 지구에서 달의 약 59%가 보이는 이유는 지구의 중력 때문입니다. 지구의 중력은 달을 지속적으로 끌어당기는 힘을 가지고 있습니다. 이로 인해 달은 지구 주변을 공전하면서 동시에 자전을 하게 됩니다. 이 자전과 공전이 같은 기간을 가지기 때문에 달의 한쪽 면이 항상 지구에서 보이게 됩니다. 지구의 중력은 달을 끌어당기는 힘을 가지고 있기 때문에 달의 약 59%가 지구에서 보이는 것입니다. 이러한 현상은 지구와 달 사이의 중력 상호작용으로 인해 발생합니다. 
    문서: 달의 본질적으로 같은 면을 항상 보는 이유는 달의 자전 주기와 궤도 주기가 같기 때문입니다. 달은 지구 주위를 공전하면서 동시에 자전을 합니다. 이때 달의 자전 주기와 궤도 주기가 정확히 일치하면서, 항상 같은 면을 우리에게 보여줍니다. 이러한 현상을 '달의 결합 자전'이라고 합니다. 달의 결합 자전은 지구의 중력과 달의 자력이 서로 작용하여 발생합니다. 이로 인해 달은 항상 같은 면을 우리에게 보여주는 것입니다. 이러한 현상은 우리가 달의 다른 면을 볼 수 없는 이유이기도 합니다. 달의 결합 자전은 우리에게 매우 익숙한 모습을 제공하며, 우리가 달을 관찰하고 연구하는 데 많은 도움을 주고 있습니다. 

**************************

In [71]:
print(check_doc)

문서: 지구에서 달의 약 59%가 보이는 이유는 지구의 중력 때문입니다. 지구의 중력은 달을 지속적으로 끌어당기는 힘을 가지고 있습니다. 이로 인해 달은 지구 주변을 공전하면서 동시에 자전을 하게 됩니다. 이 자전과 공전이 같은 기간을 가지기 때문에 달의 한쪽 면이 항상 지구에서 보이게 됩니다. 지구의 중력은 달을 끌어당기는 힘을 가지고 있기 때문에 달의 약 59%가 지구에서 보이는 것입니다. 이러한 현상은 지구와 달 사이의 중력 상호작용으로 인해 발생합니다. 
문서: 달의 본질적으로 같은 면을 항상 보는 이유는 달의 자전 주기와 궤도 주기가 같기 때문입니다. 달은 지구 주위를 공전하면서 동시에 자전을 합니다. 이때 달의 자전 주기와 궤도 주기가 정확히 일치하면서, 항상 같은 면을 우리에게 보여줍니다. 이러한 현상을 '달의 결합 자전'이라고 합니다. 달의 결합 자전은 지구의 중력과 달의 자력이 서로 작용하여 발생합니다. 이로 인해 달은 항상 같은 면을 우리에게 보여주는 것입니다. 이러한 현상은 우리가 달의 다른 면을 볼 수 없는 이유이기도 합니다. 달의 결합 자전은 우리에게 매우 익숙한 모습을 제공하며, 우리가 달을 관찰하고 연구하는 데 많은 도움을 주고 있습니다. 
문서: 지구에서 보이는 달의 면이 항상 같은 이유는 달이 지구 주위를 한 바퀴 돌 때마다 한 번 회전하기 때문입니다. 이는 달의 자전과 지구 주위를 공전하는 속도가 동일하기 때문에 발생합니다. 달은 지구 주위를 약 27.3일 동안 공전하며, 동시에 자전하면서 항상 같은 면이 지구에서 보입니다. 이러한 현상은 달의 자전 주기와 공전 주기가 정확히 일치하기 때문에 발생합니다. 달의 자전 주기와 공전 주기가 일치하는 이유는 지구의 조력과 달의 조력이 서로 상쇄되기 때문입니다. 이러한 현상은 천문학적으로 매우 흥미로운 현상으로 알려져 있습니다. 
문서: 달이 항상 지구에 같은 면을 보여준다는 사실은 달이 축을 중심으로 약 한 달을 주기로 회전한다는 증거입니다. 이는 달의 자전 주기로 인해 발생하는 현상으

In [34]:
docus = ''

for 



'aaaa'

In [21]:
query = '달의 한쪽 면만 보이는 이유'

In [77]:
check_doc = ''
for i, doc in enumerate(result):
    check_doc += f"문서 {i+1}: {doc_mapping[doc[0]]['content']} \n"

# 시스템 메시지 생성
system_message = {
    "role": "system",
    "content": f"""
    당신은 문서와 질문을 비교하여, 질문에 가장 적합한 문서들을 찾고 그 순서를 반환하는 전문가입니다.
    
    다음의 규칙을 반드시 따르세요:
    1. 주어진 질문과 문서들의 내용을 비교하여, 질문의 답을 찾을 수 있는 문서를 가장 적합한 순서대로 나열하세요.
    2. 반드시 리스트 형태로만 출력하세요. 다른 형식은 허용되지 않습니다. 예를 들어 [1, 3, 2, 4]와 같이 반환하세요.
    3. 리스트에 들어갈 문서 번호는 1부터 시작하며, 문서들의 순서를 중요도에 따라 배열하세요.
    4. 리스트 외의 다른 정보를 출력하지 마세요. 리스트 이외의 내용은 모델이 자동으로 무시해야 합니다.

    질문: {query}

    문서들:
    {check_doc}
    """
}

# 사용자 메시지 준비


In [81]:
print(system_message['content'])


    당신은 문서와 질문을 비교하여, 질문에 가장 적합한 문서들을 찾고 그 순서를 반환하는 전문가입니다.
    주어진 질문과 문서들의 내용을 기반으로, 질문의 답을 찾을 수 있는 문서를 가장 적합한 순서대로 배열하세요.
    각 문서의 순서를 리스트 형태로 반환하며, 예를 들어 [1, 3, 2, 4]의 형식으로 출력합니다.

    질문: 달의 한쪽 면만 보이는 이유

    문서들:
    문서 1: 지구에서 달의 약 59%가 보이는 이유는 지구의 중력 때문입니다. 지구의 중력은 달을 지속적으로 끌어당기는 힘을 가지고 있습니다. 이로 인해 달은 지구 주변을 공전하면서 동시에 자전을 하게 됩니다. 이 자전과 공전이 같은 기간을 가지기 때문에 달의 한쪽 면이 항상 지구에서 보이게 됩니다. 지구의 중력은 달을 끌어당기는 힘을 가지고 있기 때문에 달의 약 59%가 지구에서 보이는 것입니다. 이러한 현상은 지구와 달 사이의 중력 상호작용으로 인해 발생합니다. 
문서 2: 달의 본질적으로 같은 면을 항상 보는 이유는 달의 자전 주기와 궤도 주기가 같기 때문입니다. 달은 지구 주위를 공전하면서 동시에 자전을 합니다. 이때 달의 자전 주기와 궤도 주기가 정확히 일치하면서, 항상 같은 면을 우리에게 보여줍니다. 이러한 현상을 '달의 결합 자전'이라고 합니다. 달의 결합 자전은 지구의 중력과 달의 자력이 서로 작용하여 발생합니다. 이로 인해 달은 항상 같은 면을 우리에게 보여주는 것입니다. 이러한 현상은 우리가 달의 다른 면을 볼 수 없는 이유이기도 합니다. 달의 결합 자전은 우리에게 매우 익숙한 모습을 제공하며, 우리가 달을 관찰하고 연구하는 데 많은 도움을 주고 있습니다. 
문서 3: 지구에서 보이는 달의 면이 항상 같은 이유는 달이 지구 주위를 한 바퀴 돌 때마다 한 번 회전하기 때문입니다. 이는 달의 자전과 지구 주위를 공전하는 속도가 동일하기 때문에 발생합니다. 달은 지구 주위를 약 27.3일 동안 공전하며, 동시에 자전하면서 항상 같은 면이 지구에서 보입니다

In [40]:

system_message = {
    "role": "system",
    "content": f"""
    당신은 문서들과 질문들을 비교해서 맞는 문서를 찾는 전문가입니다.
    질문과 문서들을 받으면 그 질문의 정답을 답할 수 있는 가장 적합한 순서대로 정렬 하고 그 이유를 설명합니다.
    질문 : {query}
    """
}

In [76]:
print(system_message['content'] + check_doc)


    당신은 문서들과 질문들을 비교해서 맞는 문서를 찾는 전문가입니다.
    질문과 문서들을 받으면 그 질문의 정답을 답할 수 있는 가장 적합한 순서대로 정렬 하고 그 이유를 설명합니다.
    질문 : 달의 한쪽 면만 보이는 이유
    
    
    
    문서: 지구에서 달의 약 59%가 보이는 이유는 지구의 중력 때문입니다. 지구의 중력은 달을 지속적으로 끌어당기는 힘을 가지고 있습니다. 이로 인해 달은 지구 주변을 공전하면서 동시에 자전을 하게 됩니다. 이 자전과 공전이 같은 기간을 가지기 때문에 달의 한쪽 면이 항상 지구에서 보이게 됩니다. 지구의 중력은 달을 끌어당기는 힘을 가지고 있기 때문에 달의 약 59%가 지구에서 보이는 것입니다. 이러한 현상은 지구와 달 사이의 중력 상호작용으로 인해 발생합니다. 
    문서: 달의 본질적으로 같은 면을 항상 보는 이유는 달의 자전 주기와 궤도 주기가 같기 때문입니다. 달은 지구 주위를 공전하면서 동시에 자전을 합니다. 이때 달의 자전 주기와 궤도 주기가 정확히 일치하면서, 항상 같은 면을 우리에게 보여줍니다. 이러한 현상을 '달의 결합 자전'이라고 합니다. 달의 결합 자전은 지구의 중력과 달의 자력이 서로 작용하여 발생합니다. 이로 인해 달은 항상 같은 면을 우리에게 보여주는 것입니다. 이러한 현상은 우리가 달의 다른 면을 볼 수 없는 이유이기도 합니다. 달의 결합 자전은 우리에게 매우 익숙한 모습을 제공하며, 우리가 달을 관찰하고 연구하는 데 많은 도움을 주고 있습니다. 
    문서: 지구에서 보이는 달의 면이 항상 같은 이유는 달이 지구 주위를 한 바퀴 돌 때마다 한 번 회전하기 때문입니다. 이는 달의 자전과 지구 주위를 공전하는 속도가 동일하기 때문에 발생합니다. 달은 지구 주위를 약 27.3일 동안 공전하며, 동시에 자전하면서 항상 같은 면이 지구에서 보입니다. 이러한 현상은 달의 자전 주기와 공전 주기가 정확히 일치하기 때문에 발생합니다. 달의 자전 주기와 공전 주기가 일치하는 이유

In [41]:
system_message['content']

'\n    당신은 문서들과 질문들을 비교해서 맞는 문서를 찾는 전문가입니다.\n    질문과 문서들을 받으면 그 질문의 정답을 답할 수 있는 가장 적합한 순서대로 정렬 하고 그 이유를 설명합니다.\n    질문 : 달의 한쪽 면만 보이는 이유\n    \n    \n    \n    '

In [84]:
result

[('35c5dcc7-4720-4318-901e-770105ae63fd', 1.0),
 ('553989d9-ee23-4203-b244-a941b6fa8d99', 0.4002819498232448),
 ('464ace62-ddf2-423d-a5d7-2f17e6785c8e', 0.3634118239354528),
 ('b2e0e809-c9e9-4465-9248-07a9b49b034f', 0.34216546142370263),
 ('bb6d04b6-a6cf-4a9f-8324-4e06e6e81c86', 0.33430138519401303)]

In [72]:
len(result)

304

In [67]:
result

[('35c5dcc7-4720-4318-901e-770105ae63fd', 1.0),
 ('553989d9-ee23-4203-b244-a941b6fa8d99', 0.23412504819441607),
 ('464ace62-ddf2-423d-a5d7-2f17e6785c8e', 0.14491570051327474),
 ('8a78364e-63bf-4915-b718-fdc461bc62c9', 0.14011413098361244),
 ('b2e0e809-c9e9-4465-9248-07a9b49b034f', 0.13611011325451836),
 ('bb6d04b6-a6cf-4a9f-8324-4e06e6e81c86', 0.12577977996262749),
 ('59a8259f-4a39-4ab6-ad3f-2e4161ad458d', 0.11593999491723594),
 ('efb313ef-d7af-4d82-86f4-b5f013714a0c', 0.11180761383894434),
 ('340485f8-4e78-44f4-a53a-2df21915367f', 0.11085558745846859),
 ('79216c43-fe13-4413-abcc-a8b9f70dcdad', 0.10492632499124462),
 ('f016fa89-bfab-44ab-a7e4-a8979cf931ec', 0.09308762657414664),
 ('fcfee923-af42-43d1-b664-09a0112e4f74', 0.09106507265372116),
 ('b4fe4b36-8e44-4e11-baee-08971e8ea9d3', 0.09024043264021732),
 ('d1cbb6a8-6346-4e84-b294-0c8e84d37c07', 0.09006067498879455),
 ('3eff8a03-46a4-4e9e-8473-1e7f08b81a33', 0.08667543075772886),
 ('cefe7caf-6cd1-422a-b41e-e82b543556e9', 0.086667547036

In [64]:
result

[('35c5dcc7-4720-4318-901e-770105ae63fd', 3.0),
 ('553989d9-ee23-4203-b244-a941b6fa8d99', 1.4563722931761598),
 ('b2e0e809-c9e9-4465-9248-07a9b49b034f', 1.3336548339262635),
 ('bb6d04b6-a6cf-4a9f-8324-4e06e6e81c86', 1.257797799626275),
 ('efb313ef-d7af-4d82-86f4-b5f013714a0c', 1.1180761383894438),
 ('340485f8-4e78-44f4-a53a-2df21915367f', 1.1085558745846862),
 ('79216c43-fe13-4413-abcc-a8b9f70dcdad', 1.0222800995099177),
 ('8a78364e-63bf-4915-b718-fdc461bc62c9', 0.9831312848871009),
 ('464ace62-ddf2-423d-a5d7-2f17e6785c8e', 0.9457210446216906),
 ('f016fa89-bfab-44ab-a7e4-a8979cf931ec', 0.9308762657414666),
 ('d1cbb6a8-6346-4e84-b294-0c8e84d37c07', 0.9006067498879458),
 ('fcfee923-af42-43d1-b664-09a0112e4f74', 0.8943584270287235),
 ('3eff8a03-46a4-4e9e-8473-1e7f08b81a33', 0.8667543075772888),
 ('cefe7caf-6cd1-422a-b41e-e82b543556e9', 0.8666754703665849),
 ('43b53301-468b-41a2-ad67-63d8ecd84596', 0.865019766767529),
 ('b4fe4b36-8e44-4e11-baee-08971e8ea9d3', 0.8605979727924203),
 ('3aca3a

In [42]:
knn_retrieved_docs, bm25_retrieved_docs = retrieval_with_score('금성에서 달의 관측 모습')

In [53]:
retrieved_chunks = []
for i, idx in enumerate(merged_docs):
    doc_info = doc_mapping[idx[0]]
    content = doc_info['content']
    retrieved_chunks.append(content)


In [55]:
len(retrieved_chunks)

304

In [36]:
bm25_retrieved_docs

[('35c5dcc7-4720-4318-901e-770105ae63fd', 1.0),
 ('553989d9-ee23-4203-b244-a941b6fa8d99', 0.9444848888812836),
 ('b2e0e809-c9e9-4465-9248-07a9b49b034f', 0.8842733457626346),
 ('efb313ef-d7af-4d82-86f4-b5f013714a0c', 0.8193222278618356),
 ('bb6d04b6-a6cf-4a9f-8324-4e06e6e81c86', 0.6831144213783673),
 ('340485f8-4e78-44f4-a53a-2df21915367f', 0.6269314548145943),
 ('2b40e339-174c-462f-8607-7a6be35ccd6e', 0.6226977606628725),
 ('d1cbb6a8-6346-4e84-b294-0c8e84d37c07', 0.6169972099203739),
 ('8a78364e-63bf-4915-b718-fdc461bc62c9', 0.6149938243837528),
 ('79216c43-fe13-4413-abcc-a8b9f70dcdad', 0.5730639633952893),
 ('f016fa89-bfab-44ab-a7e4-a8979cf931ec', 0.5609983469940513),
 ('89cc1287-30e4-4319-8288-9453ea1ebdac', 0.5548349253922615),
 ('4d59d546-b596-4a07-aa3f-6d6e4e6d1129', 0.54882121864961),
 ('68fa9783-3297-4a26-a50d-5a3d0b7a4159', 0.5453978045150492),
 ('cb579876-ea7a-4bd8-aae3-ceee27238435', 0.532464206211747),
 ('940293bb-5bbf-475c-bae4-12b46e9fe8a3', 0.5304403908427593),
 ('fcfee92

In [35]:
knn_retrieved_docs

[('35c5dcc7-4720-4318-901e-770105ae63fd', 1.0),
 ('464ace62-ddf2-423d-a5d7-2f17e6785c8e', 0.6624043815507087),
 ('bb6d04b6-a6cf-4a9f-8324-4e06e6e81c86', 0.5746833782479076),
 ('340485f8-4e78-44f4-a53a-2df21915367f', 0.4816244197700918),
 ('59a8259f-4a39-4ab6-ad3f-2e4161ad458d', 0.47103169937071604),
 ('3eff8a03-46a4-4e9e-8473-1e7f08b81a33', 0.47061854927924707),
 ('da6c8a3f-45a9-4025-a63a-47c05ba2b336', 0.44975586949096547),
 ('b2e0e809-c9e9-4465-9248-07a9b49b034f', 0.445460588360926),
 ('79216c43-fe13-4413-abcc-a8b9f70dcdad', 0.4453614003428386),
 ('43b53301-468b-41a2-ad67-63d8ecd84596', 0.4451028315779454),
 ('7f74fa3d-e61c-4cb2-b5ef-623f3b1f8fb9', 0.42644188582019293),
 ('b4fe4b36-8e44-4e11-baee-08971e8ea9d3', 0.40700923328461924),
 ('cefe7caf-6cd1-422a-b41e-e82b543556e9', 0.39390801591167784),
 ('a4e3a126-3eb3-4b06-8f4d-082c698fa366', 0.38638732441121115),
 ('553989d9-ee23-4203-b244-a941b6fa8d99', 0.3854762344708759),
 ('f016fa89-bfab-44ab-a7e4-a8979cf931ec', 0.3698779187474153),
 

In [43]:
def merge_and_sum_scores(knn_retrieved_docs, bm25_retrieved_docs):
    """
    knn_retrieved_docs와 bm25_retrieved_docs에서 동일한 id를 가진 항목은 스코어를 더하고,
    그렇지 않은 항목은 그대로 유지하며, 마지막에 점수대로 정렬하는 함수.
    
    Args:
        knn_retrieved_docs (list of tuple): [(id, score), ...] 형태의 리스트
        bm25_retrieved_docs (list of tuple): [(id, score), ...] 형태의 리스트
    
    Returns:
        merged_docs (list of tuple): [(id, combined_score), ...] 형태의 리스트, 점수 내림차순 정렬
    """
    # 두 리스트를 딕셔너리로 변환하여 빠르게 검색할 수 있게 함
    knn_dict = {doc_id: score for doc_id, score in knn_retrieved_docs}
    bm25_dict = {doc_id: score for doc_id, score in bm25_retrieved_docs}
    
    # 모든 unique id를 set으로 결합
    all_ids = set(knn_dict.keys()).union(set(bm25_dict.keys()))

    # 동일한 id가 있으면 스코어를 더하고, 없으면 그대로 유지
    merged_docs = []
    for doc_id in all_ids:
        knn_score = knn_dict.get(doc_id, 0)  # knn에 없으면 0으로 간주
        bm25_score = bm25_dict.get(doc_id, 0)  # bm25에 없으면 0으로 간주
        combined_score = knn_score + bm25_score
        merged_docs.append((doc_id, combined_score))
    
    # 점수 내림차순으로 정렬
    merged_docs = sorted(merged_docs, key=lambda x: x[1], reverse=True)
    
    return merged_docs




In [38]:
merged_docs = merge_and_sum_scores(knn_retrieved_docs, bm25_retrieved_docs)


In [45]:
merged_docs

[('35c5dcc7-4720-4318-901e-770105ae63fd', 2.0),
 ('553989d9-ee23-4203-b244-a941b6fa8d99', 1.3299611233521595),
 ('b2e0e809-c9e9-4465-9248-07a9b49b034f', 1.3297339341235606),
 ('bb6d04b6-a6cf-4a9f-8324-4e06e6e81c86', 1.257797799626275),
 ('efb313ef-d7af-4d82-86f4-b5f013714a0c', 1.1180761383894438),
 ('340485f8-4e78-44f4-a53a-2df21915367f', 1.1085558745846862),
 ('79216c43-fe13-4413-abcc-a8b9f70dcdad', 1.018425363738128),
 ('f016fa89-bfab-44ab-a7e4-a8979cf931ec', 0.9308762657414666),
 ('8a78364e-63bf-4915-b718-fdc461bc62c9', 0.9234155670372404),
 ('d1cbb6a8-6346-4e84-b294-0c8e84d37c07', 0.9006067498879458),
 ('fcfee923-af42-43d1-b664-09a0112e4f74', 0.892030955670368),
 ('464ace62-ddf2-423d-a5d7-2f17e6785c8e', 0.8738016216915396),
 ('3eff8a03-46a4-4e9e-8473-1e7f08b81a33', 0.8667543075772888),
 ('cefe7caf-6cd1-422a-b41e-e82b543556e9', 0.8666754703665849),
 ('43b53301-468b-41a2-ad67-63d8ecd84596', 0.865019766767529),
 ('b4fe4b36-8e44-4e11-baee-08971e8ea9d3', 0.8546256365624556),
 ('3aca3a90

In [88]:
d = doc_mapping['25de4ffd-cee4-4f27-907e-fd6b802c6ede']

In [96]:
y = {"docid": "82b095fd-2fb6-48ae-8476-2f457f8b6650", "src": "ko_ai2_arc__ARC_Challenge__train", "content": "인간 남성과 여성의 생식세포는 같은 수의 염색체를 가지고 있습니다. 남성은 23쌍의 염색체를 가지고 있으며, 여성도 23쌍의 염색체를 가지고 있습니다. 따라서, 일반적인 남성과 여성의 생식세포가 결합하여 자손을 생산할 때, 염색체의 수는 두 배가 됩니다. 이는 염색체의 유전 정보를 조합하여 새로운 개체를 형성하는 과정입니다. 염색체는 DNA를 포함하고 있으며, DNA는 유전 정보를 담고 있는 분자입니다. 따라서, 염색체의 수가 두 배가 되면, 자손은 부모의 유전 정보를 조합하여 새로운 개체를 형성하게 됩니다."}

In [95]:
d

{'docid': '25de4ffd-cee4-4f27-907e-fd6b802c6ede',
 'src': 'ko_mmlu__high_school_biology__test',
 'content': '기억 상실은 대뇌의 기능 장애로 인해 발생할 가능성이 가장 높습니다. 대뇌는 인간의 중추신경계에서 가장 중요한 역할을 담당하며, 인지, 기억, 감정 등 다양한 기능을 조절합니다. 따라서 대뇌의 어느 부분에서 문제가 발생하면 기억 상실이나 기타 인지 기능의 장애가 발생할 수 있습니다. 대뇌는 뇌의 가장 큰 부분으로, 뇌의 표면에 위치한 뇌피질과 그 아래에 있는 심층뇌로 구성되어 있습니다. 이러한 대뇌의 구조와 기능이 원활하게 작동하지 않으면 기억 상실이 발생할 수 있습니다. 따라서 대뇌는 기억 상실과 관련된 가장 중요한 부분으로 간주됩니다.'}

#### 4. Document Relevance Check


In [9]:
def document_relevance_checker(document):
    """
    LLM을 사용하여 문서에서 논리적 오류, 상식적 타당성, 과학적 평가 및 수학적 평가를 진행합니다.
    네 가지 조건이 모두 충족되지 않을 경우 'False'를 반환하고, 그렇지 않으면 'True'를 반환합니다.
    """
    system_message = {
        "role": "system",
        "content": """
        당신은 과학적 문서를 평가하는 전문가입니다.

        **1. 논리적 오류 평가**:
            - 문서의 설명에서 과학적 원리와 모순되거나 자명한 논리적 오류가 있는지 확인하세요.
            - 예: '해머로 철벽을 치는 것이 열을 발생시키기 위한 목적이다'라는 설명은 과학적 원리와 어긋납니다.

        **2. 상식적 타당성 평가**:
            - 설명이 과학적으로 맞다고 해도 현실적인지 상식적으로 타당한지 판단하세요.
            - 예: '가스레인지를 이용해 집안을 따뜻하게 한다'는 설명은 상식적이지 않을 수 있습니다.

        **3. 과학적 평가**:
            - 문서의 설명이 학계에서 널리 받아들여지는 과학적 사실과 일치하는지 평가하세요.
            - 예: 생식세포의 염색체 수가 23쌍(46개)라는 사실은 맞지만, '염색체의 수가 두 배가 된다'는 해석은 잘못된 것입니다.

        **4. 수학적 평가**:
            - 문서에 포함된 수학적 표현이 논리적이고 타당한지 확인하세요.
            - 예: '매년 13개월의 월급을 지급합니다'는 수학적으로 잘못된 표현입니다.

        **결과**:
            - 네 가지 평가 기준 중 두개이상 충족되면 "True"로 답변하세요.
            - 네 가지 중 3가지 기준이상 틀렸을 경우 "False"로 답변하세요.
        """
    }

    # 사용자 문서 입력 준비
    document_message = {"role": "user", "content": f"문서 출처: {document['src']}\n내용: {document['content']}"}

    # LLM 호출을 위한 전체 메시지 배열 생성
    full_message = [system_message, document_message]
    # OpenAI API 호출하여 적절한 질문 생성
    result = client_gpt.chat.completions.create(
        model="gpt-4o",  # 사용할 모델 지정
        messages=full_message,
        temperature=0
    )
    
    
    print(document['content'])
    # LLM이 생성한 응답을 가져오기
    llm_response = result.choices[0].message.content

    if llm_response == 'False':
        print("Logical Fallacy")
    else:
        print("Good Doc")

    return llm_response
    # 논리적 오류와 상식적 타당성을 반환하거나, 논리적인 질문 5개를 반환
    # if "논리적 오류가 있으며, 상식적이지 않습니다" in llm_response:
    #     print("문서가 논리적 오류가 있으며, 상식적이지 않습니다.")
    #     return "이 문서는 논리적 오류가 있으며, 상식적이지 않습니다."
    # else:
    #     # 논리적인 질문 5개를 반환
    #     questions = llm_response.strip().split("\n")
    #     print(f"생성된 질문들: {questions}")
    #     return questions

In [10]:
def document_relevance_checker(document, query):
    """
    LLM을 사용하여 문서에서 논리적 오류, 상식적 타당성, 과학적 평가, 수학적 평가 및 쿼리 연관도를 바탕으로 점수를 계산합니다.
    각 항목에서 0~200점 사이의 점수를 부여하여 최종 점수를 반환합니다.
    """

    # 프롬프트에 평가 기준과 점수화 요구를 설정
    system_message = {
        "role": "system",
        "content": """
        당신은 문서의 적합성을 평가하는 전문가입니다.
        각 항목에 대해 0~200점 사이의 점수를 부여하세요:

        **1. 논리적 오류 평가**:
            - 문서에서 과학적 원리와 모순되거나 논리적 오류가 있으면 감점하세요.
            - 예: '해머로 철벽을 치는 것이 열을 발생시키기 위한 목적이다'라는 설명은 논리적 오류입니다.
            - 논리적으로 완벽할 경우 200점, 일부 모순이 있을 경우 100~150점, 명백한 오류가 있을 경우 0~50점을 부여하세요.

        **2. 상식적 타당성 평가**:
            - 설명이 상식적으로 타당한지 평가하세요.
            - 예: '가스레인지를 이용해 집안을 따뜻하게 한다'는 설명은 상식적이지 않으며, 감점 대상입니다.
            - 상식적으로 완벽할 경우 200점, 다소 현실성 부족한 경우 100~150점, 상식적으로 어긋날 경우 0~50점을 부여하세요.

        **3. 과학적 평가**:
            - 문서의 설명이 학계에서 널리 받아들여지는 과학적 사실과 일치하는지 평가하세요.
            - 예: 생식세포의 염색체 수가 23쌍(46개)라는 사실은 맞지만, '염색체의 수가 두 배가 된다'는 해석은 잘못된 것입니다.
            - 과학적으로 타당할 경우 200점, 일부 오해의 소지가 있을 경우 100~150점, 명백한 오류일 경우 0~50점을 부여하세요.

        **4. 수학적 평가**:
            - 문서에서 수학적 오류나 논리적 모순이 있는지 평가하세요.
            - 예: '매년 13개월의 월급을 지급합니다'는 수학적으로 잘못된 표현입니다.
            - 수학적으로 타당할 경우 200점, 일부 계산 오류가 있을 경우 100~150점, 명백한 오류일 경우 0~50점을 부여하세요.

        **5. 쿼리 연관도 평가**:
            - 질문(query)과 문서의 내용이 얼마나 연관되어 있는지 평가하세요.
            - 문서가 질문과 밀접하게 연관되어 있을 경우 200점, 일부 관련이 있을 경우 100~150점, 관련성이 거의 없을 경우 0~50점을 부여하세요.

        **결과**:
            - 각 항목의 점수를 0~200점 사이로 부여하고, 최종 점수를 계산하세요.
            - 최종 점수를 정수로 숫자로 답하시오.
            - 숫자만 생성하세요.
            - 최종 점수 외엔 아무것도 생성하지 마세요
        """
    }

    # 사용자 쿼리 및 문서 메시지
    query_message = {"role": "user", "content": f"질문: {query}"}
    document_message = {"role": "user", "content": f"문서 내용: {document['content']}"}

    # LLM 호출을 위한 전체 메시지 배열 생성
    full_message = [system_message, query_message, document_message]

    # OpenAI API를 사용하여 LLM 호출
    result = client_gpt.chat.completions.create(
        model="gpt-4o",  # 사용할 모델 지정
        messages=full_message,
        temperature=0
    )
    print(document['content'])
    # LLM이 생성한 점수를 받아오기 (최종 점수는 1000점 만점)
    llm_response = result.choices[0].message.content.strip()
    
    import re
    scores = re.findall(r'\d+', llm_response)  # 모든 숫자 추출
    if scores:
        final_score = sum(map(int, scores)) // len(scores)  # 최종 점수 계산
    else:
        final_score = 0

    # 점수 출력
    print(f"문서 평가 최종 점수: {int(final_score)}/200")

    return final_score

In [645]:
document_relevance_checker(z)

TypeError: document_relevance_checker() missing 1 required positional argument: 'query'

In [468]:
z = {"docid": "41ca41ac-66e3-4a6b-a604-87bf8b3a8d4d", "src": "ko_ai2_arc__ARC_Challenge__test", "content": "한 건설 작업자가 해머로 단단한 철벽을 치려 했지만 벽은 움직이지 않았습니다. 작업자가 사용한 에너지의 일부는 열로 변환되었습니다. 철벽은 고강도 강재로 만들어져 있어서 해머로 강력하게 치더라도 움직이지 않는 것이 정상입니다. 철벽은 건물의 구조를 지탱하고 보호하기 위해 설계되었으며, 그 내부에는 철과 다른 금속들이 견고하게 결합되어 있습니다. 따라서 해머로 치는 작업은 벽을 움직이게 하는 것보다는 에너지를 전달하여 열로 변환시키는 역할을 합니다. 이러한 열 변환은 작업자가 사용한 에너지를 벽에 흡수시키고, 벽의 온도를 조절하는 역할을 합니다. 따라서 작업자가 해머로 철벽을 치는 것은 벽을 움직이게 하는 것보다는 벽과 상호작용하여 열을 발생시키는 과정입니다."}


In [688]:
retreived_doc_ids

[('41ca41ac-66e3-4a6b-a604-87bf8b3a8d4d', -0.5539787411689758),
 ('8132c4ab-f1bd-4915-8196-a13fc3df1f05', -3.493497610092163),
 ('6211fd17-f7cf-4680-82be-365da49b211e', -4.1826372146606445),
 ('a395b38f-af11-409e-9c17-89a21f4a9521', -4.888504981994629),
 ('96459005-ecc6-4db9-914a-14043f280a08', -5.461760997772217)]

인간의 DNA 결합 과정
기름과 물의 혼합 가능성
금성에서 달의 관측 모습

In [877]:
final_ids = []
final_ids_2 = []
for doc_id in retreived_doc_ids:
    current_doc = doc_mapping[doc_id[0]]
    checked = document_relevance_checker(current_doc, '금성에서 달의 관측 모습')
    final_ids.append((doc_id[0], checked))
    # if checked == 'True':
    #     final_ids.append(doc_id[0])
    # else:
    #     final_ids_2.append(doc_id[0])

당신은 금성에 살고 있고, 당신의 망원경이 금성의 두꺼운 구름을 볼 수 있다고 가정한다면, 지구의 달을 관찰할 수 있습니다. 지구의 달은 월식과 만월을 거치며 다양한 단계를 보여줍니다. 여기서는 월식이 아닌 단계를 중심으로 설명하겠습니다.

첫 번째로, 상현 단계를 볼 수 있습니다. 이 단계에서는 달이 점점 커지면서 반달 모양을 보여줍니다. 달의 오른쪽 절반은 밝고, 왼쪽 절반은 어둡습니다. 이는 달이 점점 지구와 태양 사이의 각도가 커지면서 발생하는 현상입니다.

두 번째로, 망 단계를 볼 수 있습니다. 이 단계에서는 달이 지구와 태양 사이에 정확히 위치하여 완전한 원형을 보여줍니다. 이는 달이 지구와 태양 사이의 각도가 180도가 되는 순간입니다. 이때 달은 가장 밝게 보입니다.

세 번째로, 하현 단계를 볼 수 있습니다. 이 단계에서는 달이 점점 작아지면서 반달 모양을 보여줍니다. 달의 왼쪽 절반은 밝고, 오른쪽 절반은 어둡습니다. 이는 달이 점점 지구와 태양 사이의 각도가 작아지면서 발생하는 현상입니다.

따라서, 당신이 금성에 살고 있다면 상현, 망, 하현 단계의 지구의 달을 관찰할 수 있을 것입니다.
문서 평가 최종 점수: 0/1000
달이 정오에 지면 달의 위상은 상현달입니다. 상현달은 달의 위상 중 하나로, 달이 지구와 태양 사이에 위치하여 태양의 빛을 받아 반사되는 달의 모습을 말합니다. 이 때 달은 지구에서 볼 때 반달 모양으로 보이며, 달의 오른쪽 절반은 밝고 왼쪽 절반은 어둡게 보입니다. 이러한 상현달은 달의 위상 중 가장 두드러진 모습으로 알려져 있습니다. 달의 위상은 달이 지구 주위를 공전하면서 변하는데, 이는 태양과 달의 상대적인 위치에 따라 결정됩니다. 따라서 달의 위상은 매일 변하며, 상현달은 그 중 하나입니다. 상현달은 우리가 보는 달의 모습을 통해 달의 움직임과 태양계의 운동을 이해하는 데 도움을 줍니다.
문서 평가 최종 점수: 0/1000
금성이 다른 행성들보다 더 밝게 보이는 이유는 지구 쪽으로 가장 많은 햇빛을 반사하기 

In [878]:
sorted_ids = sorted(final_ids, key=lambda x: x[1], reverse=True)


In [879]:
sorted_ids

[('35c5dcc7-4720-4318-901e-770105ae63fd', 0),
 ('553989d9-ee23-4203-b244-a941b6fa8d99', 0),
 ('464ace62-ddf2-423d-a5d7-2f17e6785c8e', 0),
 ('8a78364e-63bf-4915-b718-fdc461bc62c9', 0),
 ('59a8259f-4a39-4ab6-ad3f-2e4161ad458d', 0)]

In [880]:
final_ids

[('35c5dcc7-4720-4318-901e-770105ae63fd', 0),
 ('553989d9-ee23-4203-b244-a941b6fa8d99', 0),
 ('464ace62-ddf2-423d-a5d7-2f17e6785c8e', 0),
 ('8a78364e-63bf-4915-b718-fdc461bc62c9', 0),
 ('59a8259f-4a39-4ab6-ad3f-2e4161ad458d', 0)]

#### 5. Top Agent

In [885]:
data = [
    ('a', 100),
    ('b', 50),
    ('c', 150),
    ('d', 30),
    ('e', 200),
    ('f', 75)
]

data[1:]

[('b', 50), ('c', 150), ('d', 30), ('e', 200), ('f', 75)]

In [886]:
data = [
    ('a', 100),
    ('b', 50),
    ('c', 150),
    ('d', 30),
    ('e', 200),
    ('f', 75)
]

# 슬라이스 범위 지정 (예: 인덱스 1부터 4까지 정렬)
start_index = 1
end_index = 4

# 슬라이스 부분을 정렬 (두 번째 요소 기준으로 오름차순 정렬)
sorted_slice = sorted(data[start_index:], key=lambda x: x[1])

# 정렬된 슬라이스를 원본 리스트의 해당 위치에 삽입
data[start_index:] = sorted_slice

In [887]:
data

[('a', 100), ('d', 30), ('b', 50), ('f', 75), ('c', 150), ('e', 200)]

In [10]:
def top_agent(queries):
    if len(queries) > 1:
        transformed_query = query_transformer(queries)
        query = [
            {"role": "user", "content": transformed_query}
        ]
        checked_query = science_query_detector(query)
        
        if checked_query == '과학 관련 질문이 아닙니다.':
            return False
    else:
        checked_query = science_query_detector(queries)
        if checked_query == '과학 관련 질문이 아닙니다.':
            return False

    retrieved_doc = retrieval(checked_query)
    
    
    final_result = [checked_query]
    for doc_id in retrieved_doc:
        current_doc = doc_mapping[doc_id]
        checked = document_relevance_checker(current_doc, checked_query)
        final_result.append((doc_id, checked))
        
    sorted_slice = sorted(final_result[1:], key=lambda x: x[1], reverse=True)
    final_result[1:] = sorted_slice
    return final_result

In [910]:
top_agent_result = top_agent(messages)

변환된 쿼리: 기억 상실증의 원인
변환된 쿼리: 기억 상실증의 원인
상위 5개의 문서:
기억 상실은 대뇌의 기능 장애로 인해 발생할 가능성이 가장 높습니다. 대뇌는 인간의 중추신경계에서 가장 중요한 역할을 담당하며, 인지, 기억, 감정 등 다양한 기능을 조절합니다. 따라서 대뇌의 어느 부분에서 문제가 발생하면 기억 상실이나 기타 인지 기능의 장애가 발생할 수 있습니다. 대뇌는 뇌의 가장 큰 부분으로, 뇌의 표면에 위치한 뇌피질과 그 아래에 있는 심층뇌로 구성되어 있습니다. 이러한 대뇌의 구조와 기능이 원활하게 작동하지 않으면 기억 상실이 발생할 수 있습니다. 따라서 대뇌는 기억 상실과 관련된 가장 중요한 부분으로 간주됩니다.
문서 평가 최종 점수: 200/200
치매는 뇌손상이 되돌릴 수 없고 증가하고 있는 질환입니다. 이 질환은 뇌 기능의 저하와 기억력 손실을 초래합니다. 치매는 주로 노인들에게서 발생하지만, 어떤 경우에는 어린이나 중년층에서도 발생할 수 있습니다. 치매는 점진적으로 증상이 악화되며, 일상 생활에 심각한 영향을 미칩니다. 이 질환은 현재로서는 완전히 치료할 수 없지만, 조기 발견과 관리를 통해 증상의 진행을 늦출 수 있습니다. 치매에 대한 연구는 계속 진행되고 있으며, 예방과 치료에 대한 새로운 방법들이 개발되고 있습니다. 치매는 많은 사람들에게 영향을 미치는 질환으로, 사회적인 문제로도 대두되고 있습니다. 따라서 치매에 대한 인식과 이해를 높이는 것이 중요합니다. 치매는 현재로서는 완전히 치료할 수 없지만, 연구와 노력을 통해 이 질환을 극복할 수 있는 방법들이 발전될 것을 기대합니다.
문서 평가 최종 점수: 50/200
일화적 기억은 나이가 들수록 가장 큰 결핍을 보이는 기억 유형입니다. 이는 일상적인 사건이나 경험에 대한 기억을 의미합니다. 나이가 들면서 기억력이 저하되고, 일화적인 기억을 잘 기억하지 못하는 경우가 많습니다. 이는 노화로 인해 뇌의 기능이 감소하고, 정보 처리 속도가 느려지는 것과 관련이 있습니다. 따라서, 일화적 기억

In [908]:
message = [
    {"role": "user", "content": "요새 너무 힘들다."}
]

In [911]:
top_agent_result

['기억 상실증의 원인',
 ('25de4ffd-cee4-4f27-907e-fd6b802c6ede', 200),
 ('df495f22-6315-42a8-9553-b43ab707b683', 100),
 ('2566f872-32e5-4def-8ed9-3341740e81e8', 100),
 ('0c2d5267-638a-4954-8974-55d28a3fe600', 100),
 ('ad56325d-400f-416c-b5de-2053eedecac4', 50)]

In [903]:
top_agent_result

['기억 상실증 원인',
 ('25de4ffd-cee4-4f27-907e-fd6b802c6ede', 200),
 ('2566f872-32e5-4def-8ed9-3341740e81e8', 150),
 ('df495f22-6315-42a8-9553-b43ab707b683', 150),
 ('ad56325d-400f-416c-b5de-2053eedecac4', 50),
 ('0c2d5267-638a-4954-8974-55d28a3fe600', 50)]

In [218]:
message

[{'role': 'user', 'content': '글루텐이 에너지 흡수에 미치는 영향은?'}]

In [597]:
top_agent_result = ['query', ['doc1', 0.9], ['doc2', 0.8]]
print(top_agent_result[1:4])

[['doc1', 0.9], ['doc2', 0.8]]


In [221]:
eval_doc_mapping[0]

{'eval_id': 78,
 'msg': [{'role': 'user', 'content': '나무의 분류에 대해 조사해 보기 위한 방법은?'}]}

In [912]:
with open('/upstage-ai-advanced-ir7/data/eval.jsonl') as f, open('/upstage-ai-advanced-ir7/output_10_07.csv', "w") as of:
    for line in f:
        eval_doc = json.loads(line)
        top_agent_result = top_agent(eval_doc['msg'])
        if top_agent_result ==  False:
            output = {"eval_id": eval_doc['eval_id'], "standalone_query": '', "topk": [], "answer": eval_doc['msg'][0]['content'], "references": [] }
            print(eval_doc['msg'][0]['content'])

        else:   
            output = {"eval_id": eval_doc['eval_id'], "standalone_query": top_agent_result[0], "topk": [], "answer": doc_mapping[top_agent_result[1][0]]['content'], "references": [] }
            for doc in top_agent_result[1:4]:
                output['topk'].append(doc[0])
                # doc_mapping[doc[0]]['content']
                output['references'].append({"score" : doc[1], "content" : doc_mapping[doc[0]]['content']})
            print('*' * 100)
        of.write(f'{json.dumps(output, ensure_ascii=False)}\n')

변환된 쿼리: 나무의 분류 방법 연구
상위 5개의 문서:
한 학생이 다양한 종류의 나무를 조사하고 있습니다. 이 학생은 성장 속도, 온도 범위, 크기가 비슷한 두 나무를 발견했습니다. 그러나 이 두 나무의 잎과 꽃은 서로 다릅니다. 이러한 특징을 고려하면, 이 나무들은 대체로 같은 속에 속해 있을 것으로 추측됩니다. 같은 속에 속한 나무들은 종류별로 유사한 특징을 가지고 있으며, 이는 생물 분류학에서 중요한 기준 중 하나입니다. 따라서 이 학생의 조사 결과는 나무의 분류와 관련된 중요한 정보를 제공할 수 있습니다. 이러한 조사는 나무의 성장과 생태에 대한 이해를 높이는 데 도움이 될 것입니다.
문서 평가 최종 점수: 190/200
나무는 재생 가능한 에너지원으로 간주됩니다. 이는 나무가 석탄이 형성되는 것보다 빠르게 자라기 때문입니다. 나무는 태양 에너지를 흡수하여 광합성을 통해 자라납니다. 이 과정에서 나무는 이산화탄소를 흡수하고 산소를 방출합니다. 이러한 특성으로 인해 나무는 지속적으로 재생되는 에너지원으로 사용될 수 있습니다. 반면, 석탄은 수백만 년 동안 지속적인 압력과 열에 노출되어 형성됩니다. 석탄은 화석 연료로 분류되며, 한 번 사용되면 재생되지 않습니다. 따라서 나무는 석탄보다 더욱 지속 가능한 에너지원으로 간주됩니다.
문서 평가 최종 점수: 0/200
찰스 다윈은 젊은 생물학자로서 남미의 온대 지역 식물과 유럽의 온대 지역 식물이 유사할 것으로 예상했습니다. 그러나 그는 놀랍게도 남미의 열대 지역 식물과 더 유사하다는 것을 발견했습니다. 이 관찰은 생물지리학과 가장 적절하게 연관되어 있습니다. 생물지리학은 생물의 분포와 분류를 연구하는 학문으로, 지리적인 요인이 생물의 진화와 다양성에 어떤 영향을 미치는지를 연구합니다. 다윈의 관찰은 지리적인 요인이 식물의 진화에 영향을 미치는 것을 보여주었으며, 이는 생물지리학의 중요성을 강조합니다.
문서 평가 최종 점수: 50/200
축축한 습지에서 자라고 있는 식물은 비종자 유관속식물로 분류됩니다. 비종자

In [14]:
with open('/upstage-ai-advanced-ir7/data/eval.jsonl') as f, open('/upstage-ai-advanced-ir7/output_10_08.csv', "w") as of:
    for line in f:
        eval_doc = json.loads(line)
        top_agent_result = top_agent(eval_doc['msg'])
        if top_agent_result ==  False:
            output = {"eval_id": eval_doc['eval_id'], "standalone_query": '', "topk": [], "answer": eval_doc['msg'][0]['content'], "references": [] }
            print(eval_doc['msg'][0]['content'])

        else:   
            output = {"eval_id": eval_doc['eval_id'], "standalone_query": top_agent_result[0], "topk": [], "answer": doc_mapping[top_agent_result[1][0]]['content'], "references": [] }
            for doc in top_agent_result[1:4]:
                output['topk'].append(doc[0])
                # doc_mapping[doc[0]]['content']
                output['references'].append({"score" : doc[1], "content" : doc_mapping[doc[0]]['content']})
            print('*' * 100)
        of.write(f'{json.dumps(output, ensure_ascii=False)}\n')

변환된 쿼리: '나무의 분류 방법 조사'
상위 5개의 문서:
한 학생이 다양한 종류의 나무를 조사하고 있습니다. 이 학생은 성장 속도, 온도 범위, 크기가 비슷한 두 나무를 발견했습니다. 그러나 이 두 나무의 잎과 꽃은 서로 다릅니다. 이러한 특징을 고려하면, 이 나무들은 대체로 같은 속에 속해 있을 것으로 추측됩니다. 같은 속에 속한 나무들은 종류별로 유사한 특징을 가지고 있으며, 이는 생물 분류학에서 중요한 기준 중 하나입니다. 따라서 이 학생의 조사 결과는 나무의 분류와 관련된 중요한 정보를 제공할 수 있습니다. 이러한 조사는 나무의 성장과 생태에 대한 이해를 높이는 데 도움이 될 것입니다.
문서 평가 최종 점수: 200/200
킨제이 척도는 성적 지향성을 측정하는 도구로, 사람들을 다양한 등급으로 분류합니다. 4등급을 가진 사람들은 상당한 이성애 경험이 있는 동성애자로 분류됩니다. 이 등급은 이성애와 동성애 경험을 모두 가지고 있으며, 이성애에 대한 강한 성적 지향성을 가지고 있습니다. 이 등급에 속하는 사람들은 동성애 경험이 있지만, 이성애에 대한 성적인 끌림이 여전히 존재합니다. 이러한 사람들은 성적 지향성이 다양하며, 이성애와 동성애 모두에 대한 성적인 욕구를 가지고 있을 수 있습니다. 킨제이 척도는 성적 지향성을 이해하고 분류하는 데 도움을 주는 중요한 도구입니다.
문서 평가 최종 점수: 0/200
나무는 재생 가능한 에너지원으로 간주됩니다. 이는 나무가 석탄이 형성되는 것보다 빠르게 자라기 때문입니다. 나무는 태양 에너지를 흡수하여 광합성을 통해 자라납니다. 이 과정에서 나무는 이산화탄소를 흡수하고 산소를 방출합니다. 이러한 특성으로 인해 나무는 지속적으로 재생되는 에너지원으로 사용될 수 있습니다. 반면, 석탄은 수백만 년 동안 지속적인 압력과 열에 노출되어 형성됩니다. 석탄은 화석 연료로 분류되며, 한 번 사용되면 재생되지 않습니다. 따라서 나무는 석탄보다 더욱 지속 가능한 에너지원으로 간주됩니다.
문서 평가 최종 점수: 0/200
나무는 시

x =

In [16]:
def top_agent_without_relevance_check(queries):
    if len(queries) > 1:
        transformed_query = query_transformer(queries)
        query = [
            {"role": "user", "content": transformed_query}
        ]
        checked_query = science_query_detector(query)
        
        if checked_query == '과학 관련 질문이 아닙니다.':
            return False
    else:
        checked_query = science_query_detector(queries)
        if checked_query == '과학 관련 질문이 아닙니다.':
            return False

    retrieved_doc = retrieval_with_score(checked_query)
    
    
    final_result = [checked_query]
    for doc in retrieved_doc:
        # current_doc = doc_mapping[doc[0]]
        # checked = document_relevance_checker(current_doc, checked_query)
        final_result.append((doc[0], doc[1]))
        
    # sorted_slice = sorted(final_result[1:], key=lambda x: x[1], reverse=True)
    # final_result[1:] = sorted_slice
    return final_result

In [17]:
with open('/upstage-ai-advanced-ir7/data/eval.jsonl') as f, open('/upstage-ai-advanced-ir7/output_10_11_2.csv', "w") as of:
    for line in f:
        eval_doc = json.loads(line)
        top_agent_result = top_agent_without_relevance_check(eval_doc['msg'])
        if top_agent_result ==  False:
            output = {"eval_id": eval_doc['eval_id'], "standalone_query": '', "topk": [], "answer": eval_doc['msg'][0]['content'], "references": [] }
            print(eval_doc['msg'][0]['content'])

        else:   
            output = {"eval_id": eval_doc['eval_id'], "standalone_query": top_agent_result[0], "topk": [], "answer": doc_mapping[top_agent_result[1][0]]['content'], "references": [] }
            for doc in top_agent_result[1:6]:
                output['topk'].append(doc[0])
                # doc_mapping[doc[0]]['content']
                output['references'].append({"score" : doc[1], "content" : doc_mapping[doc[0]]['content']})
            print('*' * 100)
        of.write(f'{json.dumps(output, ensure_ascii=False)}\n')

변환된 쿼리: 나무의 분류 방법
상위 5개의 문서:
****************************************************************************************************
변환된 쿼리: 각 나라의 공교육 지출 현황
상위 5개의 문서:
****************************************************************************************************
변환된 쿼리: 기억 상실증 원인
변환된 쿼리: 기억 상실증의 원인
상위 5개의 문서:
****************************************************************************************************
변환된 쿼리: 통학 버스의 가치
상위 5개의 문서:
****************************************************************************************************
변환된 쿼리: Dmitri Ivanovsky
상위 5개의 문서:
****************************************************************************************************
변환된 쿼리: 피임약의 효과와 사용법
상위 5개의 문서:
****************************************************************************************************
변환된 쿼리: 헬륨의 화학적 비활성 이유
상위 5개의 문서:
****************************************************************************************************
변환된 쿼리: 문맹 비율이 사회 발전에 미치는 영향
상위 5개의 문서:
**********

#### 일할거리

In [106]:
result

[('fcfee923-af42-43d1-b664-09a0112e4f74', 5.381111486045304),
 ('8a78364e-63bf-4915-b718-fdc461bc62c9', 4.45264525239844),
 ('cefe7caf-6cd1-422a-b41e-e82b543556e9', 4.400076349872069),
 ('892eb0be-055f-47c5-8895-1afb97c8f406', 3.822421126579113),
 ('340485f8-4e78-44f4-a53a-2df21915367f', 2.8626556358845967)]

In [105]:
check_list

[('fcfee923-af42-43d1-b664-09a0112e4f74', 5.381111486045304),
 ('8a78364e-63bf-4915-b718-fdc461bc62c9', 4.45264525239844),
 ('cefe7caf-6cd1-422a-b41e-e82b543556e9', 4.400076349872069),
 ('892eb0be-055f-47c5-8895-1afb97c8f406', 3.822421126579113)]

In [111]:
llm_result

'[2, 3, 1, 4, 5]'

In [112]:
import ast

# 문자열을 리스트로 변환
string = "[2, 3, 1, 4, 5]"
result_lists = ast.literal_eval(string)

print(result_lists)  # 출력: [2, 3, 1, 4, 5]


[2, 3, 1, 4, 5]


In [113]:
llm_result = ast.literal_eval(llm_result)

In [114]:
llm_result

[2, 3, 1, 4, 5]

In [132]:
result[55:]

[]

In [133]:
new_result + result[55:]

[('8a78364e-63bf-4915-b718-fdc461bc62c9', 4.45264525239844),
 ('cefe7caf-6cd1-422a-b41e-e82b543556e9', 4.400076349872069),
 ('fcfee923-af42-43d1-b664-09a0112e4f74', 5.381111486045304),
 ('892eb0be-055f-47c5-8895-1afb97c8f406', 3.822421126579113),
 ('340485f8-4e78-44f4-a53a-2df21915367f', 2.8626556358845967)]

In [128]:
new_result = []
for i ,f_r in enumerate(llm_result):
    # print(result[f_r  - 1])
    new_result.append(result[f_r - 1])

In [135]:
2 > 1

True

In [129]:
new_result

[('8a78364e-63bf-4915-b718-fdc461bc62c9', 4.45264525239844),
 ('cefe7caf-6cd1-422a-b41e-e82b543556e9', 4.400076349872069),
 ('fcfee923-af42-43d1-b664-09a0112e4f74', 5.381111486045304),
 ('892eb0be-055f-47c5-8895-1afb97c8f406', 3.822421126579113),
 ('340485f8-4e78-44f4-a53a-2df21915367f', 2.8626556358845967)]

In [205]:
def top_agent_without_relevance_check(queries):
    if len(queries) > 1:
        transformed_query = query_transformer(queries)
        query = [
            {"role": "user", "content": transformed_query}
        ]
        checked_query = science_query_detector(query)
        
        if checked_query == '과학 관련 질문이 아닙니다.':
            return False
    else:
        checked_query = science_query_detector(queries)
        if checked_query == '과학 관련 질문이 아닙니다.':
            return False

    retrieved_doc = retrieval_with_score(checked_query)
    
    check_list = []

    for doc in retrieved_doc:
        print(retrieved_doc[0][1] * 0.3)
        print(doc[1])
        print('*' * 100)
        if doc[1] > retrieved_doc[0][1] * 0.3:
            check_list.append(doc)
    
    if len(check_list) > 1:
        llm_result = llm_reranking(checked_query, check_list)
        llm_result = ast.literal_eval(llm_result)
            
        
        new_result = []
        for f_r in llm_result:
            # print(result[f_r  - 1])
            new_result.append(retrieved_doc[f_r - 1])

        retrieved_doc = new_result + retrieved_doc[len(new_result):]
    
    
    final_result = [checked_query]
    for doc in retrieved_doc:
        # current_doc = doc_mapping[doc[0]]
        # checked = document_relevance_checker(current_doc, checked_query)
        final_result.append((doc[0], doc[1]))
        
    # sorted_slice = sorted(final_result[1:], key=lambda x: x[1], reverse=True)
    # final_result[1:] = sorted_slice
    return final_result

In [ ]:
{"eval_id": 43, "msg": [{"role": "user", "content": "달을 보면 항상 같은 면만 보이더라구"}, {"role": "assistant", "content": "네 맞습니다."}, {"role": "user", "content": "그 이유가 뭐야?"}]}

In [149]:
let_see_2 = top_agent_without_relevance_check([{"role": "user", "content": "나무의 분류에 대해 조사해 보기 위한 방법은?"}])

변환된 쿼리: 나무의 분류 방법
상위 5개의 문서:
3.3107282560050293
5.255124215880999
****************************************************************************************************
3.3107282560050293
2.692293622697727
****************************************************************************************************
3.3107282560050293
2.676450241618026
****************************************************************************************************
3.3107282560050293
1.9075196258003642
****************************************************************************************************
3.3107282560050293
1.881075850469356
****************************************************************************************************


In [151]:
let_see

['달의 한쪽 면만 보이는 이유',
 ('8a78364e-63bf-4915-b718-fdc461bc62c9', 4.45264525239844),
 ('cefe7caf-6cd1-422a-b41e-e82b543556e9', 4.400076349872069),
 ('fcfee923-af42-43d1-b664-09a0112e4f74', 5.381111486045304),
 ('892eb0be-055f-47c5-8895-1afb97c8f406', 3.822421126579113),
 ('340485f8-4e78-44f4-a53a-2df21915367f', 2.8626556358845967)]

In [150]:
let_see_2

['나무의 분류 방법',
 ('c63b9e3a-716f-423a-9c9b-0bcaa1b9f35d', 5.255124215880999),
 ('a9f2c21e-9d44-4dd5-bc02-9e4f84077139', 2.692293622697727),
 ('8018337f-15cb-4341-b6fa-e311b4372df9', 2.676450241618026),
 ('9712bdf6-9419-4953-a8f1-8a4015dee986', 1.9075196258003642),
 ('35395c59-d1e0-4b63-803c-590220e906ad', 1.881075850469356)]

In [ ]:
[{"role": "user", "content": "나무의 분류에 대해 조사해 보기 위한 방법은?"}]

In [185]:
let_see = top_agent_without_relevance_check([{"role": "user", "content": "달을 보면 항상 같은 면만 보이더라구"}, {"role": "assistant", "content": "네 맞습니다."}, {"role": "user", "content": "그 이유가 뭐야?"}])

[{'role': 'user', 'content': '달을 보면 항상 같은 면만 보이더라구'}, {'role': 'assistant', 'content': '네 맞습니다.'}, {'role': 'user', 'content': '그 이유가 뭐야?'}]
[{'role': 'system', 'content': '\n        당신은 사용자의 여러 대화 메시지를 하나의 검색 쿼리로 변환하는 전문가입니다. \n        사용자의 대화 중 중요한 정보만을 추출하여 간결하고 명확한 검색용 쿼리로 변환하세요. \n        대화 형식이 아닌, 연구나 조사를 위한 검색어처럼 정확한 질문을 생성하세요.\n        절대 답을 생성하지 마시오.\n        '}, {'role': 'user', 'content': '달을 보면 항상 같은 면만 보이더라구'}, {'role': 'assistant', 'content': '네 맞습니다.'}, {'role': 'user', 'content': '그 이유가 뭐야?'}]
변환된 쿼리: 달의 한쪽 면만 보이는 이유
변환된 쿼리: 달의 한쪽 면만 보이는 이유
상위 5개의 문서:
1.6143334458135912
5.381111486045304
****************************************************************************************************
1.6143334458135912
4.45264525239844
****************************************************************************************************
1.6143334458135912
4.400076349872069
****************************************************************************************************
1.6143334458

In [206]:
with open('/upstage-ai-advanced-ir7/data/eval.jsonl') as f, open('/upstage-ai-advanced-ir7/output_10_15_2.csv', "w") as of:
    for line in f:
        eval_doc = json.loads(line)
        top_agent_result = top_agent_without_relevance_check(eval_doc['msg'])
        if top_agent_result ==  False:
            output = {"eval_id": eval_doc['eval_id'], "standalone_query": '', "topk": [], "answer": eval_doc['msg'][0]['content'], "references": [] }
            print(eval_doc['msg'][0]['content'])

        else:   
            output = {"eval_id": eval_doc['eval_id'], "standalone_query": top_agent_result[0], "topk": [], "answer": doc_mapping[top_agent_result[1][0]]['content'], "references": [] }
            for doc in top_agent_result[1:6]:
                output['topk'].append(doc[0])
                # doc_mapping[doc[0]]['content']
                output['references'].append({"score" : doc[1], "content" : doc_mapping[doc[0]]['content']})
            print('*' * 100)
        of.write(f'{json.dumps(output, ensure_ascii=False)}\n')

변환된 쿼리: 나무의 분류 방법
상위 5개의 문서:
1.5765372647642997
5.255124215880999
****************************************************************************************************
1.5765372647642997
2.692293622697727
****************************************************************************************************
1.5765372647642997
2.676450241618026
****************************************************************************************************
1.5765372647642997
1.9075196258003642
****************************************************************************************************
1.5765372647642997
1.881075850469356
****************************************************************************************************
변환된 쿼리: [1, 4, 5]
****************************************************************************************************
변환된 쿼리: 각 나라의 공교육 지출 현황
상위 5개의 문서:
2.850523493379814
9.501744977932713
******************************************************************************************

In [143]:
let_see_2

['나무의 분류 방법',
 ('c63b9e3a-716f-423a-9c9b-0bcaa1b9f35d', 5.255124215880999),
 ('a9f2c21e-9d44-4dd5-bc02-9e4f84077139', 2.692293622697727),
 ('8018337f-15cb-4341-b6fa-e311b4372df9', 2.676450241618026),
 ('9712bdf6-9419-4953-a8f1-8a4015dee986', 1.9075196258003642),
 ('35395c59-d1e0-4b63-803c-590220e906ad', 1.881075850469356)]

In [146]:
let_see

['달의 한쪽 면만 보이는 이유',
 ('8a78364e-63bf-4915-b718-fdc461bc62c9', 4.45264525239844),
 ('cefe7caf-6cd1-422a-b41e-e82b543556e9', 4.400076349872069),
 ('fcfee923-af42-43d1-b664-09a0112e4f74', 5.381111486045304),
 ('892eb0be-055f-47c5-8895-1afb97c8f406', 3.822421126579113),
 ('340485f8-4e78-44f4-a53a-2df21915367f', 2.8626556358845967)]

In [187]:
top_agent_result = top_agent_without_relevance_check([{"role": "user", "content": "사람이나 물체가 지구 위에서 땅속으로 꺼지거나 바깥으로 튕겨나가지 않고 가만히 서 있을 수 있잖아?"}, {"role": "assistant", "content": "네 맞습니다."}, {"role": "user", "content": "그 이유를 힘의 원리로 설명해줘."}])

변환된 쿼리: 중력과 지구의 표면에서의 정지 상태에 대한 힘의 균형 원리
변환된 쿼리: 중력과 지구의 표면에서의 정지 상태에 대한 힘의 균형 원리
상위 5개의 문서:


In [188]:
output = {"eval_id": 243, "standalone_query": top_agent_result[0], "topk": [], "answer": doc_mapping[top_agent_result[1][0]]['content'], "references": [] }
for doc in top_agent_result[1:4]:
    output['topk'].append(doc[0])
    # doc_mapping[doc[0]]['content']
    output['references'].append({"score" : doc[1], "content" : doc_mapping[doc[0]]['content']})

In [189]:
print(output)

{'eval_id': 243, 'standalone_query': '중력과 지구의 표면에서의 정지 상태에 대한 힘의 균형 원리', 'topk': ['92cb353a-89e7-45fe-8c99-debf0f33bba9', '5ff8f00a-a4e6-43fd-8616-3104a4c4d637', '9e4969bb-8a41-4719-bb22-5b89d223daf9'], 'answer': '평형의 힘은 물체가 정지하거나 일정한 속도로 운동할 때 작용하는 힘입니다. 이는 물체에 작용하는 모든 외력의 합이 0이 되는 상태를 말합니다. 예를 들어, 차고에 주차된 버스는 평형의 힘을 나타내는 좋은 예시입니다. 버스가 차고에 주차되어 있을 때, 지면과 버스 사이에 작용하는 중력과 지지력이 서로 상쇄되어 평형을 이루게 됩니다. 이렇게 평형의 힘이 작용하면 물체는 움직이지 않고 정지한 상태를 유지합니다. 따라서 차고에 주차된 버스는 평형의 힘에 대한 가장 알맞은 예시입니다.', 'references': [{'score': 0.9762017039637478, 'content': '평형의 힘은 물체가 정지하거나 일정한 속도로 운동할 때 작용하는 힘입니다. 이는 물체에 작용하는 모든 외력의 합이 0이 되는 상태를 말합니다. 예를 들어, 차고에 주차된 버스는 평형의 힘을 나타내는 좋은 예시입니다. 버스가 차고에 주차되어 있을 때, 지면과 버스 사이에 작용하는 중력과 지지력이 서로 상쇄되어 평형을 이루게 됩니다. 이렇게 평형의 힘이 작용하면 물체는 움직이지 않고 정지한 상태를 유지합니다. 따라서 차고에 주차된 버스는 평형의 힘에 대한 가장 알맞은 예시입니다.'}, {'score': 0.7453971776911065, 'content': '지구의 중력이 발생하는 원인은 지구의 질량이다. 중력은 모든 물체 간에 작용하는 자연 법칙으로, 지구의 질량이 크기 때문에 지구 주변에 있는 물체들에게 중력을 발생시킨다. 이러한 중력은 물체를 지구로 끌어당기는 힘을 가지고 있으며, 이로 인해 물체들은 지구의

In [86]:
top_agent_result

['25de4ffd-cee4-4f27-907e-fd6b802c6ede',
 'ad56325d-400f-416c-b5de-2053eedecac4',
 'df495f22-6315-42a8-9553-b43ab707b683',
 '2566f872-32e5-4def-8ed9-3341740e81e8',
 '0c2d5267-638a-4954-8974-55d28a3fe600']

In [43]:
top_agent_result

'기억 상실증 원인 연구'

In [207]:
with open("/upstage-ai-advanced-ir7/data/eval.jsonl", "r") as f:
    eval_doc_mapping = [json.loads(line) for line in f]
    

In [211]:
for eval_doc in eval_doc_mapping:
    top_agent_result = top_agent(messages)

{'eval_id': 107,
 'msg': [{'role': 'user', 'content': '기억 상실증 걸리면 너무 무섭겠다.'},
  {'role': 'assistant', 'content': '네 맞습니다.'},
  {'role': 'user', 'content': '어떤 원인 때문에 발생하는지 궁금해.'}]}

In [212]:
messages

[{'role': 'user', 'content': '기억 상실증 걸리면 너무 무섭겠다.'},
 {'role': 'assistant', 'content': '네 맞습니다.'},
 {'role': 'user', 'content': '원인이 뭘까.'}]

[{'eval_id': 78,
  'msg': [{'role': 'user', 'content': '나무의 분류에 대해 조사해 보기 위한 방법은?'}]},
 {'eval_id': 213,
  'msg': [{'role': 'user', 'content': '각 나라에서의 공교육 지출 현황에 대해 알려줘.'}]},
 {'eval_id': 107,
  'msg': [{'role': 'user', 'content': '기억 상실증 걸리면 너무 무섭겠다.'},
   {'role': 'assistant', 'content': '네 맞습니다.'},
   {'role': 'user', 'content': '어떤 원인 때문에 발생하는지 궁금해.'}]},
 {'eval_id': 81, 'msg': [{'role': 'user', 'content': '통학 버스의 가치에 대해 말해줘.'}]},
 {'eval_id': 280,
  'msg': [{'role': 'user', 'content': 'Dmitri Ivanovsky가 누구야?'}]},
 {'eval_id': 10,
  'msg': [{'role': 'user', 'content': '피임을 하기 위한 방법중 약으로 처리하는 방법은 쓸만한가?'}]},
 {'eval_id': 100,
  'msg': [{'role': 'user', 'content': '헬륨이 다른 원소들과 반응을 잘 안하는 이유는?'}]},
 {'eval_id': 279,
  'msg': [{'role': 'user', 'content': '문맹 비율이 사회 발전에 미치는 영향은?'}]},
 {'eval_id': 42,
  'msg': [{'role': 'user', 'content': '이란 콘트라 사건이 뭐야'},
   {'role': 'assistant',
    'content': '이란-콘트라 사건은 로널드 레이건 집권기인 1986년에 레이건 행정부와 CIA가 적성국이었던 이란에게 무기를 몰래 수출한 대금으로 니카라과의 우익 성향 반군 콘트라를 

In [65]:
def top_agent_without_relevance_check(queries):
    if len(queries) > 1:
        transformed_query = query_transformer(queries)
        query = [
            {"role": "user", "content": transformed_query}
        ]
        original_query = transformed_query
        checked_query = science_query_detector(query)
        
        if checked_query == '과학 관련 질문이 아닙니다.':
            return False
    else:
        original_query = queries[0]['content']
        checked_query = science_query_detector(queries)
        if checked_query == '과학 관련 질문이 아닙니다.':
            return False

    retrieved_doc = retrieval_with_score(checked_query)
    
    check_list = []

    for doc in retrieved_doc:
        print(retrieved_doc[0][1] * 0.1)
        print(doc[1])
        print('*' * 100)
        if doc[1] > retrieved_doc[0][1] * 0.1:
            check_list.append(doc)
    
    if len(check_list) > 1:
        llm_result = llm_reranking(original_query, check_list)
        llm_result = ast.literal_eval(llm_result)
            
        
        new_result = []
        for f_r in llm_result:
            # print(result[f_r  - 1])
            new_result.append(retrieved_doc[f_r - 1])

        retrieved_doc = new_result + retrieved_doc[len(new_result):]
    
    
    final_result = [checked_query]
    for doc in retrieved_doc:
        # current_doc = doc_mapping[doc[0]]
        # checked = document_relevance_checker(current_doc, checked_query)
        final_result.append((doc[0], doc[1]))
        
    # sorted_slice = sorted(final_result[1:], key=lambda x: x[1], reverse=True)
    # final_result[1:] = sorted_slice
    return final_result

In [66]:
with open('/upstage-ai-advanced-ir7/data/eval.jsonl') as f, open('/upstage-ai-advanced-ir7/output_10_24_1.csv', "w") as of:
    for line in f:
        eval_doc = json.loads(line)
        top_agent_result = top_agent_without_relevance_check(eval_doc['msg'])
        if top_agent_result ==  False:
            output = {"eval_id": eval_doc['eval_id'], "standalone_query": '', "topk": [], "answer": eval_doc['msg'][0]['content'], "references": [] }
            print(eval_doc['msg'][0]['content'])

        else:   
            output = {"eval_id": eval_doc['eval_id'], "standalone_query": top_agent_result[0], "topk": [], "answer": doc_mapping[top_agent_result[1][0]]['content'], "references": [] }
            for doc in top_agent_result[1:6]:
                output['topk'].append(doc[0])
                # doc_mapping[doc[0]]['content']
                output['references'].append({"score" : doc[1], "content" : doc_mapping[doc[0]]['content']})
            print('*' * 100)
        of.write(f'{json.dumps(output, ensure_ascii=False)}\n')

상위 5개의 문서:
0.722899238025441
7.228992380254409
****************************************************************************************************
0.722899238025441
3.58624425264097
****************************************************************************************************
0.722899238025441
2.4284540597010773
****************************************************************************************************
0.722899238025441
2.133000629733557
****************************************************************************************************
0.722899238025441
1.9653361618684921
****************************************************************************************************
0.722899238025441
1.7722815628400612
****************************************************************************************************
0.722899238025441
1.6851216809573566
****************************************************************************************************
0.722899238025441
1.6612740

In [16]:
with open('/upstage-ai-advanced-ir7/data/eval_4.jsonl') as f, open('/upstage-ai-advanced-ir7/output_10_21_4.csv', "w") as of:
    for line in f:
        eval_doc = json.loads(line)
        top_agent_result = top_agent_without_relevance_check(eval_doc['msg'])
        if top_agent_result ==  False:
            output = {"eval_id": eval_doc['eval_id'], "standalone_query": '', "topk": [], "answer": eval_doc['msg'][0]['content'], "references": [] }
            print(eval_doc['msg'][0]['content'])

        else:   
            output = {"eval_id": eval_doc['eval_id'], "standalone_query": top_agent_result[0], "topk": [], "answer": doc_mapping[top_agent_result[1][0]]['content'], "references": [] }
            for doc in top_agent_result[1:6]:
                output['topk'].append(doc[0])
                # doc_mapping[doc[0]]['content']
                output['references'].append({"score" : doc[1], "content" : doc_mapping[doc[0]]['content']})
            print('*' * 100)
        of.write(f'{json.dumps(output, ensure_ascii=False)}\n')

상위 5개의 문서:
0.4959049915416633
4.9590499154166325
****************************************************************************************************
0.4959049915416633
4.191615575456861
****************************************************************************************************
0.4959049915416633
3.6623332868153655
****************************************************************************************************
0.4959049915416633
3.6045192678971207
****************************************************************************************************
0.4959049915416633
2.143479742970571
****************************************************************************************************
0.4959049915416633
1.8661252555435532
****************************************************************************************************
0.4959049915416633
1.8141444233937305
****************************************************************************************************
0.4959049915416633

In [17]:
with open('/upstage-ai-advanced-ir7/data/eval_5.jsonl') as f, open('/upstage-ai-advanced-ir7/output_10_21_5.csv', "w") as of:
    for line in f:
        eval_doc = json.loads(line)
        top_agent_result = top_agent_without_relevance_check(eval_doc['msg'])
        if top_agent_result ==  False:
            output = {"eval_id": eval_doc['eval_id'], "standalone_query": '', "topk": [], "answer": eval_doc['msg'][0]['content'], "references": [] }
            print(eval_doc['msg'][0]['content'])

        else:   
            output = {"eval_id": eval_doc['eval_id'], "standalone_query": top_agent_result[0], "topk": [], "answer": doc_mapping[top_agent_result[1][0]]['content'], "references": [] }
            for doc in top_agent_result[1:6]:
                output['topk'].append(doc[0])
                # doc_mapping[doc[0]]['content']
                output['references'].append({"score" : doc[1], "content" : doc_mapping[doc[0]]['content']})
            print('*' * 100)
        of.write(f'{json.dumps(output, ensure_ascii=False)}\n')

상위 5개의 문서:
0.7720268838667121
7.7202688386671205
****************************************************************************************************
0.7720268838667121
5.176557684657467
****************************************************************************************************
0.7720268838667121
3.9702504859028585
****************************************************************************************************
0.7720268838667121
3.553695036619711
****************************************************************************************************
0.7720268838667121
2.4846550891406958
****************************************************************************************************
0.7720268838667121
2.143627967822791
****************************************************************************************************
0.7720268838667121
2.018080979663269
****************************************************************************************************
0.7720268838667121
1

In [18]:
with open('/upstage-ai-advanced-ir7/data/eval_6.jsonl') as f, open('/upstage-ai-advanced-ir7/output_10_21_6.csv', "w") as of:
    for line in f:
        eval_doc = json.loads(line)
        top_agent_result = top_agent_without_relevance_check(eval_doc['msg'])
        if top_agent_result ==  False:
            output = {"eval_id": eval_doc['eval_id'], "standalone_query": '', "topk": [], "answer": eval_doc['msg'][0]['content'], "references": [] }
            print(eval_doc['msg'][0]['content'])

        else:   
            output = {"eval_id": eval_doc['eval_id'], "standalone_query": top_agent_result[0], "topk": [], "answer": doc_mapping[top_agent_result[1][0]]['content'], "references": [] }
            for doc in top_agent_result[1:6]:
                output['topk'].append(doc[0])
                # doc_mapping[doc[0]]['content']
                output['references'].append({"score" : doc[1], "content" : doc_mapping[doc[0]]['content']})
            print('*' * 100)
        of.write(f'{json.dumps(output, ensure_ascii=False)}\n')

상위 5개의 문서:
0.5089602710265161
5.089602710265161
****************************************************************************************************
0.5089602710265161
4.477084539504587
****************************************************************************************************
0.5089602710265161
4.464051919720704
****************************************************************************************************
0.5089602710265161
3.41284373948993
****************************************************************************************************
0.5089602710265161
3.1054983085027095
****************************************************************************************************
0.5089602710265161
2.3760474886955745
****************************************************************************************************
0.5089602710265161
2.2250394705583876
****************************************************************************************************
0.5089602710265161
1.

KeyboardInterrupt: 

In [ ]:
def llm_reranking(query, documents):
    """
    LLM을 사용하여 사용자의 여러 메시지를 기반으로 검색에 적합한 단일 쿼리 생성.
    """
    # 시스템 메시지로 LLM에게 과제 부여 (한국어로)
    
    
    
    check_doc = ''
    for i, doc in enumerate(documents):
        check_doc += f"문서 {i+1}: {doc_mapping[doc[0]]['content']} \n"

    # 시스템 메시지 생성
    system_message = {
        "role": "system",
        "content": f"""
        당신은 문서와 질문을 비교하여, 질문에 가장 적합한 문서들을 찾고 그 순서를 반환하는 전문가입니다.
        
        다음의 규칙을 반드시 따르세요:
        1. 주어진 질문과 문서들의 내용을 비교하여, 질문의 답을 찾을 수 있는 문서를 가장 적합한 순서대로 나열하세요.
        2. 반드시 리스트 형태로만 출력하세요. 다른 형식은 허용되지 않습니다. 예를 들어 [1, 3, 2, 4]와 같이 반환하세요.
        3. 리스트에 들어갈 문서 번호는 1부터 시작하며, 문서들의 순서를 중요도에 따라 배열하세요.
        4. 리스트 외의 다른 정보를 출력하지 마세요. 리스트 이외의 내용은 모델이 자동으로 무시해야 합니다.

        질문: {query}

        문서들:
        {check_doc}
        """
    }

# 사용자 메시지 준비


    # 사용자 메시지 준비
    # system_message['content'] = system_message['content'] + check_doc

    # LLM 호출을 위한 메시지 배열 생성
    system_message = [system_message]

    # OpenAI API 호출하여 적절한 검색 쿼리 생성
    result = client_gpt.chat.completions.create(
        model="o1-preview",  # LLM 모델 지정
        messages=system_message,
        temperature=0
    )

    # LLM이 생성한 쿼리 반환 (ChatCompletionMessage 형식에서 content에 직접 접근)
    transformed_query = result.choices[0].message.content
    print(f"변환된 쿼리: {transformed_query}")
    return transformed_query

In [122]:
def llm_reranking(query, doc1, doc2):
    """
    LLM을 사용하여 두 개의 문서를 쿼리와 비교하고, 더 적합한 문서를 반환합니다.
    """
    # 문서들의 내용을 준비
    
    document1 = doc_mapping[doc1[0]]['content']
    document2 = doc_mapping[doc2[0]]['content']
    
    print(document1, document2)
    check_docs = f"""
    문서 1: {document1}
    문서 2: {document2}
    """

    # 시스템 메시지 생성
    system_message = {
        "role": "system",
        "content": f"""
        당신은 문서와 질문을 비교하여, 질문에 가장 적합한 문서를 찾는 전문가입니다.

        다음 규칙을 따르세요:
        1. 주어진 질문과 문서 1, 문서 2를 비교하여, 질문의 답을 찾을 수 있는 문서를 반드시 선택하세요.
        2. 문서 1이 질문에 더 적합하면 "True"라고 답하고, 문서 2가 질문에 더 적합하면 "False"라고 답하세요.
        3. 반드시 True 또는 False 중 하나만 답하세요. 둘 중 하나의 문서가 더 적합하다는 결정을 내려야 합니다.
        4. 다른 형식의 답변은 허용되지 않습니다.
        
        질문: {query}

        문서들:
        {check_docs}
        """
    }

    # LLM 호출을 위한 메시지 배열 생성
    messages = [system_message]

    # OpenAI API 호출하여 문서 간 비교
    result = client_gpt.chat.completions.create(
        model="chatgpt-4o-latest",  # LLM 모델 지정
        messages=messages,
        temperature=0
    )

    # LLM이 생성한 답변을 반환 (True 또는 False)
    response = result.choices[0].message.content.strip()
    
    print(response)
    if response == "True":
        return True
    else:
        return False



In [131]:
def llm_reranking(query, doc1, doc2):
    """
    LLM을 사용하여 두 개의 문서를 쿼리와 비교하고, 더 적합한 문서를 반환합니다.
    """
    # 문서들의 내용을 준비
    document1 = doc_mapping[doc1[0]]['content']
    document2 = doc_mapping[doc2[0]]['content']
    
    print("문서 1:", document1)
    print("문서 2:", document2)

    check_docs = f"""
    문서 1: {document1}
    문서 2: {document2}
    """

    # 시스템 메시지 생성
    system_message = {
        "role": "system",
        "content": f"""
        당신은 문서와 질문을 비교하여, 질문에 가장 적합한 문서를 찾는 전문가입니다.

        다음 규칙을 따르세요:
        1. 주어진 질문과 문서 1, 문서 2를 비교하여, 질문의 답을 찾을 수 있는 문서를 반드시 선택하세요.
        2. 문서 1이 질문에 더 적합하면 "True"라고 답하고, 문서 2가 질문에 더 적합하면 "False"라고 답하세요.
        3. 반드시 True 또는 False 중 하나만 답하세요. 둘 중 하나의 문서가 더 적합하다는 결정을 내려야 합니다.
        4. 다른 형식의 답변은 허용되지 않습니다.
        
        질문: {query}

        문서들:
        {check_docs}
        """
    }

    # LLM 호출을 위한 메시지 배열 생성
    messages = [system_message]

    # OpenAI API 호출하여 문서 간 비교
    result = client_gpt.chat.completions.create(
        model="chatgpt-4o-latest",  # LLM 모델 지정
        messages=messages,
        temperature=0
    )

    # LLM이 생성한 답변을 반환 (True 또는 False)
    response = result.choices[0].message.content.strip()

    # LLM의 응답을 디버깅하기 위한 출력
    print(f"LLM 응답: {response}")
    
    if response == "True":
        return True
    elif response == "False":
        return False
    else:
        raise ValueError(f"Unexpected response from LLM: {response}")


In [182]:
def llm_reranking(query, doc1, doc2):
    """
    LLM을 사용하여 두 개의 문서를 쿼리와 비교하고, 더 적합한 문서를 반환합니다.
    """
    # 문서들의 내용을 준비
    document1 = doc_mapping[doc1[0]]['content']
    document2 = doc_mapping[doc2[0]]['content']
    
    print("문서 1:", document1)
    print("문서 2:", document2)

    check_docs = f"""
    문서 1: {document1}
    문서 2: {document2}
    """

    # 시스템 메시지 생성
    system_message = {
        "role": "system",
        "content": f"""
        당신은 문서와 질문을 비교하여, 질문에 가장 적합한 문서를 찾는 전문가입니다.

        주어진 질문에 답변한다고 가정하고, 질문에 대한 답을 어느 문서에서 가져올지 선택하세요.
        문서 1이 질문에 더 적합한 답을 제공한다고 생각하면 "True"라고 답하고, 
        문서 2가 더 적합한 답을 제공한다고 생각하면 "False"라고 답하세요.

        반드시 문서 1 또는 문서 2 중 하나를 선택해야 합니다.

        질문: {query}

        문서들:
        {check_docs}

        주의: 
        1. 질문에 대한 답을 실제로 작성한다고 생각하고, 그 답변을 가장 잘 설명하는 문서를 선택하세요.
        2. 주어진 질문에 답을 할때 인용해야 한다면 어떤 문서를 인용할지 생각하고 답하세요.
        """
    }

    # LLM 호출을 위한 메시지 배열 생성
    messages = [system_message]

    # OpenAI API 호출하여 문서 간 비교
    result = client_gpt.chat.completions.create(
        model="chatgpt-4o-latest",  # LLM 모델 지정
        messages=messages,
        temperature=0
    )

    # LLM이 생성한 답변을 반환 (True 또는 False)
    response = result.choices[0].message.content.strip()

    # LLM의 응답을 디버깅하기 위한 출력
    print(f"LLM 응답: {response}")
    
    if response == "True":
        return True
    elif response == "False":
        return False
    else:
        raise ValueError(f"Unexpected response from LLM: {response}")


금성이 행성중 가장 밝게 보이는 이유는?

헬륨이 다른 원소들과 반응을 잘 안하는 이유는?

건설 현장에서 망치로 벽을 치는 이유는?

인간이 2세를 생산할 때 DNA의 결합 과정에 대히 설명해줘.

DNA 조각들이 서로 결합되도록 돕는 것은?

연구자가 갖추어야 할 태도와 자세가 뭘까?

디엔에이와 단백질의 관계와 역할에 대해 설명해줘.

In [187]:
result = retrieval_with_score('헬륨이 다른 원소들과 반응을 잘 안하는 이유는?')

상위 5개의 문서:


In [59]:
result.reverse()

변경전

In [72]:
result

[('41ca41ac-66e3-4a6b-a604-87bf8b3a8d4d', 8.56777154395715),
 ('8132c4ab-f1bd-4915-8196-a13fc3df1f05', 4.706500680804354),
 ('6211fd17-f7cf-4680-82be-365da49b211e', 4.415756458544138),
 ('a395b38f-af11-409e-9c17-89a21f4a9521', 4.348489176775233),
 ('96459005-ecc6-4db9-914a-14043f280a08', 2.822469286815097),
 ('293d2c65-1b0c-4dde-bae4-9aae8c6ca12c', 2.131814371254728),
 ('028975ec-06f7-4e78-bfb1-f960c9a95fe6', 1.9137315541883944),
 ('5fd8794e-1d2b-4c33-b40c-2fa40c6c37c8', 1.842925311162632),
 ('208437c5-3df4-4fae-84b0-f70a0daf6876', 1.6842667023562745),
 ('35b115e9-f7da-4840-bf2d-0cda838fc217', 1.6235752234159926)]

In [137]:
doc_mapping[result[2][0]]['content']

'역학적 에너지의 사용을 가장 잘 나타내는 것은 못을 망치질하는 것입니다. 못을 망치로 내리치는 행위는 에너지를 전달하고, 물체를 움직이는 데 사용되는 역학적 에너지의 전형적인 예입니다. 못을 망치질하는 과정에서 에너지는 못과 망치 사이에서 변환되며, 못을 망치로 내리치는 힘은 에너지의 형태를 변화시킵니다. 이러한 과정은 역학적 에너지의 특성과 활용을 보여주는 중요한 예시입니다. 못을 망치질하는 것은 물리적인 작업을 수행하는 동시에 에너지의 전달과 변환을 나타내는 실생활의 사례입니다.'

In [138]:
doc_mapping[result[1][0]]['content']

'한 목수가 얇은 종이로 나무 조각을 덮었습니다. 그는 덮인 나무 조각을 망치로 쳤습니다. 이 과정에서 충격이 발생하였고, 종이에는 작은 구멍이 생기고 연기 냄새가 났습니다. 이 사건은 역학 에너지에서 열 에너지로 전달되는 에너지 전달 방식을 보여주었습니다. 목수가 망치로 나무 조각을 칠 때, 망치의 운동 에너지가 나무 조각에 전달되어 역학 에너지로 변환됩니다. 이 역학 에너지는 나무 조각과 종이 사이에 전달되어 종이에 충격을 주게 됩니다. 충격으로 인해 종이는 변형되고, 작은 구멍이 생기며 연기 냄새가 나타납니다. 이는 역학 에너지가 종이에 전달되어 열 에너지로 변환되는 과정입니다. 따라서, 이 사건은 역학 에너지에서 열 에너지로 전달되는 에너지 전달 방식을 보여주었을 가능성이 가장 높습니다.'

In [140]:
see = llm_reranking('건설 현장에서 망치로 벽을 치는 이유는?', result[1], result[2])

문서 1: 한 목수가 얇은 종이로 나무 조각을 덮었습니다. 그는 덮인 나무 조각을 망치로 쳤습니다. 이 과정에서 충격이 발생하였고, 종이에는 작은 구멍이 생기고 연기 냄새가 났습니다. 이 사건은 역학 에너지에서 열 에너지로 전달되는 에너지 전달 방식을 보여주었습니다. 목수가 망치로 나무 조각을 칠 때, 망치의 운동 에너지가 나무 조각에 전달되어 역학 에너지로 변환됩니다. 이 역학 에너지는 나무 조각과 종이 사이에 전달되어 종이에 충격을 주게 됩니다. 충격으로 인해 종이는 변형되고, 작은 구멍이 생기며 연기 냄새가 나타납니다. 이는 역학 에너지가 종이에 전달되어 열 에너지로 변환되는 과정입니다. 따라서, 이 사건은 역학 에너지에서 열 에너지로 전달되는 에너지 전달 방식을 보여주었을 가능성이 가장 높습니다.
문서 2: 역학적 에너지의 사용을 가장 잘 나타내는 것은 못을 망치질하는 것입니다. 못을 망치로 내리치는 행위는 에너지를 전달하고, 물체를 움직이는 데 사용되는 역학적 에너지의 전형적인 예입니다. 못을 망치질하는 과정에서 에너지는 못과 망치 사이에서 변환되며, 못을 망치로 내리치는 힘은 에너지의 형태를 변화시킵니다. 이러한 과정은 역학적 에너지의 특성과 활용을 보여주는 중요한 예시입니다. 못을 망치질하는 것은 물리적인 작업을 수행하는 동시에 에너지의 전달과 변환을 나타내는 실생활의 사례입니다.
LLM 응답: False


In [130]:

see

False

In [188]:
i = 1
for doc in result:
    print(i)
    print(doc_mapping[doc[0]]['content'])
    i += 1
    

1
희귀한 기체인 헬륨, 네온, 아르곤, 크립톤, 크세논, 라돈은 다른 원소들과 거의 반응하지 않습니다. 이는 최외각 에너지 준위가 완전하게 채워져 있기 때문입니다. 이러한 특성으로 인해 이러한 희귀 기체들은 안정하고 비활성인 성질을 가지고 있습니다. 이들은 화학 반응에서 거의 관여하지 않으며, 다른 원소들과 결합하여 화합물을 형성하지 않습니다. 이러한 특징은 희귀 기체들을 다양한 산업 분야에서 사용할 수 있게 만들어줍니다. 예를 들어, 헬륨은 기체 냉매로 사용되며, 네온은 광고 표시판에 사용됩니다. 아르곤은 용접 작업에 사용되고, 크세논은 높은 휘도를 가진 조명 장치에 사용됩니다. 이러한 희귀 기체들은 우리 일상 생활에서도 다양한 용도로 활용되고 있습니다.
2
주기율표는 원소들을 그룹과 주기로 나누어 표현하는 방법입니다. 그룹은 세로로 배열된 열을 의미하며, 가장 반응성이 낮은 원소를 포함하는 그룹은 그룹 18 (8A)입니다. 이 그룹은 흔히 비활성 기체라고도 불리며, 헬륨, 네온, 아르곤 등을 포함하고 있습니다. 이러한 원소들은 전자 껍질이 완전히 채워져 있어 다른 원소와의 화학적 반응이 거의 일어나지 않습니다. 따라서, 그룹 18은 가장 반응성이 낮은 원소들을 포함하고 있습니다.
3
원소 주기율표에 따르면, He, Ne, Ar 원소 집합은 유사한 성질을 가지고 있습니다. 이들은 모두 비활성 기체로서, 외부 원소와 거의 반응하지 않는 특성을 가지고 있습니다. 또한, 이들은 모두 8개의 전자를 외께 전자껍질에 가지고 있으며, 이는 전자 구성의 안정성을 나타냅니다. 따라서, He, Ne, Ar 원소는 주기율표에서 같은 주기에 위치하고 있으며, 이는 그들이 유사한 성질을 가지고 있다는 것을 의미합니다.
4
비활성기체가 포함된 등전자는 대표 원소의 1원자 이온으로 종종 나타납니다. 이 등전자는 전자 구성이 완전한 상태이기 때문에 화학적으로 매우 안정하며, 반응성이 낮습니다. 이러한 특성으로 인해 비활성기체인 헬륨, 네온, 아르곤 등이 종종 대표 원소의 1원자 이온

변경후

In [166]:
result

[('97ed16c9-cc0b-4db0-962f-2f00b19f63e2', 4.9191985705880885),
 ('ae30b754-a275-43dc-a2c9-95ab33a7c557', 2.97633994357108),
 ('51dd3dc3-8bee-46ca-96af-de0af97900df', 4.006756761839501),
 ('a78116b6-398d-4453-a42b-3b4719b730b9', 1.993902338812477),
 ('85bd0c53-10c4-4c8b-993e-3a00ef7cbd84', 1.8882969555120004),
 ('17b034ae-7ed0-4698-9714-d89cf822f7fd', 1.8539707900430797),
 ('a19f30a9-eb0a-4985-9ecd-0cda8e54e549', 2.321521270482699),
 ('86d9d3cf-7f25-462e-b6a4-acea602fa1a2', 2.133682334986716),
 ('c8fd4323-9af9-4a0d-ab53-e563c71f9795', 2.1144502206554647),
 ('eace55fd-4d5e-4aab-a733-c8a91f553a5b', 1.8661814847688378)]

In [172]:
result

[('ae30b754-a275-43dc-a2c9-95ab33a7c557', 2.97633994357108),
 ('51dd3dc3-8bee-46ca-96af-de0af97900df', 4.006756761839501),
 ('85bd0c53-10c4-4c8b-993e-3a00ef7cbd84', 1.8882969555120004),
 ('97ed16c9-cc0b-4db0-962f-2f00b19f63e2', 4.9191985705880885),
 ('a78116b6-398d-4453-a42b-3b4719b730b9', 1.993902338812477),
 ('a19f30a9-eb0a-4985-9ecd-0cda8e54e549', 2.321521270482699),
 ('17b034ae-7ed0-4698-9714-d89cf822f7fd', 1.8539707900430797),
 ('86d9d3cf-7f25-462e-b6a4-acea602fa1a2', 2.133682334986716),
 ('c8fd4323-9af9-4a0d-ab53-e563c71f9795', 2.1144502206554647),
 ('eace55fd-4d5e-4aab-a733-c8a91f553a5b', 1.8661814847688378)]

In [190]:
i = 1
for doc in result:
    print(i)
    print(doc_mapping[doc[0]]['content'])
    i += 1

1
태양과 같은 별에서 가장 흔한 원소는 수소입니다. 수소는 우주에서 가장 풍부한 원소로, 태양의 주요 구성 요소입니다. 태양은 수소 원자들이 핵융합되어 헬륨을 생성하는 핵융합 반응을 일으키는 핵별입니다. 이러한 핵융합 반응은 태양의 열과 빛을 발생시키며, 지구에 에너지를 공급합니다. 수소는 또한 우주의 다른 별들에서도 매우 흔하게 발견되며, 우리 은하의 모든 별들의 주요 구성 요소입니다. 따라서 태양과 같은 별에서 가장 흔한 원소는 수소입니다.
2
크립톤은 화학적 활성이 거의 없는 기체입니다. 이러한 특성을 가진 다른 원소를 찾기 위해서는 학생은 원소 주기율표에서 같은 족의 원소를 찾아야 합니다. 같은 족의 원소들은 비슷한 화학적 특성을 가지고 있으며, 크립톤과 같이 화학 반응에 참여하기 어렵습니다. 따라서, 학생은 주기율표에서 크립톤과 같은 특성을 가진 다른 원소들을 찾을 수 있을 것입니다.
3
알칼리 토금속 원소들은 주기율표에서 같은 족에 속하는데, 그 이유는 그 원소들이 모두 두 개의 원자가 전자를 가지고 있기 때문입니다. 이러한 특성은 이들 원소들이 비슷한 화학적 특성을 가지고 있음을 의미합니다. 알칼리 토금속 원소들은 전자 구성이 비슷하므로, 동일한 족에 속하는 다른 원소들과 비교하여 비슷한 화학 반응을 보입니다. 이러한 특성은 주기율표를 사용하여 원소들의 화학적 특성을 예측하는 데에도 도움을 줍니다. 알칼리 토금속 원소들은 전자 구성이 비슷하므로, 동일한 족에 속하는 다른 원소들과 비교하여 비슷한 물리적 특성을 가지기도 합니다. 이러한 특성은 알칼리 토금속 원소들이 비슷한 화학 반응을 보이는 것뿐만 아니라, 비슷한 물리적 특성을 가지는 이유를 설명해줍니다. 따라서, 알칼리 토금속 원소들은 주기율표에서 같은 족에 속하는 이유는 그들이 모두 두 개의 원자가 전자를 가지고 있기 때문입니다.
4
희귀한 기체인 헬륨, 네온, 아르곤, 크립톤, 크세논, 라돈은 다른 원소들과 거의 반응하지 않습니다. 이는 최외각 에너지 준위가 완전하게 채워져 있기 때문입니다.

In [189]:
for i in range(3):
    for j in range(-1, -len(result) + i, -1):
        large = llm_reranking('연구자가 갖추어야 할 태도와 자세가 뭘까?', result[j -1], result[j])
        # print(j)
        # print(doc_mapping[result[j-1][0]]['content'])
        # print(doc_mapping[result[j][0]]['content'])
        print(large)
        if large == False:
            result[j] , result[j - 1] = result[j - 1], result[j]
            print('changed')

문서 1: 크립톤은 화학적 활성이 거의 없는 기체입니다. 이러한 특성을 가진 다른 원소를 찾기 위해서는 학생은 원소 주기율표에서 같은 족의 원소를 찾아야 합니다. 같은 족의 원소들은 비슷한 화학적 특성을 가지고 있으며, 크립톤과 같이 화학 반응에 참여하기 어렵습니다. 따라서, 학생은 주기율표에서 크립톤과 같은 특성을 가진 다른 원소들을 찾을 수 있을 것입니다.
문서 2: 태양과 같은 별에서 가장 흔한 원소는 수소입니다. 수소는 우주에서 가장 풍부한 원소로, 태양의 주요 구성 요소입니다. 태양은 수소 원자들이 핵융합되어 헬륨을 생성하는 핵융합 반응을 일으키는 핵별입니다. 이러한 핵융합 반응은 태양의 열과 빛을 발생시키며, 지구에 에너지를 공급합니다. 수소는 또한 우주의 다른 별들에서도 매우 흔하게 발견되며, 우리 은하의 모든 별들의 주요 구성 요소입니다. 따라서 태양과 같은 별에서 가장 흔한 원소는 수소입니다.
LLM 응답: False
False
changed
문서 1: 알칼리 토금속 원소들은 주기율표에서 같은 족에 속하는데, 그 이유는 그 원소들이 모두 두 개의 원자가 전자를 가지고 있기 때문입니다. 이러한 특성은 이들 원소들이 비슷한 화학적 특성을 가지고 있음을 의미합니다. 알칼리 토금속 원소들은 전자 구성이 비슷하므로, 동일한 족에 속하는 다른 원소들과 비교하여 비슷한 화학 반응을 보입니다. 이러한 특성은 주기율표를 사용하여 원소들의 화학적 특성을 예측하는 데에도 도움을 줍니다. 알칼리 토금속 원소들은 전자 구성이 비슷하므로, 동일한 족에 속하는 다른 원소들과 비교하여 비슷한 물리적 특성을 가지기도 합니다. 이러한 특성은 알칼리 토금속 원소들이 비슷한 화학 반응을 보이는 것뿐만 아니라, 비슷한 물리적 특성을 가지는 이유를 설명해줍니다. 따라서, 알칼리 토금속 원소들은 주기율표에서 같은 족에 속하는 이유는 그들이 모두 두 개의 원자가 전자를 가지고 있기 때문입니다.
문서 2: 태양과 같은 별에서 가장 흔한 원소는 수소입니다. 수소는 우주에서 가장 풍부한 원소

In [68]:
large

False

In [30]:
result

[('464ace62-ddf2-423d-a5d7-2f17e6785c8e', 8.518137952908454),
 ('40d1d16f-bffe-4196-a0b3-ae88be0a5106', 1.3433907646651053),
 ('25d04e49-b589-4147-a5e3-59b949673d8a', 1.3653075598359854),
 ('7ddfa9e7-0782-4079-930e-4758077ce4fd', 1.5205123265147749),
 ('a4e3a126-3eb3-4b06-8f4d-082c698fa366', 1.6007516592480513),
 ('1bae8047-4652-4155-9396-780dd46bbca3', 2.1485863606700972),
 ('52f35132-97a3-4010-b409-3a7e40126c63', 2.2868996574189566),
 ('35c5dcc7-4720-4318-901e-770105ae63fd', 2.6885562485096655),
 ('da6c8a3f-45a9-4025-a63a-47c05ba2b336', 2.9966446479205953),
 ('45b8eb6a-87e3-4333-b01b-7c8b772f827f', 3.124107354992668)]

In [28]:
result

[('40d1d16f-bffe-4196-a0b3-ae88be0a5106', 1.3433907646651053),
 ('25d04e49-b589-4147-a5e3-59b949673d8a', 1.3653075598359854),
 ('7ddfa9e7-0782-4079-930e-4758077ce4fd', 1.5205123265147749),
 ('a4e3a126-3eb3-4b06-8f4d-082c698fa366', 1.6007516592480513),
 ('1bae8047-4652-4155-9396-780dd46bbca3', 2.1485863606700972),
 ('52f35132-97a3-4010-b409-3a7e40126c63', 2.2868996574189566),
 ('35c5dcc7-4720-4318-901e-770105ae63fd', 2.6885562485096655),
 ('da6c8a3f-45a9-4025-a63a-47c05ba2b336', 2.9966446479205953),
 ('45b8eb6a-87e3-4333-b01b-7c8b772f827f', 3.124107354992668),
 ('464ace62-ddf2-423d-a5d7-2f17e6785c8e', 8.518137952908454)]

reanking  -- 뒤 앞

j = -1
j - 1 = -2

In [20]:
let_see = llm_reranking('금성이 행성중 가장 밝게 보이는 이유는?', result[1], result[0])

In [21]:
let_see

True

In [11]:
i = 1
for doc in result:
    print(i)
    print(doc_mapping[doc[0]]['content'])
    i += 1
    

1
금성이 다른 행성들보다 더 밝게 보이는 이유는 지구 쪽으로 가장 많은 햇빛을 반사하기 때문입니다. 케빈은 맑은 밤에 하늘을 관찰하고 있습니다. 그는 맨눈으로 금성, 화성, 목성, 토성을 볼 수 있습니다. 금성은 햇빛을 많이 반사하기 때문에 다른 행성들보다 더 밝게 보입니다. 이는 금성의 표면이 반사율이 높기 때문입니다. 금성은 태양으로부터 받은 햇빛을 표면에 반사하여 지구에서 관찰하기 쉽게 만듭니다. 따라서 케빈은 맑은 밤에 금성을 더 밝게 볼 수 있습니다.
2
금성은 태양계에서 가장 가까운 행성 중 하나입니다. 그러나 화성이나 지구처럼 계절이 없는 이유는 금성의 자전축이 태양계의 평면에 거의 수직이기 때문입니다. 자전축이 수직이기 때문에 금성은 태양으로부터 받는 햇빛의 양이 일정하게 유지됩니다. 이로 인해 금성은 계절 변화가 없으며 항상 일정한 온도를 유지합니다. 이러한 환경은 생명체에게는 적합하지 않을 수 있지만, 금성의 특이한 기후 조건은 우주 탐사에 대한 연구에 많은 도움을 주고 있습니다. 금성은 여전히 우리에게 알려지지 않은 많은 비밀을 품고 있으며, 미래에 더 많은 연구와 탐사가 이루어질 것으로 기대됩니다.
3
금성은 태양계의 두 번째로 가까운 행성입니다. 이 행성의 대략적인 나이는 7억 5천만 년으로 추정됩니다. 금성은 지구와 매우 비슷한 크기와 구성을 가지고 있으며, 약 90% 이상이 이산화탄소로 이루어져 있습니다. 이 행성은 매우 뜨거운 온도와 압력을 가지고 있어서 인간이 살 수 있는 환경이 아닙니다. 금성의 대기는 두꺼워서 태양의 열을 가두고 있어서 행성의 표면은 평균 온도가 약 450도로 매우 뜨거운 상태입니다. 또한, 금성은 자전 속도가 매우 빠르기 때문에 하루가 지구의 약 243일과 같습니다. 이러한 특징들로 인해 금성은 우리 태양계에서 가장 가혹한 환경을 가진 행성 중 하나로 알려져 있습니다.
4
당신은 금성에 살고 있고, 당신의 망원경이 금성의 두꺼운 구름을 볼 수 있다고 가정한다면, 지구의 달을 관찰할 수 있습니다. 지구의 달은

In [39]:
a = [1, 2, 3, 4, 5, 6, 7, 8, 9]

In [40]:
def bigger(a, b):
    if a > b:
        return 'True'


for i in range(2):
    for j in range(-1, -len(a) + i, -1):
        large = bigger(a[j], a[j - 1])
        if large == 'True':
            a[j] , a[j - 1] = a[j - 1], a[j]

In [41]:
print(a)

[9, 8, 1, 2, 3, 4, 5, 6, 7]
